<a href="https://colab.research.google.com/github/fabiorcerqueira/Florence/blob/main/testing_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiments with several models for the work Florence of assistive monitoring

# Cloning repos

In [ ]:
import os
import subprocess
from pathlib import Path

BASE = Path("/content/florence_visapp")
BASE.mkdir(parents=True, exist_ok=True)

repos = {
    "images": "https://github.com/fabiorcerqueira/pain_nopain_img_rep.git",
    "validation": "https://github.com/fabiorcerqueira/create_form_for_img_human_validation.git",
    "florence": "https://github.com/fabiorcerqueira/Florence.git"
}

for name, url in repos.items():
    dest = BASE / name
    if not dest.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", url, str(dest)],
            check=True
        )
    print(f"\n=== {name.upper()} ===")
    print("Commit:", subprocess.check_output(
        ["git", "-C", str(dest), "rev-parse", "HEAD"],
        text=True
    ).strip())
    for root, dirs, files in os.walk(dest):
        dirs[:] = [d for d in dirs if d != ".git"]
        level = len(Path(root).relative_to(dest).parts)
        if level > 2:
            dirs[:] = []
            continue
        print("  " * level + Path(root).name + "/")
        for file in sorted(files)[:20]:
            print("  " * (level + 1) + file)


=== IMAGES ===
Commit: 0d56313dbea1884b26171e540a4a9c77eed1f128
images/
  README.md
  form_answers_in_portuguese.ods
  form_links.txt
  PainImages/
    pain/
      image_1.jpg
      image_103.jpg
      image_104.png
      image_106.jpg
      image_107.jpg
      image_118.jpg
      image_13.jpg
      image_130.jpg
      image_131.jpg
      image_14.jpg
      image_154.jpg
      image_156.jpg
      image_157.jpg
      image_158.jpg
      image_159.jpg
      image_16.png
      image_160.jpg
      image_161.jpg
      image_162.jpg
      image_163.jpg
  NoPainImages/
    neutral/
      image_96.jfif
    surprise/
      image_105.jfif
      image_108.jfif
      image_109.jfif
      image_110.jpg
      image_111.jfif
      image_112.jfif
      image_113.jfif
      image_114.jfif
      image_115.jfif
      image_116.jfif
      image_119.jfif
      image_120.jfif
      image_121.jfif
      image_122.jfif
      image_123.jfif
      image_124.jfif
      image_125.jfif
      image_126.jfif
      

# Testing CLIP ViT-B/32 zero-shot

In [ ]:

!pip -q install transformers accelerate scikit-learn pandas pillow

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

BASE = Path("/content/florence_visapp")
IMG_ROOT = BASE / "images"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

MODEL_ID = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE)
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model.eval()

PROMPTS = {
    "binary": [
        "A person experiencing pain.",
        "A person not experiencing pain."
    ],
    "affective": [
        "A person experiencing pain.",
        "A happy person.",
        "A sad person.",
        "An angry person.",
        "A disgusted person.",
        "A surprised person."
    ]
}

extensions = {".jpg", ".jpeg", ".png", ".jfif", ".webp"}

images = []

for folder, label in [("PainImages", 1), ("NoPainImages", 0)]:
    for path in sorted((IMG_ROOT / folder).rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            images.append((path, label))

print("Total de imagens:", len(images))
assert len(images) == 189, "Quantidade diferente de 189!"

results = []

with torch.inference_mode():
    for index, (path, true_label) in enumerate(images, 1):
        with Image.open(path) as im:
            image = im.convert("RGB")

        for strategy, prompts in PROMPTS.items():
            inputs = processor(
                text=prompts,
                images=image,
                return_tensors="pt",
                padding=True
            ).to(DEVICE)

            outputs = model(**inputs)
            scores = outputs.logits_per_image[0]
            predicted_prompt = int(scores.argmax().item())

            prediction = 1 if predicted_prompt == 0 else 0

            results.append({
                "image": str(path.relative_to(IMG_ROOT)),
                "true_label": true_label,
                "model": MODEL_ID,
                "strategy": strategy,
                "predicted_label": prediction,
                "predicted_prompt": predicted_prompt,
                "pain_score": float(scores[0].item()),
                "correct": int(prediction == true_label)
            })

        if index % 25 == 0:
            print(f"Processadas: {index}/{len(images)}")

df = pd.DataFrame(results)

output_dir = BASE / "results"
output_dir.mkdir(exist_ok=True)

df.to_csv(
    output_dir / "clip_vit_b32_predictions.csv",
    index=False
)

print("\n=== RESULTADOS ===")

for strategy, group in df.groupby("strategy"):
    y_true = group["true_label"]
    y_pred = group["predicted_label"]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    print(f"\nEstratégia: {strategy}")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("Specificity:", round(tn / (tn + fp), 4))
    print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nArquivo salvo em:", output_dir)


Dispositivo: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

... (5 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Total de imagens: 189
Processadas: 25/189
Processadas: 50/189
Processadas: 75/189
Processadas: 100/189
Processadas: 125/189
Processadas: 150/189
Processadas: 175/189

=== RESULTADOS ===

Estratégia: affective
Accuracy: 0.8836
Precision: 0.8468
Recall: 0.9495
Specificity: 0.8111
F1: 0.8952
Balanced accuracy: 0.8803
Confusion matrix:
[[73 17]
 [ 5 94]]

Estratégia: binary
Accuracy: 0.672
Precision: 0.6434
Recall: 0.8384
Specificity: 0.4889
F1: 0.7281
Balanced accuracy: 0.6636
Confusion matrix:
[[44 46]
 [16 83]]

Arquivo salvo em: /content/florence_visapp/results


## McNemar's exact test between the two CLIP strategies. A paired comparison

```
# Isto está formatado como código
```



In [ ]:

!pip -q install statsmodels

import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from pathlib import Path

file = Path(
    "/content/florence_visapp/results/"
    "clip_vit_b32_predictions.csv"
)

df = pd.read_csv(file)

paired = df.pivot(
    index="image",
    columns="strategy",
    values="correct"
)

assert len(paired) == 189
assert not paired.isna().any().any()

a = int(((paired["binary"] == 1) &
         (paired["affective"] == 1)).sum())

b = int(((paired["binary"] == 1) &
         (paired["affective"] == 0)).sum())

c = int(((paired["binary"] == 0) &
         (paired["affective"] == 1)).sum())

d = int(((paired["binary"] == 0) &
         (paired["affective"] == 0)).sum())

table = [[a, b], [c, d]]

test = mcnemar(table, exact=True)

print("=== McNemar: Binary vs Affective ===")
print("Ambos corretos:", a)
print("Somente Binary correto:", b)
print("Somente Affective correto:", c)
print("Ambos incorretos:", d)
print("p-value:", test.pvalue)

print("\nDiferença de accuracy:",
      round((c-b)/189*100, 2),
      "pontos percentuais")


=== McNemar: Binary vs Affective ===
Ambos corretos: 124
Somente Binary correto: 3
Somente Affective correto: 43
Ambos incorretos: 19
p-value: 4.6219383875722997e-10

Diferença de accuracy: 21.16 pontos percentuais


# Testing SigLIP2 zero-shot

In [ ]:

!pip -q install -U transformers accelerate safetensors

import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import AutoModel, AutoProcessor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)

BASE = Path("/content/florence_visapp")
IMG_ROOT = BASE / "images"
OUT = BASE / "results"
OUT.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "google/siglip2-base-patch16-224"

print("Dispositivo:", DEVICE)
print("Modelo:", MODEL_ID)

model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()

PROMPTS = {
    "binary": [
        "A person experiencing pain.",
        "A person not experiencing pain."
    ],
    "affective": [
        "A person experiencing pain.",
        "A happy person.",
        "A sad person.",
        "An angry person.",
        "A disgusted person.",
        "A surprised person."
    ]
}

extensions = {".jpg", ".jpeg", ".png", ".jfif", ".webp"}
images = []

for folder, label in [("PainImages", 1), ("NoPainImages", 0)]:
    for path in sorted((IMG_ROOT / folder).rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            images.append((path, label))

assert len(images) == 189, "FPD-189 incompleto!"

# Preparar os prompts uma única vez
text_inputs = {}
for strategy, prompts in PROMPTS.items():
    text_inputs[strategy] = processor(
        text=prompts,
        padding="max_length",
        return_tensors="pt"
    ).to(DEVICE)

results = []

with torch.inference_mode():
    for index, (path, true_label) in enumerate(images, 1):
        with Image.open(path) as im:
            image = im.convert("RGB")

        image_inputs = processor(
            images=image,
            return_tensors="pt"
        ).to(DEVICE)

        for strategy in PROMPTS:
            inputs = {
                **image_inputs,
                **text_inputs[strategy]
            }

            outputs = model(**inputs)
            scores = outputs.logits_per_image[0]

            predicted_prompt = int(scores.argmax().item())
            prediction = int(predicted_prompt == 0)

            results.append({
                "image": str(path.relative_to(IMG_ROOT)),
                "true_label": true_label,
                "model": MODEL_ID,
                "strategy": strategy,
                "predicted_label": prediction,
                "predicted_prompt": predicted_prompt,
                "pain_score": float(scores[0].item()),
                "correct": int(prediction == true_label)
            })

        if index % 25 == 0:
            print(f"Processadas: {index}/189")

df = pd.DataFrame(results)

file = OUT / "siglip2_base_predictions.csv"
df.to_csv(file, index=False)

print("\n=== RESULTADOS SIGLIP2 ===")

for strategy, group in df.groupby("strategy"):
    y_true = group.true_label
    y_pred = group.predicted_label

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    print("\nEstratégia:", strategy)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("Specificity:", round(tn/(tn+fp), 4))
    print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Balanced accuracy:",
          round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nArquivo salvo:", file)


Dispositivo: cuda
Modelo: google/siglip2-base-patch16-224


config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Processadas: 25/189
Processadas: 50/189
Processadas: 75/189
Processadas: 100/189
Processadas: 125/189
Processadas: 150/189
Processadas: 175/189

=== RESULTADOS SIGLIP2 ===

Estratégia: affective
Accuracy: 0.9312
Precision: 0.9479
Recall: 0.9192
Specificity: 0.9444
F1: 0.9333
Balanced accuracy: 0.9318
Confusion matrix:
[[85  5]
 [ 8 91]]

Estratégia: binary
Accuracy: 0.6825
Precision: 0.8824
Recall: 0.4545
Specificity: 0.9333
F1: 0.6
Balanced accuracy: 0.6939
Confusion matrix:
[[84  6]
 [54 45]]

Arquivo salvo: /content/florence_visapp/results/siglip2_base_predictions.csv


## McNemar's exact test with Holm correction

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

OUT = Path("/content/florence_visapp/results")

clip = pd.read_csv(OUT / "clip_vit_b32_predictions.csv")
sig = pd.read_csv(OUT / "siglip2_base_predictions.csv")

def prepare(df, name):
    df = df.copy()
    df["model_name"] = name
    assert not df.duplicated(["image", "strategy"]).any()
    assert set(df["strategy"]) == {"binary", "affective"}
    return df

clip = prepare(clip, "CLIP")
sig = prepare(sig, "SigLIP2")

combined = pd.concat([clip, sig], ignore_index=True)

wide = combined.pivot(
    index="image",
    columns=["model_name", "strategy"],
    values="correct"
)

labels = combined.pivot(
    index="image",
    columns=["model_name", "strategy"],
    values="true_label"
)

assert len(wide) == 189
assert not wide.isna().any().any()
assert (labels.nunique(axis=1) == 1).all()

comparisons = [
    (("CLIP", "binary"), ("CLIP", "affective")),
    (("SigLIP2", "binary"), ("SigLIP2", "affective")),
    (("CLIP", "affective"), ("SigLIP2", "affective")),
    (("CLIP", "binary"), ("SigLIP2", "binary"))
]

results = []

for first, second in comparisons:
    x = wide[first].astype(int)
    y = wide[second].astype(int)

    both_correct = int(((x == 1) & (y == 1)).sum())
    only_first = int(((x == 1) & (y == 0)).sum())
    only_second = int(((x == 0) & (y == 1)).sum())
    both_wrong = int(((x == 0) & (y == 0)).sum())

    table = [
        [both_correct, only_first],
        [only_second, both_wrong]
    ]

    p = mcnemar(table, exact=True).pvalue

    results.append({
        "comparison": f"{first} vs {second}",
        "only_first_correct": only_first,
        "only_second_correct": only_second,
        "accuracy_difference_pp":
            round((only_second - only_first) / 189 * 100, 2),
        "p_raw": p
    })

res = pd.DataFrame(results)

res["p_holm"] = multipletests(
    res["p_raw"],
    method="holm"
)[1]

res["significant_0_05"] = res["p_holm"] < 0.05

print("\n=== COMPARAÇÕES PAREADAS ===")
print(res.to_string(index=False))

res.to_csv(
    OUT / "clip_siglip2_statistical_comparison.csv",
    index=False
)



=== COMPARAÇÕES PAREADAS ===
                                       comparison  only_first_correct  only_second_correct  accuracy_difference_pp        p_raw       p_holm  significant_0_05
      ('CLIP', 'binary') vs ('CLIP', 'affective')                   3                   43                   21.16 4.621938e-10 1.386582e-09              True
('SigLIP2', 'binary') vs ('SigLIP2', 'affective')                   4                   51                   24.87 2.047423e-11 8.189693e-11              True
('CLIP', 'affective') vs ('SigLIP2', 'affective')                   8                   17                    4.76 1.077521e-01 2.155043e-01             False
      ('CLIP', 'binary') vs ('SigLIP2', 'binary')                  53                   55                    1.06 9.234010e-01 9.234010e-01             False


# Testing SigLIP Original

In [ ]:

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import AutoModel, AutoProcessor
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

# ==========================================
# CONFIGURAÇÃO
# ==========================================

BASE = Path("/content/florence_visapp")
IMG_ROOT = BASE / "images"
OUT = BASE / "results"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "google/siglip-base-patch16-224"

print("Dispositivo:", DEVICE)
print("Modelo:", MODEL_ID)

# ==========================================
# CARREGAMENTO DO MODELO
# ==========================================

model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()

# ==========================================
# PROMPTS IDÊNTICOS AOS ANTERIORES
# ==========================================

PROMPTS = {
    "binary": [
        "A person experiencing pain.",
        "A person not experiencing pain."
    ],
    "affective": [
        "A person experiencing pain.",
        "A happy person.",
        "A sad person.",
        "An angry person.",
        "A disgusted person.",
        "A surprised person."
    ]
}

# ==========================================
# IMAGENS FPD-189
# ==========================================

extensions = {".jpg", ".jpeg", ".png", ".jfif", ".webp"}

images = []

for folder, label in [
    ("PainImages", 1),
    ("NoPainImages", 0)
]:
    for path in sorted((IMG_ROOT / folder).rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            images.append((path, label))

assert len(images) == 189, "FPD-189 incompleto!"

print("Total de imagens:", len(images))

# ==========================================
# PREPARAÇÃO DOS PROMPTS
# ==========================================

text_inputs = {}

for strategy, prompts in PROMPTS.items():
    text_inputs[strategy] = processor(
        text=prompts,
        padding="max_length",
        return_tensors="pt"
    ).to(DEVICE)

# ==========================================
# INFERÊNCIA ZERO-SHOT
# ==========================================

results = []

with torch.inference_mode():

    for index, (path, true_label) in enumerate(images, 1):

        with Image.open(path) as im:
            image = im.convert("RGB")

        image_inputs = processor(
            images=image,
            return_tensors="pt"
        ).to(DEVICE)

        for strategy in PROMPTS:

            inputs = {
                **image_inputs,
                **text_inputs[strategy]
            }

            outputs = model(**inputs)

            scores = outputs.logits_per_image[0]

            predicted_prompt = int(
                scores.argmax().item()
            )

            prediction = int(
                predicted_prompt == 0
            )

            results.append({
                "image": str(path.relative_to(IMG_ROOT)),
                "true_label": true_label,
                "model": MODEL_ID,
                "strategy": strategy,
                "predicted_label": prediction,
                "predicted_prompt": predicted_prompt,
                "pain_score": float(scores[0].item()),
                "correct": int(
                    prediction == true_label
                )
            })

        if index % 25 == 0:
            print(
                f"Processadas: {index}/{len(images)}"
            )

# ==========================================
# SALVAR PREDIÇÕES
# ==========================================

df = pd.DataFrame(results)

file = OUT / "siglip_base_predictions.csv"

df.to_csv(file, index=False)

# ==========================================
# MÉTRICAS
# ==========================================

print("\n=== RESULTADOS SIGLIP BASE ===")

for strategy, group in df.groupby("strategy"):

    y_true = group["true_label"]
    y_pred = group["predicted_label"]

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    print("\nEstratégia:", strategy)

    print(
        "Accuracy:",
        round(accuracy_score(y_true, y_pred), 4)
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ), 4
        )
    )

    print(
        "Recall:",
        round(
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ), 4
        )
    )

    print(
        "Specificity:",
        round(tn / (tn + fp), 4)
    )

    print(
        "F1:",
        round(
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ), 4
        )
    )

    print(
        "Balanced accuracy:",
        round(
            balanced_accuracy_score(
                y_true,
                y_pred
            ), 4
        )
    )

    print("Confusion matrix:")

    print(
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        )
    )

print("\nArquivo salvo:", file)


Dispositivo: cuda
Modelo: google/siglip-base-patch16-224


config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

... (2 outputs left out)

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

Total de imagens: 189
Processadas: 25/189
Processadas: 50/189
Processadas: 75/189
Processadas: 100/189
Processadas: 125/189
Processadas: 150/189
Processadas: 175/189

=== RESULTADOS SIGLIP BASE ===

Estratégia: affective
Accuracy: 0.9365
Precision: 0.9888
Recall: 0.8889
Specificity: 0.9889
F1: 0.9362
Balanced accuracy: 0.9389
Confusion matrix:
[[89  1]
 [11 88]]

Estratégia: binary
Accuracy: 0.709
Precision: 0.775
Recall: 0.6263
Specificity: 0.8
F1: 0.6927
Balanced accuracy: 0.7131
Confusion matrix:
[[72 18]
 [37 62]]

Arquivo salvo: /content/florence_visapp/results/siglip_base_predictions.csv


## McNemar's exact test with Holm correction

In [ ]:

import pandas as pd
from pathlib import Path
from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

OUT = Path("/content/florence_visapp/results")

files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv"
}

frames = []

for name, filename in files.items():
    df = pd.read_csv(OUT / filename)
    df["model_name"] = name
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

assert not data.duplicated(
    ["image", "model_name", "strategy"]
).any()

wide = data.pivot(
    index="image",
    columns=["model_name", "strategy"],
    values="correct"
)

labels = data.pivot(
    index="image",
    columns=["model_name", "strategy"],
    values="true_label"
)

assert len(wide) == 189
assert not wide.isna().any().any()
assert (labels.nunique(axis=1) == 1).all()

models = list(files.keys())
comparisons = []

# Comparações entre prompts no mesmo modelo
for model in models:
    comparisons.append((
        (model, "binary"),
        (model, "affective")
    ))

# Comparações entre modelos na mesma estratégia
for strategy in ["binary", "affective"]:
    for a, b in combinations(models, 2):
        comparisons.append((
            (a, strategy),
            (b, strategy)
        ))

results = []

for first, second in comparisons:
    x = wide[first].astype(int)
    y = wide[second].astype(int)

    a = int(((x == 1) & (y == 1)).sum())
    b = int(((x == 1) & (y == 0)).sum())
    c = int(((x == 0) & (y == 1)).sum())
    d = int(((x == 0) & (y == 0)).sum())

    p = mcnemar(
        [[a, b], [c, d]],
        exact=True
    ).pvalue

    results.append({
        "first": f"{first[0]} {first[1]}",
        "second": f"{second[0]} {second[1]}",
        "only_first_correct": b,
        "only_second_correct": c,
        "difference_pp": round(
            (c - b) / 189 * 100, 2
        ),
        "p_raw": p
    })

res = pd.DataFrame(results)

res["p_holm"] = multipletests(
    res["p_raw"],
    method="holm"
)[1]

res["significant"] = res["p_holm"] < 0.05

print("\n=== COMPARAÇÕES ESTATÍSTICAS ===")
print(res.to_string(index=False))

res.to_csv(
    OUT / "three_models_mcnemar_holm.csv",
    index=False
)



=== COMPARAÇÕES ESTATÍSTICAS ===
           first            second  only_first_correct  only_second_correct  difference_pp        p_raw       p_holm  significant
     CLIP binary    CLIP affective                   3                   43          21.16 4.621938e-10 3.235357e-09         True
   SigLIP binary  SigLIP affective                   2                   45          22.75 1.604405e-11 1.443965e-10         True
  SigLIP2 binary SigLIP2 affective                   4                   51          24.87 2.047423e-11 1.637939e-10         True
     CLIP binary     SigLIP binary                  40                   47           3.70 5.202916e-01 1.000000e+00        False
     CLIP binary    SigLIP2 binary                  53                   55           1.06 9.234010e-01 1.000000e+00        False
   SigLIP binary    SigLIP2 binary                  26                   21          -2.65 5.600646e-01 1.000000e+00        False
  CLIP affective  SigLIP affective                  10  

# Confidence Intervals as a complement to McNemar's exact test

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("/content/florence_visapp/results")

files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv"
}

frames = []

for name, filename in files.items():
    df = pd.read_csv(OUT / filename)
    df["model_name"] = name
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

assert not data.duplicated(
    ["image", "model_name", "strategy"]
).any()

wide = data.pivot(
    index="image",
    columns=["model_name", "strategy"],
    values="correct"
).sort_index()

assert len(wide) == 189
assert not wide.isna().any().any()

rng = np.random.default_rng(2026)
B = 10000
N = len(wide)

# Mesmos índices para todas as condições:
# bootstrap pareado por imagem.
indices = rng.integers(0, N, size=(B, N))

boot = {}
summary = []

for model in files:
    for strategy in ["binary", "affective"]:
        values = wide[(model, strategy)].to_numpy(
            dtype=float
        )

        samples = values[indices].mean(axis=1)
        boot[(model, strategy)] = samples

        lo, hi = np.percentile(samples, [2.5, 97.5])

        summary.append({
            "model": model,
            "strategy": strategy,
            "accuracy": values.mean(),
            "ci95_lower": lo,
            "ci95_upper": hi
        })

print("\n=== ACCURACY COM IC 95% ===")

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False, float_format="%.4f"))

differences = []

for model in files:
    binary = wide[(model, "binary")].to_numpy(
        dtype=float
    )
    affective = wide[(model, "affective")].to_numpy(
        dtype=float
    )

    observed = (affective - binary).mean() * 100

    boot_diff = (
        boot[(model, "affective")]
        - boot[(model, "binary")]
    ) * 100

    lo, hi = np.percentile(boot_diff, [2.5, 97.5])

    differences.append({
        "model": model,
        "gain_pp": observed,
        "ci95_lower_pp": lo,
        "ci95_upper_pp": hi
    })

print("\n=== GANHO AFFECTIVE VS BINARY ===")

diff_df = pd.DataFrame(differences)
print(diff_df.to_string(index=False, float_format="%.2f"))

summary_df.to_csv(
    OUT / "accuracy_bootstrap_ci95.csv",
    index=False
)

diff_df.to_csv(
    OUT / "prompt_gain_bootstrap_ci95.csv",
    index=False
)

print("\nArquivos salvos em:", OUT)



=== ACCURACY COM IC 95% ===
  model  strategy  accuracy  ci95_lower  ci95_upper
   CLIP    binary    0.6720      0.6032      0.7354
   CLIP affective    0.8836      0.8360      0.9259
 SigLIP    binary    0.7090      0.6455      0.7725
 SigLIP affective    0.9365      0.8995      0.9683
SigLIP2    binary    0.6825      0.6138      0.7460
SigLIP2 affective    0.9312      0.8942      0.9630

=== GANHO AFFECTIVE VS BINARY ===
  model  gain_pp  ci95_lower_pp  ci95_upper_pp
   CLIP    21.16          14.81          27.53
 SigLIP    22.75          16.40          29.10
SigLIP2    24.87          17.99          31.75

Arquivos salvos em: /content/florence_visapp/results


In [ ]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path("/content/florence_visapp/results")
destination = Path(
    "/content/drive/MyDrive/Florence_VISAPP/results"
)

destination.mkdir(parents=True, exist_ok=True)

for file in source.glob("*.csv"):
    shutil.copy2(file, destination / file.name)
    print("Salvo:", file.name)

print("\nBackup concluído em:", destination)


ValueError: mount failed

# Semantic Robustness of Prompts

Now, we will investigate whether the results obtained with CLIP, SigLIP, and SigLIP2 remain stable when we change the formulation of the textual descriptions, without modifying the images, models, or classification rule.

In [ ]:

import gc
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import AutoModel, AutoProcessor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)

BASE = Path("/content/florence_visapp")
IMG_ROOT = BASE / "images"
OUT = BASE / "results"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

MODELS = {
    "CLIP": "openai/clip-vit-base-patch32",
    "SigLIP": "google/siglip-base-patch16-224",
    "SigLIP2": "google/siglip2-base-patch16-224"
}

PROMPT_SETS = {
    "B_experiential": [
        "A person who is feeling pain.",
        "A person who is feeling happy.",
        "A person who is feeling sad.",
        "A person who is feeling angry.",
        "A person who is feeling disgusted.",
        "A person who is feeling surprised."
    ],
    "C_observable": [
        "A person showing visible signs of pain.",
        "A person showing a happy expression.",
        "A person showing a sad expression.",
        "A person showing an angry expression.",
        "A person showing an expression of disgust.",
        "A person showing a surprised expression."
    ]
}

CLASS_NAMES = [
    "pain", "happy", "sad",
    "angry", "disgusted", "surprised"
]

extensions = {".jpg", ".jpeg", ".png", ".jfif", ".webp"}

images = []

for folder, label in [
    ("PainImages", 1),
    ("NoPainImages", 0)
]:
    for path in sorted((IMG_ROOT / folder).rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            images.append((path, label))

assert len(images) == 189, "FPD-189 incompleto!"

print("Imagens:", len(images))

all_results = []

for model_name, model_id in MODELS.items():

    print("\n================================")
    print("Modelo:", model_name)
    print("================================")

    model = AutoModel.from_pretrained(model_id).to(DEVICE)
    processor = AutoProcessor.from_pretrained(model_id)
    model.eval()

    text_inputs = {}

    for strategy, prompts in PROMPT_SETS.items():
        text_inputs[strategy] = processor(
            text=prompts,
            padding="max_length" if model_name != "CLIP" else True,
            return_tensors="pt"
        ).to(DEVICE)

    model_results = []

    with torch.inference_mode():

        for index, (path, true_label) in enumerate(images, 1):

            with Image.open(path) as im:
                image = im.convert("RGB")

            image_inputs = processor(
                images=image,
                return_tensors="pt"
            ).to(DEVICE)

            for strategy in PROMPT_SETS:

                inputs = {
                    **image_inputs,
                    **text_inputs[strategy]
                }

                outputs = model(**inputs)
                scores = outputs.logits_per_image[0]

                predicted_prompt = int(
                    scores.argmax().item()
                )

                prediction = int(
                    predicted_prompt == 0
                )

                record = {
                    "image": str(path.relative_to(IMG_ROOT)),
                    "true_label": true_label,
                    "model": model_id,
                    "model_name": model_name,
                    "strategy": strategy,
                    "predicted_label": prediction,
                    "predicted_prompt": predicted_prompt,
                    "predicted_class": CLASS_NAMES[predicted_prompt],
                    "correct": int(prediction == true_label)
                }

                for j, class_name in enumerate(CLASS_NAMES):
                    record[f"score_{class_name}"] = float(
                        scores[j].item()
                    )

                model_results.append(record)

            if index % 25 == 0:
                print(f"Processadas: {index}/189")

    model_df = pd.DataFrame(model_results)

    filename = OUT / f"{model_name.lower()}_semantic_variants.csv"
    model_df.to_csv(filename, index=False)

    print("Arquivo salvo:", filename)

    all_results.append(model_df)

    del model, processor, text_inputs
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results = pd.concat(all_results, ignore_index=True)

print("\n=== RESULTADOS SEMANTIC VARIANTS ===")

for (model_name, strategy), group in results.groupby(
    ["model_name", "strategy"]
):
    y_true = group.true_label
    y_pred = group.predicted_label

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    print(f"\n{model_name} — {strategy}")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("Specificity:", round(tn / (tn + fp), 4))
    print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Balanced accuracy:",
          round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

results.to_csv(
    OUT / "semantic_variants_all_models.csv",
    index=False
)

with open(OUT / "semantic_prompt_sets.json", "w") as f:
    json.dump(PROMPT_SETS, f, indent=2)

print("\nExperimento concluído.")
print("Resultados salvos em:", OUT)


## Cochran's Q test; McNemar's exact with Holm correction; Prediction stability

Cochran's Q test, appropriate for comparing three binary classifiers evaluated on the same images. Then, McNemar's exact tests to the three paired comparisons of each model, with Holm correction on the nine comparisons. Next, calculate the percentage of images whose classification remained identical across the three sets. This measure will allow us to distinguish between performance stability and the stability of individual decisions.

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from statsmodels.stats.contingency_tables import (
    cochrans_q, mcnemar
)
from statsmodels.stats.multitest import multipletests

OUT = Path("/content/florence_visapp/results")

original_files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv"
}

variant_files = {
    "CLIP": "clip_semantic_variants.csv",
    "SigLIP": "siglip_semantic_variants.csv",
    "SigLIP2": "siglip2_semantic_variants.csv"
}

all_data = []

for model in original_files:

    original = pd.read_csv(
        OUT / original_files[model]
    )

    original = original[
        original["strategy"] == "affective"
    ].copy()

    original["prompt_set"] = "A_original"

    variants = pd.read_csv(
        OUT / variant_files[model]
    ).copy()

    variants["prompt_set"] = variants["strategy"]

    combined = pd.concat(
        [original, variants],
        ignore_index=True
    )

    combined["model_name"] = model
    all_data.append(combined)

data = pd.concat(all_data, ignore_index=True)

sets = [
    "A_original",
    "B_experiential",
    "C_observable"
]

q_results = []
pairwise_results = []
stability_results = []

for model in original_files:

    subset = data[data["model_name"] == model]

    assert not subset.duplicated(
        ["image", "prompt_set"]
    ).any()

    correct = subset.pivot(
        index="image",
        columns="prompt_set",
        values="correct"
    )[sets]

    predictions = subset.pivot(
        index="image",
        columns="prompt_set",
        values="predicted_label"
    )[sets]

    true_labels = subset.pivot(
        index="image",
        columns="prompt_set",
        values="true_label"
    )[sets]

    assert len(correct) == 189
    assert not correct.isna().any().any()
    assert not predictions.isna().any().any()
    assert (true_labels.nunique(axis=1) == 1).all()

    # Q de Cochran: correção por imagem
    q = cochrans_q(correct.to_numpy(dtype=int))

    q_results.append({
        "model": model,
        "cochran_q": q.statistic,
        "p_value": q.pvalue
    })

    # Estabilidade das decisões binárias
    stable = (
        predictions.nunique(axis=1) == 1
    )

    stability_results.append({
        "model": model,
        "stable_images": int(stable.sum()),
        "changed_images": int((~stable).sum()),
        "stability_percent": 100 * stable.mean()
    })

    # McNemar entre conjuntos de prompts
    for first, second in combinations(sets, 2):

        x = correct[first].astype(int)
        y = correct[second].astype(int)

        a = int(((x == 1) & (y == 1)).sum())
        b = int(((x == 1) & (y == 0)).sum())
        c = int(((x == 0) & (y == 1)).sum())
        d = int(((x == 0) & (y == 0)).sum())

        p = mcnemar(
            [[a, b], [c, d]],
            exact=True
        ).pvalue

        pairwise_results.append({
            "model": model,
            "first": first,
            "second": second,
            "only_first_correct": b,
            "only_second_correct": c,
            "difference_pp": 100 * (c - b) / 189,
            "p_raw": p
        })

q_df = pd.DataFrame(q_results)
pair_df = pd.DataFrame(pairwise_results)
stability_df = pd.DataFrame(stability_results)

# Correção de Holm sobre as 9 comparações pareadas
pair_df["p_holm"] = multipletests(
    pair_df["p_raw"],
    method="holm"
)[1]

pair_df["significant"] = pair_df["p_holm"] < 0.05

# Correção de Holm para os 3 testes globais
q_df["p_holm"] = multipletests(
    q_df["p_value"],
    method="holm"
)[1]

print("\n=== TESTE Q DE COCHRAN ===")
print(q_df.to_string(index=False))

print("\n=== COMPARAÇÕES PAREADAS ===")
print(pair_df.to_string(index=False))

print("\n=== ESTABILIDADE DAS PREDIÇÕES ===")
print(stability_df.to_string(index=False))

q_df.to_csv(
    OUT / "semantic_cochran_q.csv",
    index=False
)

pair_df.to_csv(
    OUT / "semantic_mcnemar_holm.csv",
    index=False
)

stability_df.to_csv(
    OUT / "semantic_prediction_stability.csv",
    index=False
)


# Binary Control of Semantic Robustness

The goal is to verify if sensitivity to textual formulation also occurs when we maintain exactly two classification alternatives: Pain and NoPain.

In [ ]:

import gc
import json
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import AutoModel, AutoProcessor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)

BASE = Path("/content/florence_visapp")
IMG_ROOT = BASE / "images"
OUT = BASE / "results"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    "CLIP": "openai/clip-vit-base-patch32",
    "SigLIP": "google/siglip-base-patch16-224",
    "SigLIP2": "google/siglip2-base-patch16-224"
}

PROMPT_SETS = {
    "B_experiential": [
        "A person who is feeling pain.",
        "A person who is not feeling pain."
    ],
    "C_observable": [
        "A person showing visible signs of pain.",
        "A person showing no visible signs of pain."
    ]
}

extensions = {
    ".jpg", ".jpeg", ".png", ".jfif", ".webp"
}

images = []

for folder, label in [
    ("PainImages", 1),
    ("NoPainImages", 0)
]:
    for path in sorted((IMG_ROOT / folder).rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            images.append((path, label))

assert len(images) == 189
assert sum(label == 1 for _, label in images) == 99
assert sum(label == 0 for _, label in images) == 90

print("Dispositivo:", DEVICE)
print("Imagens:", len(images))

all_results = []

for model_name, model_id in MODELS.items():

    print("\n================================")
    print("Modelo:", model_name)
    print("================================")

    model = AutoModel.from_pretrained(
        model_id
    ).to(DEVICE)

    processor = AutoProcessor.from_pretrained(
        model_id
    )

    model.eval()

    text_inputs = {}

    for strategy, prompts in PROMPT_SETS.items():

        text_inputs[strategy] = processor(
            text=prompts,
            padding=(
                True if model_name == "CLIP"
                else "max_length"
            ),
            return_tensors="pt"
        ).to(DEVICE)

    model_results = []

    with torch.inference_mode():

        for index, (path, true_label) in enumerate(
            images, 1
        ):

            with Image.open(path) as im:
                image = im.convert("RGB")

            image_inputs = processor(
                images=image,
                return_tensors="pt"
            ).to(DEVICE)

            for strategy in PROMPT_SETS:

                inputs = {
                    **image_inputs,
                    **text_inputs[strategy]
                }

                outputs = model(**inputs)

                scores = outputs.logits_per_image[0]

                predicted_prompt = int(
                    scores.argmax().item()
                )

                prediction = int(
                    predicted_prompt == 0
                )

                model_results.append({
                    "image": str(
                        path.relative_to(IMG_ROOT)
                    ),
                    "true_label": true_label,
                    "model": model_id,
                    "model_name": model_name,
                    "strategy": strategy,
                    "predicted_label": prediction,
                    "predicted_prompt": predicted_prompt,
                    "pain_score": float(
                        scores[0].item()
                    ),
                    "nopain_score": float(
                        scores[1].item()
                    ),
                    "correct": int(
                        prediction == true_label
                    )
                })

            if index % 25 == 0:
                print(
                    f"Processadas: {index}/{len(images)}"
                )

    model_df = pd.DataFrame(model_results)

    filename = OUT / (
        f"{model_name.lower()}_binary_variants.csv"
    )

    model_df.to_csv(
        filename,
        index=False
    )

    all_results.append(model_df)

    print("Arquivo salvo:", filename)

    del model, processor, text_inputs
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results = pd.concat(
    all_results,
    ignore_index=True
)

print("\n=== RESULTADOS BINARY VARIANTS ===")

for (model_name, strategy), group in results.groupby(
    ["model_name", "strategy"]
):

    y_true = group["true_label"]
    y_pred = group["predicted_label"]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    print(f"\n{model_name} — {strategy}")

    print(
        "Accuracy:",
        round(accuracy_score(y_true, y_pred), 4)
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_true, y_pred,
                zero_division=0
            ), 4
        )
    )

    print(
        "Recall:",
        round(
            recall_score(
                y_true, y_pred,
                zero_division=0
            ), 4
        )
    )

    print(
        "Specificity:",
        round(tn / (tn + fp), 4)
    )

    print(
        "F1:",
        round(
            f1_score(
                y_true, y_pred,
                zero_division=0
            ), 4
        )
    )

    print(
        "Balanced accuracy:",
        round(
            balanced_accuracy_score(
                y_true, y_pred
            ), 4
        )
    )

    print(
        "Confusion matrix:"
    )

    print(
        confusion_matrix(
            y_true, y_pred,
            labels=[0, 1]
        )
    )

results.to_csv(
    OUT / "binary_variants_all_models.csv",
    index=False
)

with open(
    OUT / "binary_prompt_sets.json", "w"
) as f:
    json.dump(
        PROMPT_SETS,
        f,
        indent=2
    )

print("\nExperimento concluído.")
print("Resultados salvos em:", OUT)


## Cochran's Q test; McNemar's exact with Holm correction; Prediction stability

For binary prompts

In [ ]:

import pandas as pd
from pathlib import Path
from itertools import combinations
from statsmodels.stats.contingency_tables import (
    cochrans_q, mcnemar
)
from statsmodels.stats.multitest import multipletests

OUT = Path("/content/florence_visapp/results")

original_files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv"
}

variant_files = {
    "CLIP": "clip_binary_variants.csv",
    "SigLIP": "siglip_binary_variants.csv",
    "SigLIP2": "siglip2_binary_variants.csv"
}

sets = [
    "A_original",
    "B_experiential",
    "C_observable"
]

q_results = []
pair_results = []
stability_results = []

for model in original_files:

    original = pd.read_csv(
        OUT / original_files[model]
    )

    original = original[
        original["strategy"] == "binary"
    ].copy()

    original["prompt_set"] = "A_original"

    variants = pd.read_csv(
        OUT / variant_files[model]
    ).copy()

    variants["prompt_set"] = variants["strategy"]

    df = pd.concat(
        [original, variants],
        ignore_index=True
    )

    assert not df.duplicated(
        ["image", "prompt_set"]
    ).any()

    correct = df.pivot(
        index="image",
        columns="prompt_set",
        values="correct"
    )[sets]

    predictions = df.pivot(
        index="image",
        columns="prompt_set",
        values="predicted_label"
    )[sets]

    labels = df.pivot(
        index="image",
        columns="prompt_set",
        values="true_label"
    )[sets]

    assert len(correct) == 189
    assert not correct.isna().any().any()
    assert not predictions.isna().any().any()
    assert (labels.nunique(axis=1) == 1).all()

    q = cochrans_q(
        correct.to_numpy(dtype=int)
    )

    q_results.append({
        "model": model,
        "cochran_q": q.statistic,
        "p_raw": q.pvalue
    })

    stable = predictions.nunique(axis=1) == 1

    stability_results.append({
        "model": model,
        "stable_images": int(stable.sum()),
        "changed_images": int((~stable).sum()),
        "stability_percent": 100 * stable.mean()
    })

    for first, second in combinations(sets, 2):

        x = correct[first].astype(int)
        y = correct[second].astype(int)

        a = int(((x == 1) & (y == 1)).sum())
        b = int(((x == 1) & (y == 0)).sum())
        c = int(((x == 0) & (y == 1)).sum())
        d = int(((x == 0) & (y == 0)).sum())

        p = mcnemar(
            [[a, b], [c, d]],
            exact=True
        ).pvalue

        pair_results.append({
            "model": model,
            "first": first,
            "second": second,
            "only_first_correct": b,
            "only_second_correct": c,
            "difference_pp": 100 * (c - b) / 189,
            "p_raw": p
        })

q_df = pd.DataFrame(q_results)
pair_df = pd.DataFrame(pair_results)
stability_df = pd.DataFrame(stability_results)

q_df["p_holm"] = multipletests(
    q_df["p_raw"],
    method="holm"
)[1]

pair_df["p_holm"] = multipletests(
    pair_df["p_raw"],
    method="holm"
)[1]

q_df["significant"] = q_df["p_holm"] < 0.05
pair_df["significant"] = pair_df["p_holm"] < 0.05

print("\n=== Q DE COCHRAN — BINARY ===")
print(q_df.to_string(index=False))

print("\n=== MCNEMAR — BINARY ===")
print(pair_df.to_string(index=False))

print("\n=== ESTABILIDADE — BINARY ===")
print(stability_df.to_string(index=False))

q_df.to_csv(
    OUT / "binary_cochran_q.csv",
    index=False
)

pair_df.to_csv(
    OUT / "binary_mcnemar_holm.csv",
    index=False
)

stability_df.to_csv(
    OUT / "binary_prediction_stability.csv",
    index=False
)

print("\nAnálise concluída.")


# Integrated Robustness and Error Analysis by Image

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

OUT = Path("/content/florence_visapp/results")

MODELS = ["CLIP", "SigLIP", "SigLIP2"]
SETS = ["A", "B", "C"]
PROTOCOLS = ["binary", "affective"]

original_files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv"
}

variant_files = {
    "CLIP": {
        "binary": "clip_binary_variants.csv",
        "affective": "clip_semantic_variants.csv"
    },
    "SigLIP": {
        "binary": "siglip_binary_variants.csv",
        "affective": "siglip_semantic_variants.csv"
    },
    "SigLIP2": {
        "binary": "siglip2_binary_variants.csv",
        "affective": "siglip2_semantic_variants.csv"
    }
}

frames = []

for model in MODELS:

    original = pd.read_csv(OUT / original_files[model])

    original = original[
        original["strategy"].isin(PROTOCOLS)
    ].copy()

    original["model_name"] = model
    original["protocol"] = original["strategy"]
    original["prompt_set"] = "A"

    frames.append(original)

    for protocol in PROTOCOLS:

        variants = pd.read_csv(
            OUT / variant_files[model][protocol]
        ).copy()

        variants["model_name"] = model
        variants["protocol"] = protocol

        variants["prompt_set"] = variants["strategy"].map({
            "B_experiential": "B",
            "C_observable": "C"
        })

        assert variants["prompt_set"].notna().all()

        frames.append(variants)

data = pd.concat(frames, ignore_index=True)

# Verificações de integridade
assert len(data) == 189 * 3 * 2 * 3

assert not data.duplicated(
    ["image", "model_name", "protocol", "prompt_set"]
).any()

assert data.groupby("image")["true_label"].nunique().eq(1).all()

assert data.groupby(
    ["model_name", "protocol", "prompt_set"]
).size().eq(189).all()

assert (
    data["correct"].astype(int)
    == (
        data["predicted_label"].astype(int)
        == data["true_label"].astype(int)
    ).astype(int)
).all()

data["condition"] = (
    data["model_name"]
    + "_" + data["protocol"]
    + "_" + data["prompt_set"]
)

predictions = data.pivot(
    index="image",
    columns="condition",
    values="predicted_label"
).sort_index()

correct = data.pivot(
    index="image",
    columns="condition",
    values="correct"
).sort_index()

labels = data.groupby("image")["true_label"].first()

assert predictions.shape == (189, 18)
assert not predictions.isna().any().any()
assert not correct.isna().any().any()

# ==========================================
# 1. ESTABILIDADE POR MODELO E PROTOCOLO
# ==========================================

stability = pd.DataFrame(index=predictions.index)

for model in MODELS:
    for protocol in PROTOCOLS:

        cols = [
            f"{model}_{protocol}_{s}"
            for s in SETS
        ]

        stability[f"{model}_{protocol}"] = (
            predictions[cols].nunique(axis=1) == 1
        ).astype(int)

# ==========================================
# 2. MCNEMAR PAREADO: BINARY VS AFFECTIVE
# ==========================================

comparison_results = []

for model in MODELS:

    x = stability[f"{model}_binary"]
    y = stability[f"{model}_affective"]

    both_stable = int(((x == 1) & (y == 1)).sum())
    only_binary = int(((x == 1) & (y == 0)).sum())
    only_affective = int(((x == 0) & (y == 1)).sum())
    neither_stable = int(((x == 0) & (y == 0)).sum())

    p = mcnemar(
        [
            [both_stable, only_binary],
            [only_affective, neither_stable]
        ],
        exact=True
    ).pvalue

    comparison_results.append({
        "model": model,
        "binary_stability_percent": 100 * x.mean(),
        "affective_stability_percent": 100 * y.mean(),
        "gain_pp": 100 * (y.mean() - x.mean()),
        "only_binary_stable": only_binary,
        "only_affective_stable": only_affective,
        "p_raw": p
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df["p_holm"] = multipletests(
    comparison_df["p_raw"],
    method="holm"
)[1]

comparison_df["significant"] = (
    comparison_df["p_holm"] < 0.05
)

print("\n=== ESTABILIDADE: BINARY VS AFFECTIVE ===")
print(comparison_df.to_string(index=False))

# ==========================================
# 3. ANÁLISE DE ERROS POR IMAGEM
# ==========================================

image_analysis = pd.DataFrame(
    index=predictions.index
)

image_analysis["true_label"] = labels

image_analysis["total_correct"] = (
    correct.sum(axis=1)
)

image_analysis["total_errors"] = (
    18 - image_analysis["total_correct"]
)

image_analysis["correct_binary"] = correct[
    [c for c in correct.columns if "_binary_" in c]
].sum(axis=1)

image_analysis["correct_affective"] = correct[
    [c for c in correct.columns if "_affective_" in c]
].sum(axis=1)

image_analysis["false_negatives"] = np.where(
    image_analysis["true_label"] == 1,
    image_analysis["total_errors"],
    0
)

image_analysis["false_positives"] = np.where(
    image_analysis["true_label"] == 0,
    image_analysis["total_errors"],
    0
)

image_analysis["all_18_correct"] = (
    image_analysis["total_correct"] == 18
)

image_analysis["all_18_wrong"] = (
    image_analysis["total_correct"] == 0
)

image_analysis["all_affective_correct"] = (
    image_analysis["correct_affective"] == 9
)

image_analysis["all_binary_correct"] = (
    image_analysis["correct_binary"] == 9
)

image_analysis["classification_changes"] = (
    6 - stability.sum(axis=1)
)

# Quantas das seis combinações modelo/protocolo
# apresentam mudança de decisão entre A, B e C.

image_analysis = image_analysis.sort_values(
    ["total_errors", "classification_changes"],
    ascending=False
)

print("\n=== RESUMO POR IMAGEM ===")
print("Imagens corretas nas 18 configurações:",
      int(image_analysis["all_18_correct"].sum()))

print("Imagens erradas nas 18 configurações:",
      int(image_analysis["all_18_wrong"].sum()))

print("Imagens corretas nas 9 configurações afetivas:",
      int(image_analysis["all_affective_correct"].sum()))

print("Imagens corretas nas 9 configurações binárias:",
      int(image_analysis["all_binary_correct"].sum()))

print("\n=== 15 IMAGENS COM MAIS ERROS ===")
print(image_analysis.head(15).to_string())

# ==========================================
# 4. ERROS PERSISTENTES POR MODELO
# ==========================================

persistent_results = []

for model in MODELS:
    for protocol in PROTOCOLS:

        cols = [
            f"{model}_{protocol}_{s}"
            for s in SETS
        ]

        all_wrong = (
            correct[cols].sum(axis=1) == 0
        )

        pain = labels == 1
        nopain = labels == 0

        persistent_results.append({
            "model": model,
            "protocol": protocol,
            "persistent_false_negatives": int(
                (all_wrong & pain).sum()
            ),
            "persistent_false_positives": int(
                (all_wrong & nopain).sum()
            ),
            "persistent_errors_total": int(
                all_wrong.sum()
            )
        })

persistent_df = pd.DataFrame(persistent_results)

print("\n=== ERROS PERSISTENTES ===")
print(persistent_df.to_string(index=False))

# ==========================================
# SALVAR ARQUIVOS
# ==========================================

comparison_df.to_csv(
    OUT / "integrated_stability_mcnemar.csv",
    index=False
)

image_analysis.to_csv(
    OUT / "integrated_image_error_analysis.csv",
    index_label="image"
)

persistent_df.to_csv(
    OUT / "integrated_persistent_errors.csv",
    index=False
)

stability.to_csv(
    OUT / "integrated_image_stability.csv",
    index_label="image"
)

print("\nAnálise integrada concluída.")
print("Arquivos salvos em:", OUT)


# Audit of Human Assessments of FPD-189

Checking out whether the images that where unstable along the predictions of the three models above were also the ones with weaker consensus in the human labeling.

In [ ]:
%pip install -q odfpy

In [ ]:

from pathlib import Path
import pandas as pd


ROOT = Path("/content/florence_visapp")

# Localizar o arquivo de validação humana
files = list(ROOT.rglob("form_answers_in_portuguese.ods"))

if not files:
    raise FileNotFoundError(
        "Arquivo form_answers_in_portuguese.ods não encontrado."
    )

file_path = files[0]

print("=== ARQUIVO ENCONTRADO ===")
print(file_path)

# Abrir todas as abas, preservando valores originais
xls = pd.ExcelFile(file_path, engine="odf")

print("\n=== ABAS DISPONÍVEIS ===")
print(xls.sheet_names)

for sheet_name in xls.sheet_names:

    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine="odf",
        dtype=str
    )

    print("\n" + "=" * 70)
    print("ABA:", sheet_name)
    print("=" * 70)

    print("Linhas:", len(df))
    print("Colunas:", len(df.columns))

    print("\n=== NOMES DAS COLUNAS ===")
    for i, col in enumerate(df.columns):
        print(f"{i:03d}: {repr(col)}")

    print("\n=== PRIMEIRAS 5 LINHAS ===")
    print(
        df.head(5).to_string(
            index=False,
            max_cols=15,
            max_colwidth=60
        )
    )

    print("\n=== VALORES NÃO NULOS POR COLUNA ===")
    print(
        df.notna().sum().to_string()
    )

print("\nInspeção estrutural concluída.")


# Consolidation of human consensus

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

ROOT = Path("/content/florence_visapp")
OUT = ROOT / "results"
ODS = ROOT / "images/form_answers_in_portuguese.ods"
OUT.mkdir(parents=True, exist_ok=True)

# ==========================================================
# 1. IMAGENS DO FPD-189
# ==========================================================

pred = pd.read_csv(OUT / "clip_vit_b32_predictions.csv")

images = pred[["image", "true_label"]].drop_duplicates().copy()

# Aceita image_123 e imagem_123, independentemente da extensão
images["image_number"] = pd.to_numeric(
    images["image"].astype(str).str.extract(
        r"(?:image|imagem)_(\d+)\.[^.]+$",
        flags=re.IGNORECASE,
        expand=False
    ),
    errors="coerce"
)

assert len(images) == 189, "Quantidade inesperada de imagens."
assert images["image_number"].notna().all(), \
    "Existem imagens sem número reconhecido."
assert images["image_number"].is_unique, \
    "Existem números de imagem duplicados."
assert images["true_label"].isin([0, 1]).all(), \
    "Rótulos inesperados."

images["image_number"] = images["image_number"].astype(int)

print("=== IMAGENS DO FPD-189 ===")
print("Total:", len(images))
print("Pain:", int((images.true_label == 1).sum()))
print("NoPain:", int((images.true_label == 0).sum()))

# ==========================================================
# 2. EXTRAÇÃO DAS RESPOSTAS HUMANAS
# ==========================================================

def normalize(value):
    value = str(value).strip().lower()
    return "".join(
        c for c in unicodedata.normalize("NFD", value)
        if unicodedata.category(c) != "Mn"
    )

def classify_answer(value):
    if value == "dor":
        return "Pain"
    if value.startswith("nao e possivel determinar"):
        return "Indeterminate"
    if value in {
        "felicidade", "tristeza", "raiva", "nojo",
        "surpresa", "neutro",
        "neutro (a pessoa nao esboca qualquer sentimento)"
    }:
        return "NoPain"
    return "Unrecognized"

records = []
xls = pd.ExcelFile(ODS, engine="odf")

for sheet in xls.sheet_names:
    df = pd.read_excel(
        ODS, sheet_name=sheet,
        engine="odf", dtype=str
    )

    for row_idx, row in df.iterrows():
        for col in df.columns:
            match = re.match(r"^Imagem\s+(\d+)\s*:", str(col))
            if match is None or pd.isna(row[col]):
                continue

            records.append({
                "form": sheet,
                "submission_row": row_idx + 1,
                "timestamp": row.iloc[0],
                "image_number": int(match.group(1)),
                "human_answer": str(row[col]).strip()
            })

human = pd.DataFrame(records)

human["answer_normalized"] = human["human_answer"].map(
    normalize
)

human["binary_answer"] = human["answer_normalized"].map(
    classify_answer
)

# ==========================================================
# 3. CORRESPONDÊNCIA COM O FPD-189
# ==========================================================

human = human.merge(
    images,
    on="image_number",
    how="left",
    validate="many_to_one",
    indicator=True
)

included = human[human["_merge"] == "both"].copy()

included["reference_class"] = included["true_label"].map({
    1: "Pain",
    0: "NoPain"
})

included["valid_binary"] = included["binary_answer"].isin(
    ["Pain", "NoPain"]
)

included["agrees_with_reference"] = np.where(
    included["valid_binary"],
    included["binary_answer"] == included["reference_class"],
    np.nan
)

# ==========================================================
# 4. CONSENSO POR IMAGEM
# ==========================================================

summary = []

for _, img in images.iterrows():

    subset = included[
        included["image_number"] == img["image_number"]
    ]

    n = len(subset)
    counts = subset["human_answer"].value_counts()

    n_pain = int((subset.binary_answer == "Pain").sum())
    n_nopain = int((subset.binary_answer == "NoPain").sum())
    n_ind = int(
        (subset.binary_answer == "Indeterminate").sum()
    )
    n_unknown = int(
        (subset.binary_answer == "Unrecognized").sum()
    )

    n_valid = n_pain + n_nopain

    if len(counts) == 0:
        majority = "No_data"
    elif len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        majority = "Tie"
    else:
        majority = counts.index[0]

    summary.append({
        "image": img["image"],
        "image_number": img["image_number"],
        "true_label": img["true_label"],
        "n_human": n,
        "n_pain": n_pain,
        "n_nopain": n_nopain,
        "n_indeterminate": n_ind,
        "n_unrecognized": n_unknown,
        "n_valid_binary": n_valid,
        "pain_vote_fraction_all":
            n_pain / n if n else np.nan,
        "pain_vote_fraction_valid":
            n_pain / n_valid if n_valid else np.nan,
        "reference_agreement_fraction":
            subset["agrees_with_reference"].mean()
            if n_valid else np.nan,
        "multiclass_modal_fraction":
            counts.iloc[0] / n if n else np.nan,
        "multiclass_majority_answer": majority,
        "binary_tie":
            bool(n_valid > 0 and n_pain == n_nopain)
    })

summary = pd.DataFrame(summary).sort_values("image_number")

# ==========================================================
# 5. SALVAR RESULTADOS
# ==========================================================

human.drop(columns="_merge").to_csv(
    OUT / "human_all_original_responses.csv",
    index=False
)

included.drop(columns="_merge").to_csv(
    OUT / "human_fpd189_individual_responses.csv",
    index=False
)

summary.to_csv(
    OUT / "human_fpd189_consensus.csv",
    index=False
)

# ==========================================================
# 6. RELATÓRIO
# ==========================================================

print("\n=== COBERTURA HUMANA ===")
print("Respostas originais:", len(human))
print("Respostas associadas ao FPD-189:", len(included))
print("Imagens com avaliações:",
      int((summary.n_human > 0).sum()))
print("Imagens sem avaliações:",
      int((summary.n_human == 0).sum()))

print("\n=== AVALIAÇÕES POR IMAGEM ===")
print(summary.n_human.describe().to_string())

print("\n=== CATEGORIAS DE RESPOSTA ===")
print(included.binary_answer.value_counts().to_string())

print("\n=== CONCORDÂNCIA COM O RÓTULO DE REFERÊNCIA ===")
print(
    summary.groupby("true_label")[
        "reference_agreement_fraction"
    ].agg(["count", "mean", "median", "min", "max"])
    .to_string()
)

print("\n=== 15 IMAGENS COM MENOR CONCORDÂNCIA ===")
print(
    summary.sort_values(
        "reference_agreement_fraction"
    ).head(15)[[
        "image_number", "true_label", "n_human",
        "n_pain", "n_nopain", "n_indeterminate",
        "reference_agreement_fraction"
    ]].to_string(index=False)
)

print("\n=== IMAGENS COM MAIS ERROS DOS MODELOS ===")
priority = [166, 175, 1, 179, 106, 66, 167, 169, 103]

print(
    summary[
        summary.image_number.isin(priority)
    ][[
        "image_number", "true_label", "n_human",
        "n_pain", "n_nopain", "n_indeterminate",
        "reference_agreement_fraction"
    ]].to_string(index=False)
)

print("\n=== CONTROLES DE QUALIDADE ===")
print("Respostas não reconhecidas:",
      int((included.binary_answer == "Unrecognized").sum()))
print("Empates binários:",
      int(summary.binary_tie.sum()))
print("Imagens sem votos binários válidos:",
      int((summary.n_valid_binary == 0).sum()))

print("\n=== ARQUIVOS GERADOS ===")
print("human_all_original_responses.csv")
print("human_fpd189_individual_responses.csv")
print("human_fpd189_consensus.csv")

print("\nConsolidação concluída.")


# The relationship between human consensus and model errors.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

ROOT = Path("/content/florence_visapp/results")

human = pd.read_csv(
    ROOT / "human_fpd189_consensus.csv"
)

errors = pd.read_csv(
    ROOT / "integrated_image_error_analysis.csv"
)

# Associação exata pelo identificador de imagem
df = human.merge(
    errors[["image", "total_errors"]],
    on="image",
    how="inner",
    validate="one_to_one"
)

assert len(df) == 189, (
    f"Correspondência incompleta: {len(df)} imagens."
)

df["human_disagreement"] = (
    1 - df["reference_agreement_fraction"]
)

df["indeterminate_fraction"] = (
    df["n_indeterminate"] / df["n_human"]
)

df["perfect_valid_agreement"] = (
    df["reference_agreement_fraction"] == 1
)

print("=== CORRELAÇÕES DE SPEARMAN ===")

variables = [
    "human_disagreement",
    "indeterminate_fraction",
    "reference_agreement_fraction"
]

for var in variables:
    rho, p = spearmanr(
        df[var],
        df["total_errors"]
    )
    print(
        f"{var}: rho={rho:.4f}, p={p:.6g}"
    )

print("\n=== GRUPOS DE CONCORDÂNCIA HUMANA ===")

groups = df.groupby("perfect_valid_agreement")[
    "total_errors"
].agg(["count", "mean", "median", "std", "min", "max"])

print(groups.to_string())

# Teste de permutação bilateral da diferença de médias
rng = np.random.default_rng(42)

a = df.loc[
    df["perfect_valid_agreement"],
    "total_errors"
].to_numpy()

b = df.loc[
    ~df["perfect_valid_agreement"],
    "total_errors"
].to_numpy()

observed = b.mean() - a.mean()

combined = np.concatenate([a, b])
n_a = len(a)
B = 20000

extreme = 0

for _ in range(B):
    shuffled = rng.permutation(combined)
    diff = (
        shuffled[n_a:].mean()
        - shuffled[:n_a].mean()
    )
    if abs(diff) >= abs(observed):
        extreme += 1

p_perm = (extreme + 1) / (B + 1)

print("\n=== TESTE DE PERMUTAÇÃO ===")
print("Diferença de médias (discordância - perfeita):",
      round(observed, 4))
print("p bilateral:", round(p_perm, 6))

print("\n=== IMAGENS COM MAIS ERROS ===")
print(
    df.sort_values(
        "total_errors", ascending=False
    ).head(20)[[
        "image_number",
        "true_label",
        "n_human",
        "reference_agreement_fraction",
        "n_indeterminate",
        "total_errors"
    ]].to_string(index=False)
)

df.to_csv(
    ROOT / "human_model_error_association.csv",
    index=False
)

print("\nArquivo salvo:")
print("human_model_error_association.csv")


# Class-controlled association

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path("/content/florence_visapp/results")

df = pd.read_csv(
    ROOT / "human_model_error_association.csv"
)

rng = np.random.default_rng(42)

print("=== DISTRIBUIÇÃO POR CLASSE ===")

df["human_disagreement"] = (
    1 - df["reference_agreement_fraction"]
)

df["has_disagreement"] = (
    df["human_disagreement"] > 0
)

print(
    pd.crosstab(
        df["true_label"],
        df["has_disagreement"],
        margins=True
    ).to_string()
)

print("\n=== CORRELAÇÕES POR CLASSE ===")

for label, name in [(0, "NoPain"), (1, "Pain")]:

    subset = df[df.true_label == label]

    for variable in [
        "human_disagreement",
        "indeterminate_fraction"
    ]:
        rho, p = spearmanr(
            subset[variable],
            subset["total_errors"]
        )

        print(
            f"{name} | {variable}: "
            f"n={len(subset)}, "
            f"rho={rho:.4f}, p={p:.6g}"
        )

print("\n=== ERROS MÉDIOS POR CLASSE E DISCORDÂNCIA ===")

print(
    df.groupby(
        ["true_label", "has_disagreement"]
    )["total_errors"]
    .agg(["count", "mean", "median", "std"])
    .to_string()
)

# Estatística: diferença de médias com ajuste
# pela distribuição das classes (média ponderada).
def stratified_effect(data, indicator):
    effects = []
    weights = []

    for label, group in data.groupby("true_label"):
        a = group.loc[
            group[indicator],
            "total_errors"
        ]
        b = group.loc[
            ~group[indicator],
            "total_errors"
        ]

        if len(a) == 0 or len(b) == 0:
            continue

        effects.append(a.mean() - b.mean())
        weights.append(len(group))

    if not effects:
        return np.nan

    return np.average(effects, weights=weights)

observed = stratified_effect(
    df, "has_disagreement"
)

# Permutar o indicador dentro de cada classe,
# preservando o número de imagens discordantes.
B = 20000
null = np.empty(B)

for i in range(B):
    permuted = df.copy()

    for label in [0, 1]:
        mask = permuted.true_label == label
        values = permuted.loc[
            mask, "has_disagreement"
        ].to_numpy()

        permuted.loc[
            mask, "has_disagreement"
        ] = rng.permutation(values)

    null[i] = stratified_effect(
        permuted, "has_disagreement"
    )

p_value = (
    (np.abs(null) >= abs(observed)).sum() + 1
) / (B + 1)

print("\n=== TESTE ESTRATIFICADO POR CLASSE ===")
print("Diferença ajustada de erros:",
      round(observed, 4))
print("p de permutação bilateral:",
      round(p_value, 6))

print("\n=== IMAGENS COM DISCORDÂNCIA HUMANA ===")

print(
    df[df.has_disagreement]
    .sort_values(
        "total_errors", ascending=False
    )[[
        "image_number",
        "true_label",
        "n_human",
        "reference_agreement_fraction",
        "n_indeterminate",
        "total_errors"
    ]].to_string(index=False)
)

df.to_csv(
    ROOT / "human_model_error_class_adjusted.csv",
    index=False
)

print("\nArquivo salvo:")
print("human_model_error_class_adjusted.csv")


# Robustness of the human-model association

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path("/content/florence_visapp/results")
df = pd.read_csv(
    ROOT / "human_model_error_class_adjusted.csv"
)

rng = np.random.default_rng(2026)

df["has_disagreement"] = (
    df["reference_agreement_fraction"] < 1
)

df["has_indeterminate"] = (
    df["n_indeterminate"] > 0
)

def adjusted_difference(data, indicator):
    effects, weights = [], []

    for label, g in data.groupby("true_label"):
        a = g.loc[g[indicator], "total_errors"]
        b = g.loc[~g[indicator], "total_errors"]

        if len(a) == 0 or len(b) == 0:
            return np.nan

        effects.append(a.mean() - b.mean())
        weights.append(len(g))

    return np.average(effects, weights=weights)

# ==================================================
# 1. BOOTSTRAP ESTRATIFICADO POR CLASSE E GRUPO
# ==================================================

observed = adjusted_difference(
    df, "has_disagreement"
)

B = 10000
boot = np.empty(B)

groups = [
    g for _, g in df.groupby(
        ["true_label", "has_disagreement"]
    )
]

for i in range(B):
    samples = [
        g.iloc[
            rng.integers(0, len(g), size=len(g))
        ]
        for g in groups
    ]

    sample = pd.concat(samples, ignore_index=True)

    boot[i] = adjusted_difference(
        sample, "has_disagreement"
    )

ci = np.percentile(boot, [2.5, 97.5])

print("=== BOOTSTRAP ESTRATIFICADO ===")
print(f"Efeito ajustado: {observed:.4f}")
print(f"IC 95%: [{ci[0]:.4f}, {ci[1]:.4f}]")

# ==================================================
# 2. SENSIBILIDADE LEAVE-ONE-OUT
# ==================================================

discordant = df[df.has_disagreement]
loo = []

for _, row in discordant.iterrows():

    reduced = df[
        df.image_number != row.image_number
    ]

    effect = adjusted_difference(
        reduced, "has_disagreement"
    )

    loo.append({
        "removed_image": row.image_number,
        "true_label": row.true_label,
        "removed_errors": row.total_errors,
        "adjusted_effect": effect
    })

loo = pd.DataFrame(loo)

print("\n=== LEAVE-ONE-OUT ===")
print(
    loo.sort_values("adjusted_effect")
    .to_string(index=False)
)

print("\nIntervalo de efeitos leave-one-out:")
print(
    round(loo.adjusted_effect.min(), 4),
    "a",
    round(loo.adjusted_effect.max(), 4)
)

# ==================================================
# 3. INDETERMINAÇÃO HUMANA
# ==================================================

print("\n=== RESPOSTAS INDETERMINADAS ===")

print(
    pd.crosstab(
        df.true_label,
        df.has_indeterminate,
        margins=True
    ).to_string()
)

print("\nErros médios segundo indeterminação:")
print(
    df.groupby(
        ["true_label", "has_indeterminate"]
    )["total_errors"]
    .agg(["count", "mean", "median"])
    .to_string()
)

for label, name in [(0, "NoPain"), (1, "Pain")]:
    g = df[df.true_label == label]

    rho, p = spearmanr(
        g["indeterminate_fraction"],
        g["total_errors"]
    )

    print(
        f"{name}: rho={rho:.4f}, p={p:.6g}"
    )

# ==================================================
# 4. SALVAR
# ==================================================

pd.DataFrame([{
    "adjusted_effect": observed,
    "bootstrap_ci95_lower": ci[0],
    "bootstrap_ci95_upper": ci[1],
    "loo_min": loo.adjusted_effect.min(),
    "loo_max": loo.adjusted_effect.max()
}]).to_csv(
    ROOT / "human_model_robustness_summary.csv",
    index=False
)

loo.to_csv(
    ROOT / "human_model_leave_one_out.csv",
    index=False
)

print("\nArquivos salvos:")
print("human_model_robustness_summary.csv")
print("human_model_leave_one_out.csv")


# Testing BLIP-2 OPT-2.7B zero-shot

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 10
# BLIP-2 OPT-2.7B | ZERO-SHOT | FPD-189
# VERSÃO CORRIGIDA — EXECUÇÃO COMPLETA
# ============================================================

import sys
import subprocess
import importlib.util

for package, module in [
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("pillow", "PIL"),
    ("scikit-learn", "sklearn"),
    ("pandas", "pandas"),
    ("sentencepiece", "sentencepiece")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip",
            "install", "-q", package
        ])

import re
import gc
import json
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

ROOT = Path("/content/florence_visapp")
IMAGES = ROOT / "images"
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Salesforce/blip2-opt-2.7b"

# Arquivos próprios da execução corrigida.
# Os resultados anteriores permanecem preservados.

PRED_FILE = OUT / "blip2_opt27b_predictions_v2.csv"
METRICS_FILE = OUT / "blip2_opt27b_metrics_v2.csv"
META_FILE = OUT / "blip2_opt27b_protocol_v2.json"

PROMPT = (
    "Question: Does this person show visible signs "
    "of pain? Answer only yes or no. Answer:"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

dtype = (
    torch.float16 if device == "cuda"
    else torch.float32
)

print("=== FLORENCE / BLIP-2 ===")
print("Modelo:", MODEL_ID)
print("Dispositivo:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: execução em CPU pode ser demorada.")

# ============================================================
# 2. IDENTIFICAR AS 189 IMAGENS
# ============================================================

reference = pd.read_csv(
    OUT / "clip_vit_b32_predictions.csv"
)

images = reference[
    ["image", "true_label"]
].drop_duplicates().copy()

images["image_number"] = pd.to_numeric(
    images["image"].astype(str).str.extract(
        r"(?:image|imagem)_(\d+)\.[^.]+$",
        flags=re.IGNORECASE,
        expand=False
    ),
    errors="coerce"
)

assert len(images) == 189
assert images["image_number"].notna().all()
assert images["image_number"].is_unique
assert images["true_label"].isin([0, 1]).all()

images["image_number"] = (
    images["image_number"].astype(int)
)

images = images.sort_values(
    "image_number"
).reset_index(drop=True)

def resolve_image(old_path):

    path = IMAGES / old_path

    if path.is_file():
        return path

    normalized = re.sub(
        r"(?i)imagem_(?=\d+\.)",
        "image_",
        old_path
    )

    path = IMAGES / normalized

    if path.is_file():
        return path

    raise FileNotFoundError(
        f"Imagem não encontrada: {old_path}"
    )

images["resolved_path"] = images["image"].map(
    lambda p: str(resolve_image(p))
)

print("\nImagens:", len(images))
print("Pain:", int((images.true_label == 1).sum()))
print("NoPain:", int((images.true_label == 0).sum()))

# ============================================================
# 3. CARREGAR BLIP-2
# ============================================================

print("\nCarregando processador...")

processor = Blip2Processor.from_pretrained(
    MODEL_ID
)

print("Carregando modelo...")

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)

model = model.to(device)
model.eval()

print("Modelo carregado.")

# ============================================================
# 4. CORREÇÃO DA EXTRAÇÃO DE RESPOSTAS
# ============================================================

def extract_answer(raw_text):

    if raw_text is None:
        return ""

    text = str(raw_text).strip()

    # BLIP-2 OPT pode devolver o prompt completo
    # seguido da resposta. Isolamos a continuação.
    parts = re.split(
        r"\bAnswer\s*:",
        text,
        flags=re.IGNORECASE
    )

    if len(parts) > 1:
        return parts[-1].strip()

    return text

def parse_answer(answer):

    text = answer.strip().lower()

    text = re.sub(
        r"^[^a-z]+", "", text
    )

    if re.match(r"^yes\b", text):
        return 1

    if re.match(r"^no\b", text):
        return 0

    return None

# ============================================================
# 5. RETOMADA AUTOMÁTICA DA VERSÃO 2
# ============================================================

columns = [
    "image",
    "image_number",
    "true_label",
    "model",
    "strategy",
    "prompt",
    "raw_answer",
    "extracted_answer",
    "predicted_label",
    "valid_response",
    "correct",
    "inference_seconds"
]

if PRED_FILE.exists():

    existing = pd.read_csv(PRED_FILE)

    if not set(columns).issubset(existing.columns):
        raise ValueError(
            "Arquivo V2 incompatível."
        )

    if not (
        existing["model"].eq(MODEL_ID).all()
        and existing["prompt"].eq(PROMPT).all()
    ):
        raise ValueError(
            "Arquivo V2 pertence a outro protocolo."
        )

    if existing["image_number"].duplicated().any():
        raise ValueError(
            "Arquivo V2 contém imagens duplicadas."
        )

    completed = set(
        existing["image_number"].astype(int)
    )

    print(
        "\nRetomando versão 2:",
        len(completed),
        "imagens já processadas."
    )

else:
    completed = set()

# ============================================================
# 6. INFERÊNCIA COMPLETA
# ============================================================

start_all = time.time()

for _, row in images.iterrows():

    number = int(row.image_number)

    if number in completed:
        continue

    start = time.time()

    with Image.open(row.resolved_path) as im:
        image = im.convert("RGB")

    inputs = processor(
        images=image,
        text=PROMPT,
        return_tensors="pt"
    )

    inputs = {
        k: (
            v.to(device=device, dtype=dtype)
            if torch.is_floating_point(v)
            else v.to(device)
        )
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        generated = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            num_beams=1
        )

    raw_answer = processor.batch_decode(
        generated,
        skip_special_tokens=True
    )[0].strip()

    extracted_answer = extract_answer(
        raw_answer
    )

    predicted = parse_answer(
        extracted_answer
    )

    elapsed = time.time() - start

    record = {
        "image": row.image,
        "image_number": number,
        "true_label": int(row.true_label),
        "model": MODEL_ID,
        "strategy": "generative_binary_zero_shot",
        "prompt": PROMPT,
        "raw_answer": raw_answer,
        "extracted_answer": extracted_answer,
        "predicted_label": predicted,
        "valid_response": predicted is not None,
        "correct": (
            int(predicted == int(row.true_label))
            if predicted is not None
            else np.nan
        ),
        "inference_seconds": elapsed
    }

    pd.DataFrame(
        [record],
        columns=columns
    ).to_csv(
        PRED_FILE,
        mode="a",
        header=not PRED_FILE.exists(),
        index=False
    )

    completed.add(number)

    print(
        f"[{len(completed):03d}/189] "
        f"image_{number} | "
        f"real={row.true_label} | "
        f"resposta={extracted_answer!r} | "
        f"pred={predicted} | "
        f"{elapsed:.2f}s",
        flush=True
    )

    del inputs, generated, image
    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()

# ============================================================
# 7. AUDITORIA DAS RESPOSTAS
# ============================================================

results = pd.read_csv(PRED_FILE)

assert len(results) == 189
assert results["image_number"].nunique() == 189
assert not results["image_number"].duplicated().any()

valid = results[
    results["valid_response"]
    .astype(str).str.lower() == "true"
].copy()

invalid = results[
    results["valid_response"]
    .astype(str).str.lower() != "true"
].copy()

coverage = len(valid) / len(results)

print("\n=== COBERTURA ===")
print("Respostas válidas:", len(valid))
print("Respostas inválidas:", len(invalid))
print(f"Cobertura: {coverage:.2%}")

print("\n=== DISTRIBUIÇÃO DAS PREDIÇÕES ===")
print(
    valid["predicted_label"]
    .astype(int)
    .map({0: "NoPain", 1: "Pain"})
    .value_counts()
    .to_string()
)

print("\n=== RESPOSTAS MAIS FREQUENTES ===")
print(
    results["extracted_answer"]
    .str.lower()
    .value_counts()
    .head(15)
    .to_string()
)

if len(invalid):
    print("\nRespostas não reconhecidas:")
    print(
        invalid[[
            "image_number",
            "extracted_answer"
        ]].to_string(index=False)
    )

# ============================================================
# 8. MÉTRICAS
# ============================================================

if len(valid) == 0:
    raise ValueError(
        "Nenhuma resposta binária válida."
    )

y_true = valid["true_label"].astype(int)
y_pred = valid["predicted_label"].astype(int)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

metrics = {
    "model": MODEL_ID,
    "protocol": "generative_binary_zero_shot",
    "n_images": 189,
    "n_valid": len(valid),
    "n_invalid": len(invalid),
    "coverage": coverage,
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(
        y_true, y_pred, zero_division=0
    ),
    "recall": recall_score(
        y_true, y_pred, zero_division=0
    ),
    "specificity": (
        tn / (tn + fp)
        if tn + fp else np.nan
    ),
    "f1": f1_score(
        y_true, y_pred, zero_division=0
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_true, y_pred
    ),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp)
}

pd.DataFrame([metrics]).to_csv(
    METRICS_FILE,
    index=False
)

# ============================================================
# 9. METADADOS DO EXPERIMENTO
# ============================================================

with open(META_FILE, "w") as f:
    json.dump({
        "model": MODEL_ID,
        "dataset": "FPD-189",
        "n_images": 189,
        "prompt": PROMPT,
        "generation": {
            "max_new_tokens": 8,
            "do_sample": False,
            "num_beams": 1
        },
        "device": device,
        "dtype": str(dtype),
        "zero_shot": True,
        "response_extraction": (
            "Text after final Answer: marker"
        )
    }, f, indent=2)

# ============================================================
# 10. RESULTADOS FINAIS
# ============================================================

print("\n=== RESULTADOS BLIP-2 ===")

for key in [
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy"
]:
    print(f"{key}: {metrics[key]:.4f}")

print("\n=== MATRIZ DE CONFUSÃO ===")
print("Linhas: NoPain, Pain")
print("Colunas: NoPain, Pain")
print(cm)

print("\nTempo desta execução:",
      round((time.time() - start_all) / 60, 2),
      "minutos")

print("\n=== ARQUIVOS SALVOS ===")
print(PRED_FILE.name)
print(METRICS_FILE.name)
print(META_FILE.name)

print("\nEXPERIMENTO 10 CONCLUÍDO.")


=== FLORENCE / BLIP-2 ===
Modelo: Salesforce/blip2-opt-2.7b
Dispositivo: cuda
GPU: Tesla T4

Imagens: 189
Pain: 99
NoPain: 90

Carregando processador...
Carregando modelo...


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Modelo carregado.
[001/189] image_1 | real=1 | resposta='yes' | pred=1 | 2.00s
[002/189] image_2 | real=0 | resposta='Yes' | pred=1 | 0.14s
[003/189] image_3 | real=0 | resposta='No' | pred=0 | 0.13s
[004/189] image_4 | real=1 | resposta='yes' | pred=1 | 0.14s
[005/189] image_5 | real=0 | resposta='yes' | pred=1 | 0.14s
[006/189] image_6 | real=0 | resposta='yes' | pred=1 | 0.12s
[007/189] image_7 | real=0 | resposta='Yes' | pred=1 | 0.12s
[008/189] image_8 | real=0 | resposta='yes' | pred=1 | 0.11s
[009/189] image_9 | real=0 | resposta='Yes' | pred=1 | 0.15s
[010/189] image_10 | real=0 | resposta='Yes' | pred=1 | 0.12s
[011/189] image_11 | real=0 | resposta='yes' | pred=1 | 0.14s
[012/189] image_12 | real=0 | resposta='yes' | pred=1 | 0.12s
[013/189] image_13 | real=1 | resposta='yes' | pred=1 | 0.14s
[014/189] image_14 | real=1 | resposta='yes' | pred=1 | 0.16s
[015/189] image_15 | real=0 | resposta='yes' | pred=1 | 0.11s
[016/189] image_16 | real=1 | resposta='yes' | pred=1 | 0.14s


BLIP-2 sensitivity to task formulation

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTOS 10B E 10C
# BLIP-2 OPT-2.7B | ZERO-SHOT | FPD-189
#
# 10B: comparação explícita Pain / No pain
# 10C: classificação afetiva com seis categorias
#
# 189 imagens x 2 protocolos = 378 inferências
# ============================================================

import sys
import subprocess
import importlib.util

for package, module in [
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("pillow", "PIL"),
    ("scikit-learn", "sklearn"),
    ("pandas", "pandas"),
    ("sentencepiece", "sentencepiece")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import re
import gc
import json
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

ROOT = Path("/content/florence_visapp")
IMAGES = ROOT / "images"
OUT = ROOT / "results"

OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Salesforce/blip2-opt-2.7b"

PROTOCOLS = {
    "10B_binary_comparative": {
        "prompt": (
            "Question: Is this person's facial expression "
            "more consistent with pain or no pain? "
            "Answer with pain or no pain. Answer:"
        ),
        "categories": ["pain", "no pain"]
    },
    "10C_affective": {
        "prompt": (
            "Question: Which category best describes "
            "this person's facial expression: "
            "pain, happiness, sadness, anger, "
            "disgust, or surprise? "
            "Answer with one category only. Answer:"
        ),
        "categories": [
            "pain", "happiness", "sadness",
            "anger", "disgust", "surprise"
        ]
    }
}

PRED_FILE = OUT / "blip2_experiment_10BC_predictions.csv"
METRICS_FILE = OUT / "blip2_experiment_10BC_metrics.csv"
PROTOCOL_FILE = OUT / "blip2_experiment_10BC_protocol.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

dtype = (
    torch.float16 if device == "cuda"
    else torch.float32
)

print("=== FLORENCE — EXPERIMENTOS 10B/10C ===")
print("Modelo:", MODEL_ID)
print("Dispositivo:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: execução em CPU pode ser muito lenta.")

# ============================================================
# 2. RECUPERAR O FPD-189
# ============================================================

reference = pd.read_csv(
    OUT / "clip_vit_b32_predictions.csv"
)

images = reference[
    ["image", "true_label"]
].drop_duplicates().copy()

images["image_number"] = pd.to_numeric(
    images["image"].astype(str).str.extract(
        r"(?:image|imagem)_(\d+)\.[^.]+$",
        flags=re.IGNORECASE,
        expand=False
    ),
    errors="coerce"
)

assert len(images) == 189
assert images["image_number"].notna().all()
assert images["image_number"].is_unique
assert images["true_label"].isin([0, 1]).all()

images["image_number"] = (
    images["image_number"].astype(int)
)

images = images.sort_values(
    "image_number"
).reset_index(drop=True)

def resolve_image(old_path):

    path = IMAGES / old_path

    if path.is_file():
        return path

    normalized = re.sub(
        r"(?i)imagem_(?=\d+\.)",
        "image_",
        old_path
    )

    path = IMAGES / normalized

    if path.is_file():
        return path

    raise FileNotFoundError(
        f"Imagem não encontrada: {old_path}"
    )

images["resolved_path"] = images["image"].map(
    lambda p: str(resolve_image(p))
)

print("\nTotal de imagens:", len(images))
print("Pain:", int((images.true_label == 1).sum()))
print("NoPain:", int((images.true_label == 0).sum()))
print("Protocolos:", list(PROTOCOLS))

# ============================================================
# 3. INTERPRETAÇÃO DAS RESPOSTAS
# ============================================================

def extract_answer(raw_text):

    if pd.isna(raw_text):
        return ""

    text = str(raw_text).strip()

    parts = re.split(
        r"\bAnswer\s*:",
        text,
        flags=re.IGNORECASE
    )

    return parts[-1].strip() if len(parts) > 1 else text


def normalize_text(text):

    text = str(text).strip().lower()

    text = re.sub(r"^[^a-z]+", "", text)

    text = re.sub(r"\s+", " ", text)

    return text


def parse_category(answer, protocol):

    text = normalize_text(answer)

    if protocol == "10B_binary_comparative":

        # A ordem é importante:
        # "no pain" deve ser identificado antes de "pain".

        if re.match(
            r"^(no pain|no signs of pain|without pain)\b",
            text
        ):
            return "no pain"

        if re.match(r"^pain\b", text):
            return "pain"

        return None

    if protocol == "10C_affective":

        aliases = {
            "pain": "pain",
            "happiness": "happiness",
            "happy": "happiness",
            "sadness": "sadness",
            "sad": "sadness",
            "anger": "anger",
            "angry": "anger",
            "disgust": "disgust",
            "disgusted": "disgust",
            "surprise": "surprise",
            "surprised": "surprise"
        }

        for alias in sorted(
            aliases,
            key=len,
            reverse=True
        ):
            if re.match(
                rf"^{re.escape(alias)}\b",
                text
            ):
                return aliases[alias]

        return None

    raise ValueError(
        f"Protocolo desconhecido: {protocol}"
    )


def category_to_binary(category):

    if category is None:
        return None

    return 1 if category == "pain" else 0

# ============================================================
# 4. ESTRUTURA DE RESULTADOS E RETOMADA
# ============================================================

columns = [
    "image",
    "image_number",
    "true_label",
    "model",
    "protocol",
    "prompt",
    "raw_answer",
    "extracted_answer",
    "predicted_category",
    "predicted_label",
    "valid_response",
    "correct",
    "inference_seconds"
]

if PRED_FILE.exists():

    existing = pd.read_csv(PRED_FILE)

    if not set(columns).issubset(existing.columns):
        raise ValueError(
            "Arquivo de progresso incompatível."
        )

    if existing.duplicated(
        ["image_number", "protocol"]
    ).any():
        raise ValueError(
            "Existem predições duplicadas."
        )

    for protocol, config in PROTOCOLS.items():

        subset = existing[
            existing["protocol"] == protocol
        ]

        if len(subset) and not (
            subset["model"].eq(MODEL_ID).all()
            and subset["prompt"].eq(
                config["prompt"]
            ).all()
        ):
            raise ValueError(
                f"Protocolo incompatível: {protocol}"
            )

    if not set(existing["protocol"]).issubset(
        set(PROTOCOLS)
    ):
        raise ValueError(
            "Arquivo contém protocolos desconhecidos."
        )

    completed = set(
        zip(
            existing["image_number"].astype(int),
            existing["protocol"]
        )
    )

    print(
        "\nRetomando:",
        len(completed),
        "inferências já registradas."
    )

else:
    completed = set()

# ============================================================
# 5. CARREGAR BLIP-2 UMA ÚNICA VEZ
# ============================================================

print("\nCarregando processador...")

processor = Blip2Processor.from_pretrained(
    MODEL_ID
)

print("Carregando modelo...")

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)

model = model.to(device)
model.eval()

print("Modelo carregado.")

# ============================================================
# 6. EXECUÇÃO COMPLETA DOS DOIS PROTOCOLOS
# ============================================================

start_all = time.time()
total_expected = len(images) * len(PROTOCOLS)

for protocol, config in PROTOCOLS.items():

    prompt = config["prompt"]

    print("\n" + "=" * 60)
    print("PROTOCOLO:", protocol)
    print("PROMPT:", prompt)
    print("=" * 60)

    for _, row in images.iterrows():

        number = int(row.image_number)
        key = (number, protocol)

        if key in completed:
            continue

        start = time.time()

        with Image.open(row.resolved_path) as im:
            image = im.convert("RGB")

        inputs = processor(
            images=image,
            text=prompt,
            return_tensors="pt"
        )

        inputs = {
            k: (
                v.to(device=device, dtype=dtype)
                if torch.is_floating_point(v)
                else v.to(device)
            )
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generated = model.generate(
                **inputs,
                max_new_tokens=12,
                do_sample=False,
                num_beams=1
            )

        raw_answer = processor.batch_decode(
            generated,
            skip_special_tokens=True
        )[0].strip()

        extracted_answer = extract_answer(
            raw_answer
        )

        category = parse_category(
            extracted_answer,
            protocol
        )

        predicted = category_to_binary(
            category
        )

        elapsed = time.time() - start

        record = {
            "image": row.image,
            "image_number": number,
            "true_label": int(row.true_label),
            "model": MODEL_ID,
            "protocol": protocol,
            "prompt": prompt,
            "raw_answer": raw_answer,
            "extracted_answer": extracted_answer,
            "predicted_category": category,
            "predicted_label": predicted,
            "valid_response": predicted is not None,
            "correct": (
                int(predicted == int(row.true_label))
                if predicted is not None
                else np.nan
            ),
            "inference_seconds": elapsed
        }

        pd.DataFrame(
            [record],
            columns=columns
        ).to_csv(
            PRED_FILE,
            mode="a",
            header=not PRED_FILE.exists(),
            index=False
        )

        completed.add(key)

        print(
            f"[{len(completed):03d}/{total_expected}] "
            f"{protocol} | "
            f"image_{number} | "
            f"real={row.true_label} | "
            f"resposta={extracted_answer!r} | "
            f"categoria={category} | "
            f"pred={predicted} | "
            f"{elapsed:.2f}s",
            flush=True
        )

        del inputs, generated, image
        gc.collect()

        if device == "cuda":
            torch.cuda.empty_cache()

# ============================================================
# 7. AUDITORIA E MÉTRICAS POR PROTOCOLO
# ============================================================

results = pd.read_csv(PRED_FILE)

assert len(results) == total_expected, (
    f"Esperadas {total_expected} predições; "
    f"encontradas {len(results)}."
)

assert not results.duplicated(
    ["image_number", "protocol"]
).any()

metrics_rows = []

for protocol in PROTOCOLS:

    subset = results[
        results["protocol"] == protocol
    ].copy()

    assert len(subset) == 189

    valid_mask = (
        subset["valid_response"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    valid = subset[valid_mask].copy()
    invalid = subset[~valid_mask].copy()

    coverage = len(valid) / len(subset)

    print("\n" + "=" * 60)
    print("RESULTADOS:", protocol)
    print("=" * 60)

    print("Respostas válidas:", len(valid))
    print("Respostas inválidas:", len(invalid))
    print(f"Cobertura: {coverage:.2%}")

    print("\nDistribuição das categorias:")
    print(
        subset["predicted_category"]
        .fillna("INVALID")
        .value_counts()
        .to_string()
    )

    print("\nRespostas textuais mais frequentes:")
    print(
        subset["extracted_answer"]
        .str.lower()
        .value_counts()
        .head(15)
        .to_string()
    )

    if len(invalid):

        print("\nRespostas não reconhecidas:")
        print(
            invalid[[
                "image_number",
                "extracted_answer"
            ]].to_string(index=False)
        )

    if len(valid) == 0:

        metrics_rows.append({
            "model": MODEL_ID,
            "protocol": protocol,
            "n_images": 189,
            "n_valid": 0,
            "n_invalid": 189,
            "coverage": 0
        })

        continue

    y_true = valid["true_label"].astype(int)
    y_pred = valid["predicted_label"].astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "model": MODEL_ID,
        "protocol": protocol,
        "n_images": 189,
        "n_valid": len(valid),
        "n_invalid": len(invalid),
        "coverage": coverage,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "specificity": (
            tn / (tn + fp)
            if tn + fp else np.nan
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

    metrics_rows.append(metrics)

    print("\nMétricas:")

    for key in [
        "accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy"
    ]:
        print(f"{key}: {metrics[key]:.4f}")

    print("\nMatriz de confusão:")
    print("Linhas: NoPain, Pain")
    print("Colunas: NoPain, Pain")
    print(cm)

pd.DataFrame(metrics_rows).to_csv(
    METRICS_FILE,
    index=False
)

# ============================================================
# 8. REGISTRO DO PROTOCOLO
# ============================================================

with open(PROTOCOL_FILE, "w") as f:

    json.dump({
        "model": MODEL_ID,
        "dataset": "FPD-189",
        "n_images": 189,
        "zero_shot": True,
        "protocols": PROTOCOLS,
        "generation": {
            "max_new_tokens": 12,
            "do_sample": False,
            "num_beams": 1
        },
        "device": device,
        "dtype": str(dtype),
        "response_extraction": (
            "Text after final Answer: marker"
        ),
        "binary_mapping": {
            "pain": 1,
            "other_affective_categories": 0
        }
    }, f, indent=2)

# ============================================================
# 9. RESUMO FINAL
# ============================================================

summary = pd.DataFrame(metrics_rows)

print("\n" + "=" * 60)
print("FLORENCE — RESUMO BLIP-2 10B/10C")
print("=" * 60)

display(
    summary[[
        "protocol",
        "n_valid",
        "n_invalid",
        "accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy"
    ]]
)

print("\nTempo total:",
      round((time.time() - start_all) / 60, 2),
      "minutos")

print("\nArquivos produzidos:")
print(PRED_FILE)
print(METRICS_FILE)
print(PROTOCOL_FILE)

print("\nEXPERIMENTOS 10B E 10C CONCLUÍDOS.")


=== FLORENCE — EXPERIMENTOS 10B/10C ===
Modelo: Salesforce/blip2-opt-2.7b
Dispositivo: cuda
GPU: Tesla T4

Total de imagens: 189
Pain: 99
NoPain: 90
Protocolos: ['10B_binary_comparative', '10C_affective']

Carregando processador...
Carregando modelo...


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Modelo carregado.

PROTOCOLO: 10B_binary_comparative
PROMPT: Question: Is this person's facial expression more consistent with pain or no pain? Answer with pain or no pain. Answer:
[001/378] 10B_binary_comparative | image_1 | real=1 | resposta='pain' | categoria=pain | pred=1 | 0.19s
[002/378] 10B_binary_comparative | image_2 | real=0 | resposta='pain' | categoria=pain | pred=1 | 0.14s
[003/378] 10B_binary_comparative | image_3 | real=0 | resposta='pain' | categoria=pain | pred=1 | 0.13s
[004/378] 10B_binary_comparative | image_4 | real=1 | resposta='pain' | categoria=pain | pred=1 | 0.13s
[005/378] 10B_binary_comparative | image_5 | real=0 | resposta='no pain' | categoria=no pain | pred=0 | 0.15s
[006/378] 10B_binary_comparative | image_6 | real=0 | resposta='pain' | categoria=pain | pred=1 | 0.11s
[007/378] 10B_binary_comparative | image_7 | real=0 | resposta='no pain' | categoria=no pain | pred=0 | 0.15s
[008/378] 10B_binary_comparative | image_8 | real=0 | resposta='no pain' | cate

,protocol,n_valid,n_invalid,accuracy,precision,recall,specificity,f1,balanced_accuracy
0,10B_binary_comparative,185,4,0.702703,0.646667,0.979798,0.383721,0.779116,0.681759
1,10C_affective,180,9,0.888889,0.846847,0.969072,0.795181,0.903846,0.882126



Tempo total: 2.69 minutos

Arquivos produzidos:
/content/florence_visapp/results/blip2_experiment_10BC_predictions.csv
/content/florence_visapp/results/blip2_experiment_10BC_metrics.csv
/content/florence_visapp/results/blip2_experiment_10BC_protocol.json

EXPERIMENTOS 10B E 10C CONCLUÍDOS.


# Testing Qwen2.5-VL-3B-Instruct

## Preparing Qwen

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 12
# QWEN2.5-VL-3B-INSTRUCT — PREPARAÇÃO
# Tesla T4 | Quantização 4 bits
# ============================================================

import sys
import subprocess
import gc

# Dependências
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "-U",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "qwen-vl-utils"
])

import torch
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print("=" * 60)
print("FLORENCE — QWEN2.5-VL-3B")
print("=" * 60)

print("\nGPU:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()
print(f"Memória livre: {free / 1024**3:.2f} GB")

# Liberar memória não utilizada
gc.collect()
torch.cuda.empty_cache()

# Quantização NF4
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nCarregando processador...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28
)

print("Processador carregado.")

print("\nCarregando Qwen em 4 bits...")
print("O download inicial pode levar alguns minutos.")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

print("\nMODELO CARREGADO COM SUCESSO!")

free, total = torch.cuda.mem_get_info()

print("\n=== MEMÓRIA GPU ===")
print(f"Total: {total / 1024**3:.2f} GB")
print(f"Livre: {free / 1024**3:.2f} GB")
print(
    "Reservada pelo PyTorch:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

print("\n=== MODELO ===")
print(MODEL_ID)
print("Quantização: NF4 — 4 bits")
print("Modo: inferência zero-shot")

print("\nPRONTO PARA O TESTE DE INFERÊNCIA.")


FLORENCE — QWEN2.5-VL-3B

GPU: Tesla T4
Memória livre: 14.46 GB



Carregando processador...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

... (6 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]


MODELO CARREGADO COM SUCESSO!

=== MEMÓRIA GPU ===
Total: 14.56 GB
Livre: 12.08 GB
Reservada pelo PyTorch: 2.37 GB

=== MODELO ===
Qwen/Qwen2.5-VL-3B-Instruct
Quantização: NF4 — 4 bits
Modo: inferência zero-shot

PRONTO PARA O TESTE DE INFERÊNCIA.


## Addition test for Qwen (three protocols)

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 12A
# QWEN2.5-VL-3B — TESTE DOS TRÊS PROTOCOLOS
# Uma imagem | Sem salvar resultados experimentais
# ============================================================

from pathlib import Path
from PIL import Image
import torch
import gc
import re

ROOT = Path("/content/florence_visapp")
IMAGES = ROOT / "images"

# Selecionar uma imagem do dataset para teste
candidates = [
    p for p in IMAGES.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {
        ".jpg", ".jpeg", ".jfif", ".png"
    }
]

assert candidates, "Nenhuma imagem encontrada."

# Preferir imagem com ID 1 para reprodutibilidade
matches = [
    p for p in candidates
    if re.fullmatch(
        r"(?:image|imagem)_1",
        p.stem,
        flags=re.IGNORECASE
    )
]

image_path = matches[0] if matches else sorted(candidates)[0]

print("=" * 60)
print("FLORENCE — TESTE QWEN2.5-VL-3B")
print("=" * 60)

print("\nImagem:", image_path.relative_to(ROOT))

image = Image.open(image_path).convert("RGB")

print("Dimensões:", image.size)

# Mesmos protocolos utilizados no BLIP-2
protocols = {
    "A_direct": (
        "Does this person show visible signs of pain? "
        "Answer only yes or no."
    ),
    "B_comparative": (
        "Is this person's facial expression more consistent "
        "with pain or no pain? "
        "Answer with pain or no pain."
    ),
    "C_affective": (
        "Which category best describes this person's facial "
        "expression: pain, happiness, sadness, anger, "
        "disgust, or surprise? "
        "Answer with one category only."
    )
}

def qwen_predict(image, question):

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    # Transferir entradas para a GPU
    inputs = inputs.to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            num_beams=1
        )

    generated = output[:, inputs.input_ids.shape[1]:]

    answer = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    del inputs, output, generated
    gc.collect()
    torch.cuda.empty_cache()

    return answer

print("\n=== RESULTADOS DO TESTE ===")

for name, question in protocols.items():

    print("\nProtocolo:", name)
    print("Pergunta:", question)

    try:
        answer = qwen_predict(image, question)
        print("Resposta:", repr(answer))
    except Exception as e:
        print("ERRO:", type(e).__name__, str(e)[:500])
        break

print("\n=== MEMÓRIA GPU APÓS O TESTE ===")

free, total = torch.cuda.mem_get_info()
print(f"Memória livre: {free / 1024**3:.2f} GB")

print("\nTESTE FINALIZADO.")


FLORENCE — TESTE QWEN2.5-VL-3B

Imagem: images/PainImages/pain/image_1.jpg
Dimensões: (1000, 714)

=== RESULTADOS DO TESTE ===

Protocolo: A_direct
Pergunta: Does this person show visible signs of pain? Answer only yes or no.


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


Resposta: 'Yes.'

Protocolo: B_comparative
Pergunta: Is this person's facial expression more consistent with pain or no pain? Answer with pain or no pain.
Resposta: 'No pain.'

Protocolo: C_affective
Pergunta: Which category best describes this person's facial expression: pain, happiness, sadness, anger, disgust, or surprise? Answer with one category only.
Resposta: 'Pain'

=== MEMÓRIA GPU APÓS O TESTE ===
Memória livre: 12.10 GB

TESTE FINALIZADO.


## Complete test for Qwen2.5-VL-3B-Instruct

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 12B
# QWEN2.5-VL-3B-INSTRUCT | FPD-189
# 189 imagens × 3 protocolos = 567 inferências
# Zero-shot | NF4 4 bits | Tesla T4
# Salvamento incremental e retomada
# ============================================================

from pathlib import Path
from PIL import Image
from collections import Counter
import pandas as pd
import numpy as np
import torch
import re
import csv
import json
import time
from datetime import datetime, timezone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)

ROOT = Path("/content/florence_visapp")
RESULTS = ROOT / "results"
IMAGES = ROOT / "images"
RESULTS.mkdir(parents=True, exist_ok=True)

PRED_FILE = RESULTS / "qwen25_vl3b_predictions.csv"
METRICS_FILE = RESULTS / "qwen25_vl3b_metrics.csv"
PROTOCOL_FILE = RESULTS / "qwen25_vl3b_protocol.json"

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

PROTOCOLS = {
    "A_direct": (
        "Does this person show visible signs of pain? "
        "Answer only yes or no."
    ),
    "B_comparative": (
        "Is this person's facial expression more consistent "
        "with pain or no pain? "
        "Answer with pain or no pain."
    ),
    "C_affective": (
        "Which category best describes this person's facial "
        "expression: pain, happiness, sadness, anger, "
        "disgust, or surprise? "
        "Answer with one category only."
    )
}

FIELDS = [
    "model", "protocol", "image_number", "image",
    "image_path", "true_label", "raw_answer",
    "parsed_category", "predicted_label",
    "valid", "elapsed_seconds"
]

# ------------------------------------------------------------
# 1. Recuperar referência original
# ------------------------------------------------------------

ref = pd.read_csv(
    RESULTS / "clip_vit_b32_predictions.csv"
)[["image", "true_label"]].drop_duplicates()

ref["image_number"] = (
    ref["image"].astype(str)
    .str.extract(
        r"(?:image|imagem)_(\d+)\.[^.]+$",
        flags=re.IGNORECASE,
        expand=False
    )
    .astype(int)
)

assert len(ref) == 189
assert ref["image_number"].nunique() == 189
assert set(ref["true_label"].astype(int)) == {0, 1}
assert int((ref["true_label"] == 1).sum()) == 99
assert int((ref["true_label"] == 0).sum()) == 90

# ------------------------------------------------------------
# 2. Mapear imagens incluindo JFIF
# ------------------------------------------------------------

EXTENSIONS = {
    ".jpg", ".jpeg", ".jfif", ".png",
    ".webp", ".bmp", ".tif", ".tiff"
}

image_map = {}

for path in IMAGES.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix.lower() not in EXTENSIONS:
        continue

    match = re.fullmatch(
        r"(?:image|imagem)_(\d+)",
        path.stem,
        flags=re.IGNORECASE
    )

    if match:
        image_id = int(match.group(1))
        if image_id in image_map:
            raise ValueError(
                f"Imagem duplicada: {image_id}"
            )
        image_map[image_id] = path

missing = set(ref["image_number"]) - set(image_map)

assert not missing, f"Imagens ausentes: {sorted(missing)}"

# Verificar coerência entre pasta e rótulo
for row in ref.itertuples(index=False):
    path = image_map[int(row.image_number)]
    if int(row.true_label) == 1:
        assert "PainImages" in path.parts
    else:
        assert "NoPainImages" in path.parts

# ------------------------------------------------------------
# 3. Parser conservador
# ------------------------------------------------------------

def parse_answer(answer, protocol):

    text = str(answer).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" \t\n\r.,;:!?\"'")

    if protocol == "A_direct":
        if text == "yes":
            return "pain", 1, True
        if text == "no":
            return "no_pain", 0, True
        return None, None, False

    if protocol == "B_comparative":
        if text == "pain":
            return "pain", 1, True
        if text in {"no pain", "no_pain"}:
            return "no_pain", 0, True
        return None, None, False

    if protocol == "C_affective":

        categories = {
            "pain": ("pain", 1),
            "happiness": ("happiness", 0),
            "sadness": ("sadness", 0),
            "anger": ("anger", 0),
            "disgust": ("disgust", 0),
            "surprise": ("surprise", 0)
        }

        if text in categories:
            category, label = categories[text]
            return category, label, True

        return None, None, False

    raise ValueError("Protocolo desconhecido")

# ------------------------------------------------------------
# 4. Inferência
# ------------------------------------------------------------

def infer_qwen(image, question):

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[prompt],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            num_beams=1
        )

    generated = output[
        :, inputs.input_ids.shape[1]:
    ]

    answer = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    del inputs, output, generated

    return answer

# ------------------------------------------------------------
# 5. Retomada segura
# ------------------------------------------------------------

completed = set()

if PRED_FILE.exists():

    previous = pd.read_csv(PRED_FILE)

    if len(previous):
        required = {
            "image_number", "protocol",
            "true_label", "model"
        }
        assert required.issubset(previous.columns)

        assert (
            previous["model"] == MODEL_ID
        ).all(), "Arquivo contém outro modelo."

        assert not previous.duplicated(
            ["image_number", "protocol"]
        ).any(), "Há inferências duplicadas."

        expected_labels = ref.set_index(
            "image_number"
        )["true_label"].astype(int)

        for r in previous.itertuples():
            assert int(r.true_label) == int(
                expected_labels.loc[int(r.image_number)]
            )

        completed = set(
            zip(
                previous["image_number"].astype(int),
                previous["protocol"]
            )
        )

print("=" * 65)
print("FLORENCE — QWEN2.5-VL-3B | EXPERIMENTO COMPLETO")
print("=" * 65)
print("Imagens:", len(ref))
print("Protocolos:", len(PROTOCOLS))
print("Total esperado:", len(ref) * len(PROTOCOLS))
print("Já concluídas:", len(completed))
print("Pendentes:", 567 - len(completed))

# Criar CSV novo apenas se ainda não existir
if not PRED_FILE.exists():
    with open(PRED_FILE, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()

# Registrar protocolo sem sobrescrever registro anterior
if not PROTOCOL_FILE.exists():
    protocol_metadata = {
        "model": MODEL_ID,
        "dataset": "FPD-189",
        "images": 189,
        "pain": 99,
        "no_pain": 90,
        "quantization": "NF4 4-bit",
        "generation": {
            "max_new_tokens": 20,
            "do_sample": False,
            "num_beams": 1
        },
        "protocols": PROTOCOLS,
        "reference_file": "clip_vit_b32_predictions.csv",
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    with open(PROTOCOL_FILE, "w", encoding="utf-8") as f:
        json.dump(
            protocol_metadata, f,
            indent=2, ensure_ascii=False
        )

# ------------------------------------------------------------
# 6. Execução incremental
# ------------------------------------------------------------

processed_now = 0
errors = []

# Ordem fixa: protocolo e número da imagem
ordered = ref.sort_values("image_number")

for protocol, question in PROTOCOLS.items():

    print(f"\n=== {protocol} ===", flush=True)

    for r in ordered.itertuples(index=False):

        image_id = int(r.image_number)
        key = (image_id, protocol)

        if key in completed:
            continue

        path = image_map[image_id]

        try:
            with Image.open(path) as img:
                image = img.convert("RGB")

            start = time.perf_counter()

            raw = infer_qwen(image, question)

            elapsed = time.perf_counter() - start

            category, prediction, valid = parse_answer(
                raw, protocol
            )

            record = {
                "model": MODEL_ID,
                "protocol": protocol,
                "image_number": image_id,
                "image": r.image,
                "image_path": str(path),
                "true_label": int(r.true_label),
                "raw_answer": raw,
                "parsed_category": category,
                "predicted_label": prediction,
                "valid": valid,
                "elapsed_seconds": round(elapsed, 3)
            }

            # Salvar imediatamente cada inferência
            with open(
                PRED_FILE, "a",
                newline="", encoding="utf-8"
            ) as f:
                writer = csv.DictWriter(
                    f, fieldnames=FIELDS
                )
                writer.writerow(record)
                f.flush()

            completed.add(key)
            processed_now += 1

            if processed_now % 10 == 0:
                print(
                    f"Novas inferências: {processed_now} | "
                    f"Total salvo: {len(completed)}/567 | "
                    f"Última resposta: {raw!r}",
                    flush=True
                )

            del image

        except Exception as e:
            errors.append(
                (protocol, image_id, type(e).__name__, str(e))
            )
            print(
                f"ERRO | {protocol} | imagem {image_id}: "
                f"{type(e).__name__}: {str(e)[:200]}",
                flush=True
            )

            if isinstance(e, torch.cuda.OutOfMemoryError):
                raise

            # Evitar avançar silenciosamente após falha técnica
            raise

    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 7. Métricas
# ------------------------------------------------------------

pred = pd.read_csv(PRED_FILE)

assert not pred.duplicated(
    ["image_number", "protocol"]
).any()

metrics = []

for protocol in PROTOCOLS:

    subset = pred[
        pred["protocol"] == protocol
    ].copy()

    subset["valid"] = (
        subset["valid"].astype(str).str.lower() == "true"
    )

    valid = subset[subset["valid"]].copy()

    n_total = len(subset)
    n_valid = len(valid)
    n_invalid = n_total - n_valid

    row = {
        "model": MODEL_ID,
        "protocol": protocol,
        "n_total": n_total,
        "n_valid": n_valid,
        "n_invalid": n_invalid,
        "coverage": n_valid / n_total if n_total else np.nan
    }

    if n_valid:
        y_true = valid["true_label"].astype(int)
        y_pred = valid["predicted_label"].astype(int)

        cm = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        row.update({
            "accuracy_valid": accuracy_score(y_true, y_pred),
            "precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall_sensitivity": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "specificity": (
                tn / (tn + fp)
                if tn + fp else np.nan
            ),
            "f1": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ) if len(set(y_true)) == 2 else np.nan,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "accuracy_all_invalid_as_error": (
                (tn + tp) / 189
            )
        })

    metrics.append(row)

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(METRICS_FILE, index=False)

print("\n" + "=" * 65)
print("RESULTADOS DO QWEN2.5-VL-3B")
print("=" * 65)

print(metrics_df.to_string(index=False))

print("\n=== DISTRIBUIÇÃO DAS RESPOSTAS ===")

for protocol in PROTOCOLS:

    subset = pred[pred["protocol"] == protocol]

    print(f"\n{protocol}")

    print(
        subset["parsed_category"]
        .fillna("INVALID")
        .value_counts()
        .to_string()
    )

print("\n=== INFERÊNCIAS SALVAS ===")
print("Total:", len(pred), "/ 567")
print("Erros:", len(errors))

print("\nArquivos:")
print(PRED_FILE)
print(METRICS_FILE)
print(PROTOCOL_FILE)

free, total = torch.cuda.mem_get_info()
print(f"\nMemória GPU livre: {free / 1024**3:.2f} GB")

if len(pred) == 567:
    print("\nEXPERIMENTO 12B CONCLUÍDO!")
else:
    print("\nEXECUÇÃO PARCIAL — PODE SER RETOMADA.")


FLORENCE — QWEN2.5-VL-3B | EXPERIMENTO COMPLETO
Imagens: 189
Protocolos: 3
Total esperado: 567
Já concluídas: 0
Pendentes: 567

=== A_direct ===


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Novas inferências: 10 | Total salvo: 10/567 | Última resposta: 'No'
Novas inferências: 20 | Total salvo: 20/567 | Última resposta: 'Yes.'
Novas inferências: 30 | Total salvo: 30/567 | Última resposta: 'Yes.'
Novas inferências: 40 | Total salvo: 40/567 | Última resposta: 'Yes.'
Novas inferências: 50 | Total salvo: 50/567 | Última resposta: 'Yes.'
Novas inferências: 60 | Total salvo: 60/567 | Última resposta: 'Yes.'
Novas inferências: 70 | Total salvo: 70/567 | Última resposta: 'Yes.'
Novas inferências: 80 | Total salvo: 80/567 | Última resposta: 'Yes.'
Novas inferências: 90 | Total salvo: 90/567 | Última resposta: 'No.'
Novas inferências: 100 | Total salvo: 100/567 | Última resposta: 'No.'
Novas inferências: 110 | Total salvo: 110/567 | Última resposta: 'No.'
Novas inferências: 120 | Total salvo: 120/567 | Última resposta: 'No.'
Novas inferências: 130 | Total salvo: 130/567 | Última resposta: 'No'
Novas inferências: 140 | Total salvo: 140/567 | Última resposta: 'Yes.'
Novas inferências:

## Exact McNemar, Holm correction, and bootstrap paired with 10,000 resamplings

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 13
# Comparação estatística pareada dos protocolos Qwen
# McNemar exato + Holm + Bootstrap 95%
# Somente análise dos CSVs existentes
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

RESULTS = Path("/content/florence_visapp/results")

df = pd.read_csv(
    RESULTS / "qwen25_vl3b_predictions.csv"
)

protocols = [
    "A_direct",
    "B_comparative",
    "C_affective"
]

assert len(df) == 567
assert not df.duplicated(
    ["image_number", "protocol"]
).any()

assert set(df["protocol"]) == set(protocols)

df["valid"] = (
    df["valid"].astype(str).str.lower() == "true"
)

assert df["valid"].all(), "Existem respostas inválidas."

df["correct"] = (
    df["true_label"].astype(int)
    == df["predicted_label"].astype(int)
)

wide = df.pivot(
    index="image_number",
    columns="protocol",
    values="correct"
)

assert wide.shape == (189, 3)
assert not wide.isna().any().any()

wide = wide.astype(bool)

comparisons = [
    ("C_affective", "A_direct"),
    ("C_affective", "B_comparative"),
    ("B_comparative", "A_direct")
]

rng = np.random.default_rng(20261110)

results = []

for a, b in comparisons:

    x = wide[a].to_numpy(dtype=int)
    y = wide[b].to_numpy(dtype=int)

    # Discordâncias pareadas
    a_only = int(((x == 1) & (y == 0)).sum())
    b_only = int(((x == 0) & (y == 1)).sum())

    discordant = a_only + b_only

    # McNemar exato bilateral
    p = (
        binomtest(
            min(a_only, b_only),
            discordant,
            0.5,
            alternative="two-sided"
        ).pvalue
        if discordant else 1.0
    )

    difference = x.mean() - y.mean()

    # Bootstrap pareado por imagem
    n = len(x)
    indices = rng.integers(
        0, n, size=(10000, n)
    )

    differences = (
        x[indices].mean(axis=1)
        - y[indices].mean(axis=1)
    )

    ci_low, ci_high = np.percentile(
        differences, [2.5, 97.5]
    )

    results.append({
        "protocol_a": a,
        "protocol_b": b,
        "accuracy_a": x.mean(),
        "accuracy_b": y.mean(),
        "difference_pp": 100 * difference,
        "a_correct_b_wrong": a_only,
        "a_wrong_b_correct": b_only,
        "discordant_pairs": discordant,
        "mcnemar_exact_p": p,
        "bootstrap_ci_low_pp": 100 * ci_low,
        "bootstrap_ci_high_pp": 100 * ci_high
    })

result = pd.DataFrame(results)

# Correção de Holm para as três comparações
result["holm_adjusted_p"] = multipletests(
    result["mcnemar_exact_p"],
    method="holm"
)[1]

result["significant_005"] = (
    result["holm_adjusted_p"] < 0.05
)

output = RESULTS / "qwen25_vl3b_paired_statistics.csv"
result.to_csv(output, index=False)

print("=" * 70)
print("FLORENCE — COMPARAÇÃO ESTATÍSTICA DO QWEN")
print("=" * 70)

for r in result.itertuples(index=False):

    print(
        f"\n{r.protocol_a} vs {r.protocol_b}"
    )

    print(
        f"Acurácias: {r.accuracy_a:.4f} "
        f"vs {r.accuracy_b:.4f}"
    )

    print(
        f"Diferença: {r.difference_pp:+.2f} pp"
    )

    print(
        "IC bootstrap 95%:",
        f"[{r.bootstrap_ci_low_pp:.2f}, "
        f"{r.bootstrap_ci_high_pp:.2f}] pp"
    )

    print(
        "A correto / B errado:",
        r.a_correct_b_wrong
    )

    print(
        "A errado / B correto:",
        r.a_wrong_b_correct
    )

    print(
        f"McNemar exato p = {r.mcnemar_exact_p:.8g}"
    )

    print(
        f"Holm p ajustado = {r.holm_adjusted_p:.8g}"
    )

    print(
        "Significativo (5%):",
        "SIM" if r.significant_005 else "NÃO"
    )

print("\n=== IMAGENS INCORRETAS NO PROTOCOLO AFETIVO ===")

errors = df[
    (df["protocol"] == "C_affective")
    & (~df["correct"])
].sort_values("image_number")

print(
    errors[
        [
            "image_number",
            "true_label",
            "raw_answer",
            "predicted_label"
        ]
    ].to_string(index=False)
)

print("\nTotal de erros:", len(errors))
print("\nArquivo salvo:", output)
print("\nANÁLISE CONCLUÍDA.")


FLORENCE — COMPARAÇÃO ESTATÍSTICA DO QWEN

C_affective vs A_direct
Acurácias: 0.9471 vs 0.7989
Diferença: +14.81 pp
IC bootstrap 95%: [8.99, 21.16] pp
A correto / B errado: 33
A errado / B correto: 5
McNemar exato p = 4.2559623e-06
Holm p ajustado = 8.5119245e-06
Significativo (5%): SIM

C_affective vs B_comparative
Acurácias: 0.9471 vs 0.8042
Diferença: +14.29 pp
IC bootstrap 95%: [8.99, 20.11] pp
A correto / B errado: 30
A errado / B correto: 3
McNemar exato p = 1.4011748e-06
Holm p ajustado = 4.2035244e-06
Significativo (5%): SIM

B_comparative vs A_direct
Acurácias: 0.8042 vs 0.7989
Diferença: +0.53 pp
IC bootstrap 95%: [-7.94, 8.99] pp
A correto / B errado: 36
A errado / B correto: 35
McNemar exato p = 1
Holm p ajustado = 1
Significativo (5%): NÃO

=== IMAGENS INCORRETAS NO PROTOCOLO AFETIVO ===
 image_number  true_label raw_answer  predicted_label
           44           1      Anger                0
           56           1  Happiness                0
           59           1 

# Qwen × BLIP-2

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 14
# QWEN2.5-VL-3B vs BLIP-2
# Comparação pareada sobre FPD-189
#
# Análises:
# 1. Casos completos (respostas válidas em ambos)
# 2. Todas as 189 imagens (inválidas = falhas)
#
# McNemar exato + Holm + bootstrap pareado 95%
# Não modifica os arquivos originais
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

ROOT = Path("/content/florence_visapp")
RESULTS = ROOT / "results"

QWEN_FILE = RESULTS / "qwen25_vl3b_predictions.csv"
BLIP_FILE = RESULTS / "blip2_experiment_10BC_predictions.csv"

OUT_STATS = RESULTS / "qwen_blip2_paired_statistics.csv"
OUT_IMAGES = RESULTS / "qwen_blip2_image_comparison.csv"

SEED = 20261110
N_BOOT = 10000

# ------------------------------------------------------------
# 1. Carregar dados
# ------------------------------------------------------------

qwen = pd.read_csv(QWEN_FILE)
blip = pd.read_csv(BLIP_FILE)

protocol_map = {
    "B_comparative": "10B_binary_comparative",
    "C_affective": "10C_affective"
}

def normalize_valid(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )

qwen["valid_normalized"] = normalize_valid(qwen["valid"])
blip["valid_normalized"] = normalize_valid(
    blip["valid_response"]
)

# ------------------------------------------------------------
# 2. Harmonizar os protocolos
# ------------------------------------------------------------

q = qwen[
    qwen["protocol"].isin(protocol_map.keys())
].copy()

b = blip[
    blip["protocol"].isin(protocol_map.values())
].copy()

inverse_map = {
    value: key
    for key, value in protocol_map.items()
}

b["protocol"] = b["protocol"].map(inverse_map)

q = q.rename(columns={
    "predicted_label": "qwen_prediction",
    "valid_normalized": "qwen_valid",
    "raw_answer": "qwen_raw"
})

b = b.rename(columns={
    "predicted_label": "blip_prediction",
    "valid_normalized": "blip_valid",
    "raw_answer": "blip_raw",
    "true_label": "blip_true_label"
})

assert not q.duplicated(
    ["image_number", "protocol"]
).any()

assert not b.duplicated(
    ["image_number", "protocol"]
).any()

merged = q[
    [
        "image_number", "image", "true_label",
        "protocol", "qwen_prediction",
        "qwen_valid", "qwen_raw"
    ]
].merge(
    b[
        [
            "image_number", "protocol",
            "blip_true_label",
            "blip_prediction",
            "blip_valid", "blip_raw"
        ]
    ],
    on=["image_number", "protocol"],
    how="inner",
    validate="one_to_one"
)

assert len(merged) == 378, (
    f"Esperados 378 pares; encontrados {len(merged)}"
)

assert (
    merged["true_label"].astype(int)
    == merged["blip_true_label"].astype(int)
).all(), "Rótulos de referência divergentes."

assert merged.groupby("protocol")[
    "image_number"
].nunique().eq(189).all()

# ------------------------------------------------------------
# 3. Acertos e erros
# ------------------------------------------------------------

merged["qwen_correct"] = (
    merged["qwen_valid"]
    & (
        merged["qwen_prediction"]
        == merged["true_label"]
    )
)

merged["blip_correct"] = (
    merged["blip_valid"]
    & (
        merged["blip_prediction"]
        == merged["true_label"]
    )
)

merged["both_valid"] = (
    merged["qwen_valid"] & merged["blip_valid"]
)

merged["comparison_status"] = np.select(
    [
        merged["qwen_correct"] & merged["blip_correct"],
        merged["qwen_correct"] & ~merged["blip_correct"],
        ~merged["qwen_correct"] & merged["blip_correct"],
        ~merged["qwen_correct"] & ~merged["blip_correct"]
    ],
    [
        "both_correct",
        "qwen_only_correct",
        "blip_only_correct",
        "both_incorrect"
    ],
    default="unknown"
)

merged.to_csv(OUT_IMAGES, index=False)

# ------------------------------------------------------------
# 4. Estatística pareada
# ------------------------------------------------------------

def paired_test(data, protocol, analysis, seed):

    x = data["qwen_correct"].astype(int).to_numpy()
    y = data["blip_correct"].astype(int).to_numpy()

    n = len(data)

    qwen_only = int(
        ((x == 1) & (y == 0)).sum()
    )

    blip_only = int(
        ((x == 0) & (y == 1)).sum()
    )

    discordant = qwen_only + blip_only

    p = (
        binomtest(
            min(qwen_only, blip_only),
            discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
        if discordant else 1.0
    )

    rng = np.random.default_rng(seed)

    # Reamostragem pareada por imagem
    indices = rng.integers(
        0, n, size=(N_BOOT, n)
    )

    differences = (
        x[indices].mean(axis=1)
        - y[indices].mean(axis=1)
    )

    low, high = np.percentile(
        differences, [2.5, 97.5]
    )

    return {
        "protocol": protocol,
        "analysis": analysis,
        "n_images": n,
        "qwen_correct": int(x.sum()),
        "blip_correct": int(y.sum()),
        "qwen_accuracy": float(x.mean()),
        "blip_accuracy": float(y.mean()),
        "difference_pp": float(
            100 * (x.mean() - y.mean())
        ),
        "qwen_only_correct": qwen_only,
        "blip_only_correct": blip_only,
        "discordant_pairs": discordant,
        "mcnemar_exact_p": float(p),
        "bootstrap_ci_low_pp": float(100 * low),
        "bootstrap_ci_high_pp": float(100 * high)
    }

results = []

for idx, protocol in enumerate(protocol_map):

    subset = merged[
        merged["protocol"] == protocol
    ].sort_values("image_number")

    # Análise 1: apenas casos completos
    complete = subset[
        subset["both_valid"]
    ].copy()

    results.append(
        paired_test(
            complete,
            protocol,
            "complete_cases",
            SEED + idx * 2
        )
    )

    # Análise 2: conjunto integral
    results.append(
        paired_test(
            subset,
            protocol,
            "all_189_invalid_as_error",
            SEED + idx * 2 + 1
        )
    )

stats = pd.DataFrame(results)

# Holm para as quatro hipóteses testadas
stats["holm_adjusted_p"] = multipletests(
    stats["mcnemar_exact_p"],
    method="holm"
)[1]

stats["significant_005"] = (
    stats["holm_adjusted_p"] < 0.05
)

stats.to_csv(OUT_STATS, index=False)

# ------------------------------------------------------------
# 5. Exibir resultados
# ------------------------------------------------------------

print("=" * 72)
print("FLORENCE — QWEN vs BLIP-2")
print("=" * 72)

for r in stats.itertuples(index=False):

    print("\n" + "-" * 65)
    print("Protocolo:", r.protocol)
    print("Análise:", r.analysis)
    print("Imagens:", r.n_images)

    print(
        f"Qwen: {r.qwen_correct}/{r.n_images} "
        f"= {100*r.qwen_accuracy:.2f}%"
    )

    print(
        f"BLIP-2: {r.blip_correct}/{r.n_images} "
        f"= {100*r.blip_accuracy:.2f}%"
    )

    print(
        f"Diferença: {r.difference_pp:+.2f} pp"
    )

    print(
        "IC bootstrap 95%:",
        f"[{r.bootstrap_ci_low_pp:.2f}, "
        f"{r.bootstrap_ci_high_pp:.2f}] pp"
    )

    print(
        "Qwen correto / BLIP errado:",
        r.qwen_only_correct
    )

    print(
        "Qwen errado / BLIP correto:",
        r.blip_only_correct
    )

    print(
        f"McNemar exato: p={r.mcnemar_exact_p:.8g}"
    )

    print(
        f"Holm ajustado: p={r.holm_adjusted_p:.8g}"
    )

    print(
        "Significativo:",
        "SIM" if r.significant_005 else "NÃO"
    )

print("\n" + "=" * 72)
print("DISTRIBUIÇÃO DAS DISCORDÂNCIAS")
print("=" * 72)

for protocol in protocol_map:

    subset = merged[
        merged["protocol"] == protocol
    ]

    print(f"\n{protocol}")

    print(
        subset["comparison_status"]
        .value_counts()
        .to_string()
    )

    print(
        "BLIP-2 inválidas:",
        int((~subset["blip_valid"]).sum())
    )

print("\n" + "=" * 72)
print("ARQUIVOS GERADOS")
print("=" * 72)

print(OUT_STATS)
print(OUT_IMAGES)

print("\nEXPERIMENTO 14 CONCLUÍDO.")


FLORENCE — QWEN vs BLIP-2

-----------------------------------------------------------------
Protocolo: B_comparative
Análise: complete_cases
Imagens: 185
Qwen: 148/185 = 80.00%
BLIP-2: 130/185 = 70.27%
Diferença: +9.73 pp
IC bootstrap 95%: [0.00, 19.46] pp
Qwen correto / BLIP errado: 53
Qwen errado / BLIP correto: 35
McNemar exato: p=0.069346044
Holm ajustado: p=0.10495758
Significativo: NÃO

-----------------------------------------------------------------
Protocolo: B_comparative
Análise: all_189_invalid_as_error
Imagens: 189
Qwen: 152/189 = 80.42%
BLIP-2: 130/189 = 68.78%
Diferença: +11.64 pp
IC bootstrap 95%: [1.59, 21.16] pp
Qwen correto / BLIP errado: 57
Qwen errado / BLIP correto: 35
McNemar exato: p=0.028016902
Holm ajustado: p=0.084050706
Significativo: NÃO

-----------------------------------------------------------------
Protocolo: C_affective
Análise: complete_cases
Imagens: 180
Qwen: 170/180 = 94.44%
BLIP-2: 160/180 = 88.89%
Diferença: +5.56 pp
IC bootstrap 95%: [0.56, 10

# Relationship between human difficulty and model errors.

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 16
# Dificuldade humana × erros de modelos multimodais
#
# Unidade observacional: imagem (n = 189)
# Qwen e BLIP-2: protocolo afetivo
# Contraste: total de erros nos 18 experimentos anteriores
#
# Spearman + permutação estratificada + bootstrap
# Sem novas inferências e sem sobrescrever CSVs anteriores
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path("/content/florence_visapp")
RESULTS = ROOT / "results"

SEED = 20261110
N_PERM = 10000
N_BOOT = 10000

# ------------------------------------------------------------
# 1. Carregar os três conjuntos de resultados
# ------------------------------------------------------------

human = pd.read_csv(
    RESULTS / "human_model_error_association.csv"
)

generative = pd.read_csv(
    RESULTS / "qwen_blip2_image_comparison.csv"
)

contrastive = pd.read_csv(
    RESULTS / "integrated_image_error_analysis.csv"
)

generative = generative[
    generative["protocol"] == "C_affective"
].copy()

assert len(human) == 189
assert len(generative) == 189
assert len(contrastive) == 189

assert human["image_number"].is_unique
assert generative["image_number"].is_unique
assert contrastive["image"].is_unique

# ------------------------------------------------------------
# 2. Integrar por identificador de imagem
# ------------------------------------------------------------

data = human.drop(
    columns=["total_errors"],
    errors="ignore"
).merge(
    generative[
        [
            "image_number",
            "true_label",
            "qwen_correct",
            "blip_correct",
            "qwen_valid",
            "blip_valid",
            "qwen_raw",
            "blip_raw",
            "comparison_status"
        ]
    ].rename(
        columns={"true_label": "generative_true_label"}
    ),
    on="image_number",
    how="inner",
    validate="one_to_one"
).merge(
    contrastive[
        ["image", "total_errors"]
    ].rename(
        columns={"total_errors": "contrastive_errors"}
    ),
    on="image",
    how="inner",
    validate="one_to_one"
)

assert len(data) == 189
assert data["image_number"].is_unique

assert (
    data["true_label"].astype(int)
    == data["generative_true_label"].astype(int)
).all()

def as_bool(s):
    return s.astype(str).str.lower().eq("true")

for col in [
    "qwen_correct",
    "blip_correct",
    "qwen_valid",
    "blip_valid"
]:
    data[col] = as_bool(data[col])

data["qwen_error"] = (
    ~data["qwen_correct"]
).astype(int)

data["blip_error"] = (
    ~data["blip_correct"]
).astype(int)

data["both_error"] = (
    data["qwen_error"].eq(1)
    & data["blip_error"].eq(1)
).astype(int)

data["generative_error_count"] = (
    data["qwen_error"] + data["blip_error"]
)

# Recalcular para conferir os indicadores humanos
data["indeterminate_fraction_check"] = (
    data["n_indeterminate"] / data["n_human"]
)

assert np.allclose(
    data["indeterminate_fraction"],
    data["indeterminate_fraction_check"],
    equal_nan=True
)

assert int(data["qwen_error"].sum()) == 10
assert int(data["blip_error"].sum()) == 29
assert int(data["both_error"].sum()) == 4

# ------------------------------------------------------------
# 3. Correlações exploratórias de Spearman
# ------------------------------------------------------------

human_variables = {
    "human_disagreement": "Discordância humana",
    "indeterminate_fraction": "Indeterminação humana"
}

model_variables = {
    "qwen_error": "Erro Qwen",
    "blip_error": "Falha BLIP-2",
    "generative_error_count": "Falhas generativas (0–2)",
    "contrastive_errors": "Erros contrastivos (0–18)"
}

correlations = []

for h_col, h_name in human_variables.items():

    for m_col, m_name in model_variables.items():

        x = data[h_col].astype(float)
        y = data[m_col].astype(float)

        mask = x.notna() & y.notna()

        if (
            x[mask].nunique() > 1
            and y[mask].nunique() > 1
        ):
            rho, p = spearmanr(
                x[mask], y[mask]
            )
        else:
            rho, p = np.nan, np.nan

        correlations.append({
            "human_variable": h_col,
            "model_variable": m_col,
            "n": int(mask.sum()),
            "spearman_rho": rho,
            "p_exploratory": p
        })

corr_df = pd.DataFrame(correlations)

# ------------------------------------------------------------
# 4. Diferenças entre imagens corretas e incorretas
# ------------------------------------------------------------

def stratified_permutation(
    frame, outcome, variable,
    n_perm=N_PERM, seed=SEED
):
    """
    Diferença de médias erro - acerto,
    ponderada pelas proporções das classes
    de referência. Permuta rótulos de erro
    somente dentro de cada classe.
    """

    rng = np.random.default_rng(seed)

    d = frame[
        ["true_label", outcome, variable]
    ].dropna().copy()

    groups = []

    for label, group in d.groupby("true_label"):

        y = group[outcome].to_numpy(dtype=int)
        x = group[variable].to_numpy(dtype=float)

        if y.sum() == 0 or y.sum() == len(y):
            return {
                "n": len(d),
                "difference": np.nan,
                "p_permutation": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan
            }

        weight = len(group) / len(d)
        groups.append((x, y, weight))

    def weighted_difference(groups):
        return sum(
            weight * (
                x[y == 1].mean()
                - x[y == 0].mean()
            )
            for x, y, weight in groups
        )

    observed = weighted_difference(groups)

    permuted = np.empty(n_perm)

    for i in range(n_perm):
        perm_groups = [
            (x, rng.permutation(y), weight)
            for x, y, weight in groups
        ]
        permuted[i] = weighted_difference(perm_groups)

    p = (
        1 + np.sum(
            np.abs(permuted) >= abs(observed) - 1e-12
        )
    ) / (n_perm + 1)

    # Bootstrap estratificado por classe E
    # status de erro, preservando ambas as categorias
    bootstrap = np.empty(N_BOOT)

    for i in range(N_BOOT):

        sampled_groups = []

        for x, y, weight in groups:

            x0 = x[y == 0]
            x1 = x[y == 1]

            s0 = rng.choice(
                x0, size=len(x0), replace=True
            )
            s1 = rng.choice(
                x1, size=len(x1), replace=True
            )

            xb = np.concatenate([s0, s1])
            yb = np.concatenate([
                np.zeros(len(s0), dtype=int),
                np.ones(len(s1), dtype=int)
            ])

            sampled_groups.append(
                (xb, yb, weight)
            )

        bootstrap[i] = weighted_difference(
            sampled_groups
        )

    low, high = np.percentile(
        bootstrap, [2.5, 97.5]
    )

    return {
        "n": len(d),
        "difference": observed,
        "p_permutation": p,
        "ci_low": low,
        "ci_high": high
    }

tests = []

for outcome in [
    "qwen_error",
    "blip_error",
    "both_error"
]:

    for variable in human_variables:

        result = stratified_permutation(
            data,
            outcome,
            variable,
            seed=SEED + len(tests)
        )

        tests.append({
            "outcome": outcome,
            "human_variable": variable,
            **result
        })

tests_df = pd.DataFrame(tests)

# ------------------------------------------------------------
# 5. Resumo descritivo dos erros
# ------------------------------------------------------------

descriptive = []

for model_col, name in [
    ("qwen_error", "Qwen"),
    ("blip_error", "BLIP-2"),
    ("both_error", "Ambos")
]:

    for error_value in [0, 1]:

        subset = data[
            data[model_col] == error_value
        ]

        descriptive.append({
            "model": name,
            "error": error_value,
            "n_images": len(subset),
            "mean_human_disagreement": (
                subset["human_disagreement"].mean()
            ),
            "mean_indeterminate_fraction": (
                subset["indeterminate_fraction"].mean()
            ),
            "mean_contrastive_errors": (
                subset["contrastive_errors"].mean()
            )
        })

desc_df = pd.DataFrame(descriptive)

# ------------------------------------------------------------
# 6. Identificar as quatro imagens compartilhadas
# ------------------------------------------------------------

shared = data[
    data["both_error"] == 1
].sort_values("image_number")

shared_cols = [
    "image_number",
    "true_label",
    "n_human",
    "n_pain",
    "n_nopain",
    "n_indeterminate",
    "human_disagreement",
    "indeterminate_fraction",
    "reference_agreement_fraction",
    "contrastive_errors",
    "qwen_raw",
    "blip_raw"
]

# ------------------------------------------------------------
# 7. Salvar resultados sem sobrescrever
# ------------------------------------------------------------

files = {
    "florence_exp16_integrated.csv": data,
    "florence_exp16_correlations.csv": corr_df,
    "florence_exp16_permutation_tests.csv": tests_df,
    "florence_exp16_descriptive.csv": desc_df,
    "florence_exp16_shared_errors.csv": shared[shared_cols]
}

for filename, frame in files.items():

    path = RESULTS / filename

    if path.exists():
        raise FileExistsError(
            f"Arquivo já existe: {path}. "
            "Nenhum resultado anterior será sobrescrito."
        )

    frame.to_csv(path, index=False)

# ------------------------------------------------------------
# 8. Apresentar resultados
# ------------------------------------------------------------

print("=" * 72)
print("FLORENCE — EXPERIMENTO 16")
print("DIFICULDADE HUMANA × MODELOS GENERATIVOS")
print("=" * 72)

print("\n=== INTEGRIDADE ===")
print("Imagens integradas:", len(data))
print("Erros Qwen:", int(data["qwen_error"].sum()))
print("Falhas BLIP-2:", int(data["blip_error"].sum()))
print("Erros compartilhados:", int(data["both_error"].sum()))

print("\n=== CORRELAÇÕES DE SPEARMAN ===")
print(
    corr_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== TESTES DE PERMUTAÇÃO E BOOTSTRAP ===")
print(
    tests_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== ESTATÍSTICAS DESCRITIVAS ===")
print(
    desc_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n=== QUATRO ERROS COMPARTILHADOS ===")

print(
    shared[
        [
            "image_number",
            "true_label",
            "n_human",
            "n_pain",
            "n_nopain",
            "n_indeterminate",
            "human_disagreement",
            "contrastive_errors"
        ]
    ].to_string(index=False)
)

print("\n=== ARQUIVOS GERADOS ===")

for filename in files:
    print(RESULTS / filename)

print("\nEXPERIMENTO 16 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 16
DIFICULDADE HUMANA × MODELOS GENERATIVOS

=== INTEGRIDADE ===
Imagens integradas: 189
Erros Qwen: 10
Falhas BLIP-2: 29
Erros compartilhados: 4

=== CORRELAÇÕES DE SPEARMAN ===
        human_variable         model_variable   n  spearman_rho  p_exploratory
    human_disagreement             qwen_error 189     -0.058724       0.422172
    human_disagreement             blip_error 189      0.022216       0.761562
    human_disagreement generative_error_count 189     -0.002201       0.976021
    human_disagreement     contrastive_errors 189      0.258396       0.000331
indeterminate_fraction             qwen_error 189      0.098242       0.178655
indeterminate_fraction             blip_error 189      0.225432       0.001815
indeterminate_fraction generative_error_count 189      0.186738       0.010086
indeterminate_fraction     contrastive_errors 189      0.285303       0.000069

=== TESTES DE PERMUTAÇÃO E BOOTSTRAP ===
   outcome         human_variable   n  differ

# Robustness of human associations

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 17
# Robustez das associações humanas
# Holm + Fisher exato + auditoria descritiva
# Sem novas inferências
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

RESULTS = Path("/content/florence_visapp/results")

data = pd.read_csv(
    RESULTS / "florence_exp16_integrated.csv"
)

corr = pd.read_csv(
    RESULTS / "florence_exp16_correlations.csv"
)

perm = pd.read_csv(
    RESULTS / "florence_exp16_permutation_tests.csv"
)

# 1. Corrigir as oito correlações exploratórias
corr["holm_p"] = multipletests(
    corr["p_exploratory"],
    method="holm"
)[1]

corr["significant_holm"] = corr["holm_p"] < 0.05

# 2. Corrigir os seis testes de permutação
perm["holm_p"] = multipletests(
    perm["p_permutation"],
    method="holm"
)[1]

perm["significant_holm"] = perm["holm_p"] < 0.05

# 3. Comparar imagens com erros compartilhados
#    e demais imagens quanto à presença de
#    pelo menos uma resposta indeterminada.

data["has_indeterminate"] = (
    data["n_indeterminate"] > 0
)

data["shared_error"] = (
    data["both_error"].astype(int) == 1
)

a = int((
    data["shared_error"]
    & data["has_indeterminate"]
).sum())

b = int((
    data["shared_error"]
    & ~data["has_indeterminate"]
).sum())

c = int((
    ~data["shared_error"]
    & data["has_indeterminate"]
).sum())

d = int((
    ~data["shared_error"]
    & ~data["has_indeterminate"]
).sum())

odds_ratio, fisher_p = fisher_exact(
    [[a, b], [c, d]],
    alternative="two-sided"
)

# 4. Comparar erros contrastivos
shared = data[data["shared_error"]]
others = data[~data["shared_error"]]

print("=" * 70)
print("FLORENCE — EXPERIMENTO 17")
print("AUDITORIA DE ROBUSTEZ")
print("=" * 70)

print("\n=== SPEARMAN COM CORREÇÃO DE HOLM ===")
print(
    corr[
        [
            "human_variable",
            "model_variable",
            "spearman_rho",
            "p_exploratory",
            "holm_p",
            "significant_holm"
        ]
    ].to_string(index=False)
)

print("\n=== PERMUTAÇÃO COM CORREÇÃO DE HOLM ===")
print(
    perm[
        [
            "outcome",
            "human_variable",
            "difference",
            "p_permutation",
            "holm_p",
            "significant_holm"
        ]
    ].to_string(index=False)
)

print("\n=== INDETERMINAÇÃO E ERROS COMPARTILHADOS ===")

print("Erros compartilhados com indeterminação:", a)
print("Erros compartilhados sem indeterminação:", b)
print("Demais imagens com indeterminação:", c)
print("Demais imagens sem indeterminação:", d)

print(f"\nOdds ratio: {odds_ratio:.4f}")
print(f"Fisher exato bilateral: p={fisher_p:.8f}")

print("\n=== DIFICULDADE CONTRASTIVA ===")

print(
    "Média de erros contrastivos nas 4 imagens:",
    round(shared["contrastive_errors"].mean(), 4)
)

print(
    "Média nas demais 185 imagens:",
    round(others["contrastive_errors"].mean(), 4)
)

print("\n=== ARQUIVOS ===")

outputs = {
    "florence_exp17_correlations_holm.csv": corr,
    "florence_exp17_permutations_holm.csv": perm
}

for filename, frame in outputs.items():
    path = RESULTS / filename

    if path.exists():
        raise FileExistsError(
            f"Arquivo já existente: {path}"
        )

    frame.to_csv(path, index=False)
    print(path)

print("\nEXPERIMENTO 17 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 17
AUDITORIA DE ROBUSTEZ

=== SPEARMAN COM CORREÇÃO DE HOLM ===
        human_variable         model_variable  spearman_rho  p_exploratory   holm_p  significant_holm
    human_disagreement             qwen_error     -0.058724       0.422172 1.000000             False
    human_disagreement             blip_error      0.022216       0.761562 1.000000             False
    human_disagreement generative_error_count     -0.002201       0.976021 1.000000             False
    human_disagreement     contrastive_errors      0.258396       0.000331 0.002315              True
indeterminate_fraction             qwen_error      0.098242       0.178655 0.714621             False
indeterminate_fraction             blip_error      0.225432       0.001815 0.010893              True
indeterminate_fraction generative_error_count      0.186738       0.010086 0.050430             False
indeterminate_fraction     contrastive_errors      0.285303       0.000069 0.000553              

# Testing Llama-3.2-11B-Vision-Instruct

## Preparing Llama-3.2-11B-Vision-Instruct

In [ ]:

import os
import torch
import importlib.metadata as metadata
from pathlib import Path

print("=" * 65)
print("FLORENCE — EXPERIMENTO 18")
print("DIAGNÓSTICO DO AMBIENTE PARA LLAMA 3.2 VISION")
print("=" * 65)

print("\n=== GPU ===")
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(
        "VRAM total:",
        round(props.total_memory / 1024**3, 2),
        "GiB"
    )
    print(
        "VRAM livre:",
        round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
        "GiB"
    )

print("\n=== BIBLIOTECAS ===")
for package in [
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
    "sentencepiece"
]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NÃO INSTALADO")

print("\n=== DATASET FLORENCE ===")
root = Path("/content/florence_visapp")
results = root / "results"

print("Pasta do projeto:", root.exists())
print("Pasta de resultados:", results.exists())

required = [
    "qwen25_vl3b_predictions.csv",
    "blip2_experiment_10BC_predictions.csv",
    "florence_exp16_integrated.csv",
    "florence_exp17_correlations_holm.csv"
]

for filename in required:
    print(
        filename,
        "OK" if (results / filename).exists()
        else "NÃO ENCONTRADO"
    )

print("\n=== AUTENTICAÇÃO HUGGING FACE ===")

try:
    from huggingface_hub import get_token
    token = get_token()
    print(
        "Token disponível:",
        "SIM" if token else "NÃO"
    )
except Exception as e:
    print("Verificação:", type(e).__name__)

print("\nDIAGNÓSTICO CONCLUÍDO.")


FLORENCE — EXPERIMENTO 18
DIAGNÓSTICO DO AMBIENTE PARA LLAMA 3.2 VISION

=== GPU ===
CUDA disponível: True
GPU: NVIDIA L4
VRAM total: 22.03 GiB
VRAM livre: 21.84 GiB

=== BIBLIOTECAS ===
torch 2.14.1+cu130
transformers 5.22.1
accelerate 1.15.0
bitsandbytes NÃO INSTALADO
huggingface_hub 1.33.0
sentencepiece 0.2.2

=== DATASET FLORENCE ===
Pasta do projeto: True
Pasta de resultados: True
qwen25_vl3b_predictions.csv OK
blip2_experiment_10BC_predictions.csv OK
florence_exp16_integrated.csv NÃO ENCONTRADO
florence_exp17_correlations_holm.csv NÃO ENCONTRADO

=== AUTENTICAÇÃO HUGGING FACE ===
Token disponível: SIM

DIAGNÓSTICO CONCLUÍDO.


## Llama installation and loading

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 18 — ETAPA 2
# Carregamento do Llama 3.2 Vision 11B na NVIDIA L4
# Quantização NF4 — 4 bits
# ============================================================

import sys
import subprocess
import gc
import time

# Instalar somente a dependência ausente
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "bitsandbytes>=0.46.1"
])

import torch
from transformers import (
    MllamaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from huggingface_hub import get_token

MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"

print("=" * 65)
print("FLORENCE — LLAMA 3.2 VISION 11B")
print("=" * 65)

assert torch.cuda.is_available(), "GPU CUDA não disponível"

token = get_token()
assert token is not None, "Token Hugging Face não encontrado"

print("\nGPU:", torch.cuda.get_device_name(0))
print("Modelo:", MODEL_ID)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

gc.collect()
torch.cuda.empty_cache()

start = time.time()

print("\nCarregando processador...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=token
)

print("Carregando modelo quantizado...")

model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=token,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

elapsed = time.time() - start

free, total = torch.cuda.mem_get_info()

print("\n" + "=" * 65)
print("CARREGAMENTO CONCLUÍDO")
print("=" * 65)

print("Tempo:", round(elapsed, 1), "segundos")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total:", round(total / 1024**3, 2), "GiB")
print("VRAM livre:", round(free / 1024**3, 2), "GiB")
print("VRAM utilizada:", round((total-free)/1024**3, 2), "GiB")

print("\nDispositivos do modelo:")
print(model.hf_device_map)

print("\nPronto para o teste visual.")


FLORENCE — LLAMA 3.2 VISION 11B

GPU: NVIDIA L4
Modelo: meta-llama/Llama-3.2-11B-Vision-Instruct

Carregando processador...


preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.09k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]


CARREGAMENTO CONCLUÍDO
Tempo: 95.4 segundos
GPU: NVIDIA L4
VRAM total: 22.03 GiB
VRAM livre: 14.99 GiB
VRAM utilizada: 7.05 GiB

Dispositivos do modelo:


AttributeError: 'MllamaForConditionalGeneration' object has no attribute 'hf_device_map'

## Error above. Nevertheless, can proceed to the image analysis.

Below a test with only three images

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 18 — ETAPA 3
# Teste visual do Llama 3.2 Vision 11B
# NVIDIA L4 | 3 imagens | 3 protocolos
# Sem recarregar o modelo
# ============================================================

import time
import torch
import pandas as pd
from pathlib import Path
from PIL import Image

ROOT = Path("/content/florence_visapp")
RESULTS = ROOT / "results"

assert "model" in globals(), "Modelo não encontrado na memória."
assert "processor" in globals(), "Processador não encontrado."

model.eval()

PROTOCOLS = {
    "A_direct": (
        "Does this person show visible signs of pain? "
        "Answer only yes or no."
    ),
    "B_comparative": (
        "Is this person's facial expression more consistent "
        "with pain or no pain? Answer with pain or no pain."
    ),
    "C_affective": (
        "Which category best describes this person's facial "
        "expression: pain, happiness, sadness, anger, "
        "disgust, or surprise? Answer with one category only."
    )
}

# Referência já utilizada nos experimentos anteriores
reference = pd.read_csv(
    RESULTS / "qwen25_vl3b_predictions.csv"
)

reference = (
    reference[reference["protocol"] == "C_affective"]
    .sort_values("image_number")
    .drop_duplicates("image_number")
)

# Uma imagem para cada protocolo
test_ids = [1, 59, 98]

device = next(model.parameters()).device

print("=" * 68)
print("FLORENCE — LLAMA 3.2 VISION 11B")
print("TESTE DE INFERÊNCIA VISUAL")
print("=" * 68)

print("\nDispositivo:", device)
print("GPU:", torch.cuda.get_device_name(0))

results = []

for image_id, (protocol, question) in zip(
    test_ids, PROTOCOLS.items()
):

    row = reference[
        reference["image_number"] == image_id
    ]

    if row.empty:
        raise ValueError(
            f"Imagem {image_id} não encontrada no CSV."
        )

    row = row.iloc[0]
    image_path = Path(str(row["image_path"]))

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    )

    inputs = {
        k: (
            v.to(device)
            if isinstance(v, torch.Tensor)
            else v
        )
        for k, v in inputs.items()
    }

    print("\n" + "-" * 68)
    print("Imagem:", image_id)
    print("Protocolo:", protocol)
    print("Referência:", int(row["true_label"]))
    print("Pergunta:", question)

    start = time.time()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    elapsed = time.time() - start

    input_length = inputs["input_ids"].shape[-1]

    answer = processor.decode(
        output[0][input_length:],
        skip_special_tokens=True
    ).strip()

    print("Resposta:", repr(answer))
    print("Tempo:", round(elapsed, 2), "s")

    free, total = torch.cuda.mem_get_info()

    print(
        "VRAM livre:",
        round(free / 1024**3, 2),
        "GiB"
    )

    results.append({
        "image_number": image_id,
        "true_label": int(row["true_label"]),
        "protocol": protocol,
        "raw_answer": answer,
        "elapsed_seconds": elapsed
    })

    del inputs, output

print("\n" + "=" * 68)
print("RESUMO DO TESTE")
print("=" * 68)

print(
    pd.DataFrame(results).to_string(index=False)
)

print("\nTESTE VISUAL CONCLUÍDO.")


FLORENCE — LLAMA 3.2 VISION 11B
TESTE DE INFERÊNCIA VISUAL

Dispositivo: cuda:0
GPU: NVIDIA L4

--------------------------------------------------------------------
Imagem: 1
Protocolo: A_direct
Referência: 1
Pergunta: Does this person show visible signs of pain? Answer only yes or no.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Resposta: 'Yes.'
Tempo: 7.46 s
VRAM livre: 14.0 GiB

--------------------------------------------------------------------
Imagem: 59
Protocolo: B_comparative
Referência: 1
Pergunta: Is this person's facial expression more consistent with pain or no pain? Answer with pain or no pain.
Resposta: 'pain.'
Tempo: 1.01 s
VRAM livre: 14.0 GiB

--------------------------------------------------------------------
Imagem: 98
Protocolo: C_affective
Referência: 0
Pergunta: Which category best describes this person's facial expression: pain, happiness, sadness, anger, disgust, or surprise? Answer with one category only.
Resposta: "The person's facial expression appears to be one of pain, as they are touching their face and have"
Tempo: 2.13 s
VRAM livre: 14.0 GiB

RESUMO DO TESTE
 image_number  true_label      protocol                                                                                         raw_answer  elapsed_seconds
            1           1      A_direct                            

## Response format validation

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 18 — ETAPA 4
# Llama 3.2 Vision 11B
# Validação do protocolo afetivo
#
# 6 imagens | Prompt original | 64 tokens
# Sem alterar os resultados anteriores
# ============================================================

import re
import time
import torch
import pandas as pd
from pathlib import Path
from PIL import Image

ROOT = Path("/content/florence_visapp")
RESULTS = ROOT / "results"

assert "model" in globals()
assert "processor" in globals()

model.eval()

PROMPT = (
    "Which category best describes this person's facial "
    "expression: pain, happiness, sadness, anger, disgust, "
    "or surprise? Answer with one category only."
)

CATEGORIES = [
    "pain",
    "happiness",
    "sadness",
    "anger",
    "disgust",
    "surprise"
]

# Conjunto de diagnóstico de formato, não de acurácia
TEST_IDS = [1, 44, 59, 98, 175, 181]

reference = pd.read_csv(
    RESULTS / "qwen25_vl3b_predictions.csv"
)

reference = reference[
    reference["protocol"] == "C_affective"
].drop_duplicates("image_number")

device = next(model.parameters()).device

def parse_strict(answer):
    """
    Aceita apenas uma das seis categorias,
    ignorando maiúsculas e pontuação final.
    """
    normalized = answer.strip().lower()

    normalized = re.sub(
        r"^[\s\"'`]+|[\s\"'`.,!?;:]+$",
        "",
        normalized
    )

    return (
        normalized
        if normalized in CATEGORIES
        else None
    )

results = []

print("=" * 70)
print("FLORENCE — LLAMA 3.2 VISION 11B")
print("VALIDAÇÃO DO PROTOCOLO AFETIVO")
print("=" * 70)

for image_id in TEST_IDS:

    row = reference[
        reference["image_number"] == image_id
    ].iloc[0]

    path = Path(str(row["image_path"]))

    with Image.open(path) as im:
        image = im.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

    start = time.time()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False
        )

    elapsed = time.time() - start

    input_length = inputs["input_ids"].shape[-1]

    answer = processor.decode(
        output[0][input_length:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    parsed = parse_strict(answer)

    results.append({
        "image_number": image_id,
        "true_label": int(row["true_label"]),
        "raw_answer": answer,
        "parsed_category": parsed,
        "valid": parsed is not None,
        "elapsed_seconds": elapsed
    })

    print("\n" + "-" * 70)
    print("Imagem:", image_id)
    print("Referência:", int(row["true_label"]))
    print("Resposta:", repr(answer))
    print("Categoria válida:", parsed)
    print("Tempo:", round(elapsed, 2), "s")

    del inputs, output

df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("RESUMO")
print("=" * 70)

print("Imagens:", len(df))
print("Respostas válidas:", int(df["valid"].sum()))
print(
    "Taxa de validade:",
    f"{df['valid'].mean()*100:.2f}%"
)

print("\nRespostas inválidas:")

invalid = df[~df["valid"]]

if invalid.empty:
    print("Nenhuma.")
else:
    print(
        invalid[
            ["image_number", "raw_answer"]
        ].to_string(index=False)
    )

print("\nVALIDAÇÃO CONCLUÍDA.")


FLORENCE — LLAMA 3.2 VISION 11B
VALIDAÇÃO DO PROTOCOLO AFETIVO

----------------------------------------------------------------------
Imagem: 1
Referência: 1
Resposta: 'anger'
Categoria válida: anger
Tempo: 0.96 s

----------------------------------------------------------------------
Imagem: 44
Referência: 1
Resposta: 'anger'
Categoria válida: anger
Tempo: 0.96 s

----------------------------------------------------------------------
Imagem: 59
Referência: 1
Resposta: 'anger'
Categoria válida: anger
Tempo: 0.96 s

----------------------------------------------------------------------
Imagem: 98
Referência: 0
Resposta: "The person's facial expression appears to be one of pain, as they are touching their face and have a furrowed brow."
Categoria válida: None
Tempo: 2.63 s

----------------------------------------------------------------------
Imagem: 175
Referência: 1
Resposta: 'The person in the image appears to be smiling, which suggests that their facial expression is one of happine

## Complete processing

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 18 — ETAPA 5
# LLAMA 3.2 VISION 11B — FPD-189
#
# 189 imagens × 3 protocolos = 567 inferências
# Quantização NF4 | NVIDIA L4
# Prompt original | Extração conservadora
# Salvamento incremental | Retomada automática
# ============================================================

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import torch
import re
import time
import json
import os

ROOT = Path("/content/florence_visapp")
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

OUTPUT = RESULTS / "llama32_vision11b_predictions.csv"
CONFIG = RESULTS / "llama32_vision11b_protocol.json"

assert "model" in globals(), "Modelo não carregado."
assert "processor" in globals(), "Processador não carregado."

model.eval()

MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"

PROTOCOLS = {
    "A_direct": (
        "Does this person show visible signs of pain? "
        "Answer only yes or no."
    ),
    "B_comparative": (
        "Is this person's facial expression more consistent "
        "with pain or no pain? Answer with pain or no pain."
    ),
    "C_affective": (
        "Which category best describes this person's facial "
        "expression: pain, happiness, sadness, anger, "
        "disgust, or surprise? Answer with one category only."
    )
}

CATEGORIES = {
    "pain": 1,
    "happiness": 0,
    "sadness": 0,
    "anger": 0,
    "disgust": 0,
    "surprise": 0
}

MAX_NEW_TOKENS = 64

# ------------------------------------------------------------
# 1. Referência do dataset
# ------------------------------------------------------------

reference = pd.read_csv(
    RESULTS / "qwen25_vl3b_predictions.csv"
)

reference = (
    reference[reference["protocol"] == "C_affective"]
    .sort_values("image_number")
    .drop_duplicates("image_number")
)

assert len(reference) == 189

# ------------------------------------------------------------
# 2. Parser conservador
# ------------------------------------------------------------

def normalize_answer(text):
    text = str(text).strip().lower()
    return re.sub(
        r"^[\s\"'`]+|[\s\"'`.,!?;:]+$",
        "",
        text
    )

def parse_answer(text, protocol):
    value = normalize_answer(text)

    if protocol == "A_direct":
        if value == "yes":
            return "pain", 1, True
        if value == "no":
            return "no_pain", 0, True

    elif protocol == "B_comparative":
        if value == "pain":
            return "pain", 1, True
        if value in {"no pain", "no_pain", "nopain"}:
            return "no_pain", 0, True

    elif protocol == "C_affective":
        if value in CATEGORIES:
            return value, CATEGORIES[value], True

    return None, None, False

# ------------------------------------------------------------
# 3. Recuperação incremental
# ------------------------------------------------------------

COLUMNS = [
    "model",
    "protocol",
    "image_number",
    "image",
    "image_path",
    "true_label",
    "raw_answer",
    "parsed_category",
    "predicted_label",
    "valid",
    "correct",
    "elapsed_seconds"
]

if OUTPUT.exists():
    previous = pd.read_csv(OUTPUT)
    assert set(COLUMNS).issubset(previous.columns)

    # Verificar integridade antes de retomar
    assert not previous.duplicated(
        ["protocol", "image_number"]
    ).any()

    completed = set(
        zip(
            previous["protocol"],
            previous["image_number"].astype(int)
        )
    )

    print("Resultados recuperados:", len(completed))
else:
    completed = set()
    print("Iniciando nova execução.")

# Salvar protocolo uma única vez
configuration = {
    "model": MODEL_ID,
    "dataset": "FPD-189",
    "n_images": 189,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "parser": "strict_category_only",
    "protocols": PROTOCOLS,
    "affective_mapping": CATEGORIES
}

if not CONFIG.exists():
    with open(CONFIG, "w", encoding="utf-8") as f:
        json.dump(configuration, f, indent=2)

# ------------------------------------------------------------
# 4. Inferência
# ------------------------------------------------------------

device = next(model.parameters()).device

tasks = [
    (protocol, row)
    for protocol in PROTOCOLS
    for _, row in reference.iterrows()
    if (protocol, int(row["image_number"])) not in completed
]

print("\nInferências restantes:", len(tasks))
print("GPU:", torch.cuda.get_device_name(0))

for protocol, row in tqdm(tasks, desc="Florence Llama"):

    image_id = int(row["image_number"])
    image_path = Path(str(row["image_path"]))
    true_label = int(row["true_label"])

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    with Image.open(image_path) as im:
        image = im.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": PROTOCOLS[protocol]
                }
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: (
            v.to(device)
            if isinstance(v, torch.Tensor)
            else v
        )
        for k, v in inputs.items()
    }

    start = time.time()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    elapsed = time.time() - start

    input_length = inputs["input_ids"].shape[-1]

    raw_answer = processor.decode(
        output[0][input_length:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    category, prediction, valid = parse_answer(
        raw_answer, protocol
    )

    record = {
        "model": MODEL_ID,
        "protocol": protocol,
        "image_number": image_id,
        "image": row["image"],
        "image_path": str(image_path),
        "true_label": true_label,
        "raw_answer": raw_answer,
        "parsed_category": category,
        "predicted_label": prediction,
        "valid": valid,
        "correct": (
            prediction == true_label
            if valid else False
        ),
        "elapsed_seconds": elapsed
    }

    # Escrita incremental
    pd.DataFrame(
        [record],
        columns=COLUMNS
    ).to_csv(
        OUTPUT,
        mode="a",
        header=not OUTPUT.exists(),
        index=False
    )

    del inputs, output, image

# ------------------------------------------------------------
# 5. Resumo final
# ------------------------------------------------------------

df = pd.read_csv(OUTPUT)

print("\n" + "=" * 70)
print("FLORENCE — LLAMA 3.2 VISION 11B")
print("RESUMO DAS INFERÊNCIAS")
print("=" * 70)

for protocol in PROTOCOLS:

    subset = df[
        df["protocol"] == protocol
    ].copy()

    valid = (
        subset["valid"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    correct = (
        subset["correct"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    n_valid = int(valid.sum())
    n_correct = int(correct.sum())

    print("\nProtocolo:", protocol)
    print("Imagens:", len(subset))
    print("Respostas válidas:", n_valid)
    print("Respostas inválidas:", len(subset)-n_valid)

    if n_valid:
        print(
            "Accuracy (válidas):",
            f"{100*n_correct/n_valid:.2f}%"
        )

    print(
        "Accuracy (inválidas como erro):",
        f"{100*n_correct/len(subset):.2f}%"
    )

print("\nArquivo:", OUTPUT)
print("\nEXPERIMENTO 18 CONCLUÍDO.")


Iniciando nova execução.

Inferências restantes: 567
GPU: NVIDIA L4


Florence Llama:   0%|          | 0/567 [00:00<?, ?it/s]


FLORENCE — LLAMA 3.2 VISION 11B
RESUMO DAS INFERÊNCIAS

Protocolo: A_direct
Imagens: 189
Respostas válidas: 185
Respostas inválidas: 4
Accuracy (válidas): 76.22%
Accuracy (inválidas como erro): 74.60%

Protocolo: B_comparative
Imagens: 189
Respostas válidas: 186
Respostas inválidas: 3
Accuracy (válidas): 70.43%
Accuracy (inválidas como erro): 69.31%

Protocolo: C_affective
Imagens: 189
Respostas válidas: 163
Respostas inválidas: 26
Accuracy (válidas): 96.32%
Accuracy (inválidas como erro): 83.07%

Arquivo: /content/florence_visapp/results/llama32_vision11b_predictions.csv

EXPERIMENTO 18 CONCLUÍDO.


# Complete metrics and statistical comparison

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 19 (VERSÃO CORRIGIDA)
# Llama 3.2 Vision 11B × Qwen2.5-VL-3B × BLIP-2
# Métricas, McNemar exato, Holm e bootstrap pareado
# Sem novas inferências
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)
from statsmodels.stats.multitest import multipletests

R = Path("/content/florence_visapp/results")
R.mkdir(parents=True, exist_ok=True)

N_BOOT = 10000
SEED = 20261110

llama = pd.read_csv(R / "llama32_vision11b_predictions.csv")
qwen = pd.read_csv(R / "qwen25_vl3b_predictions.csv")
blip = pd.read_csv(R / "blip2_experiment_10BC_predictions.csv")

# ------------------------------------------------------------
# 1. Padronização e validação
# ------------------------------------------------------------

def to_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )

def prepare(df, protocol, name):

    x = df.loc[df["protocol"] == protocol].copy()

    if "valid_response" in x.columns:
        x = x.rename(columns={"valid_response": "valid"})

    required = [
        "image_number", "true_label",
        "predicted_label", "valid"
    ]

    missing = [c for c in required if c not in x.columns]
    if missing:
        raise ValueError(f"{name}: colunas ausentes: {missing}")

    x = x[required].copy()

    if len(x) != 189:
        raise ValueError(
            f"{name}/{protocol}: esperadas 189 linhas; "
            f"encontradas {len(x)}"
        )

    if x["image_number"].duplicated().any():
        raise ValueError(f"{name}: imagens duplicadas")

    x["image_number"] = x["image_number"].astype(int)
    x["true_label"] = x["true_label"].astype(int)
    x["predicted_label"] = pd.to_numeric(
        x["predicted_label"], errors="coerce"
    )
    x["valid"] = to_bool(x["valid"])

    if not x["true_label"].isin([0, 1]).all():
        raise ValueError(f"{name}: referências não binárias")

    if not x.loc[x["valid"], "predicted_label"].isin([0, 1]).all():
        raise ValueError(
            f"{name}: resposta válida sem predição binária"
        )

    if x.loc[~x["valid"], "predicted_label"].notna().any():
        raise ValueError(
            f"{name}: resposta inválida com predição preenchida"
        )

    return x.sort_values("image_number").reset_index(drop=True)

# ------------------------------------------------------------
# 2. Preparar os modelos
# ------------------------------------------------------------

datasets = {
    ("Llama", "A_direct"):
        prepare(llama, "A_direct", "Llama"),

    ("Llama", "B_comparative"):
        prepare(llama, "B_comparative", "Llama"),

    ("Llama", "C_affective"):
        prepare(llama, "C_affective", "Llama"),

    ("Qwen", "A_direct"):
        prepare(qwen, "A_direct", "Qwen"),

    ("Qwen", "B_comparative"):
        prepare(qwen, "B_comparative", "Qwen"),

    ("Qwen", "C_affective"):
        prepare(qwen, "C_affective", "Qwen"),

    ("BLIP-2", "B_comparative"):
        prepare(blip, "10B_binary_comparative", "BLIP-2"),

    ("BLIP-2", "C_affective"):
        prepare(blip, "10C_affective", "BLIP-2")
}

# Validar conjunto de imagens e rótulos em todos os arquivos
reference = datasets[("Llama", "C_affective")][
    ["image_number", "true_label"]
].reset_index(drop=True)

for key, x in datasets.items():
    check = x[["image_number", "true_label"]].reset_index(drop=True)
    if not check.equals(reference):
        raise ValueError(f"Referências inconsistentes: {key}")

print("Integridade verificada: 189 imagens e rótulos coincidentes.")

# ------------------------------------------------------------
# 3. Métricas completas
# ------------------------------------------------------------

metrics = []

for (name, protocol), x in datasets.items():

    valid = x[x["valid"]]

    y = valid["true_label"].to_numpy(dtype=int)
    p = valid["predicted_label"].to_numpy(dtype=int)

    tn, fp, fn, tp = confusion_matrix(
        y, p, labels=[0, 1]
    ).ravel()

    correct = int((y == p).sum())

    metrics.append({
        "model": name,
        "protocol": protocol,
        "n_total": len(x),
        "n_valid": len(valid),
        "n_invalid": len(x) - len(valid),
        "accuracy_valid": accuracy_score(y, p),
        "accuracy_operational": correct / len(x),
        "precision": precision_score(y, p, zero_division=0),
        "recall": recall_score(y, p, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(y, p, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y, p),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

metrics_df = pd.DataFrame(metrics)

# ------------------------------------------------------------
# 4. Comparações pareadas
# ------------------------------------------------------------

def compare(a, b, name_a, name_b, protocol, regime, seed):

    merged = a.merge(
        b,
        on="image_number",
        suffixes=("_a", "_b"),
        validate="one_to_one"
    )

    if not (
        merged["true_label_a"] ==
        merged["true_label_b"]
    ).all():
        raise ValueError("Rótulos incompatíveis")

    if regime == "complete_cases":
        merged = merged.loc[
            merged["valid_a"] & merged["valid_b"]
        ].copy()

    # Inválidas são consideradas incorretas no regime all_189.
    ca = (
        merged["valid_a"] &
        (merged["predicted_label_a"] ==
         merged["true_label_a"])
    ).to_numpy(dtype=bool)

    cb = (
        merged["valid_b"] &
        (merged["predicted_label_b"] ==
         merged["true_label_b"])
    ).to_numpy(dtype=bool)

    n = len(merged)

    a_only = int(np.sum(ca & ~cb))
    b_only = int(np.sum(~ca & cb))
    both_correct = int(np.sum(ca & cb))
    both_wrong = int(np.sum(~ca & ~cb))

    discordant = a_only + b_only

    p_value = (
        binomtest(
            a_only,
            discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
        if discordant else 1.0
    )

    # Bootstrap pareado, preservando a correspondência
    # das imagens e dos resultados dos dois modelos.
    rng = np.random.default_rng(seed)

    delta = ca.astype(int) - cb.astype(int)

    bootstrap = np.empty(N_BOOT)

    for i in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        bootstrap[i] = delta[idx].mean()

    ci_low, ci_high = np.quantile(
        bootstrap, [0.025, 0.975]
    )

    return {
        "comparison": f"{name_a} vs {name_b}",
        "protocol": protocol,
        "regime": regime,
        "n": n,
        "accuracy_a": ca.mean(),
        "accuracy_b": cb.mean(),
        "difference_pp": 100 * delta.mean(),
        "ci_low_pp": 100 * ci_low,
        "ci_high_pp": 100 * ci_high,
        "both_correct": both_correct,
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "both_wrong": both_wrong,
        "mcnemar_exact_p": p_value
    }

comparisons = []
counter = 0

# Família confirmatória: seis pares modelo-protocolo,
# cada um em dois regimes de análise (12 testes).
for protocol in ["B_comparative", "C_affective"]:

    pairs = [
        ("Llama", "Qwen"),
        ("Llama", "BLIP-2"),
        ("Qwen", "BLIP-2")
    ]

    for name_a, name_b in pairs:
        for regime in ["complete_cases", "all_189"]:

            counter += 1

            comparisons.append(
                compare(
                    datasets[(name_a, protocol)],
                    datasets[(name_b, protocol)],
                    name_a,
                    name_b,
                    protocol,
                    regime,
                    SEED + counter
                )
            )

paired_df = pd.DataFrame(comparisons)

paired_df["holm_p"] = multipletests(
    paired_df["mcnemar_exact_p"],
    method="holm"
)[1]

paired_df["significant_holm"] = paired_df["holm_p"] < 0.05

# ------------------------------------------------------------
# 5. Respostas inválidas do Llama
# ------------------------------------------------------------

llama_invalid = llama.loc[
    ~to_bool(llama["valid"]),
    [
        "protocol",
        "image_number",
        "true_label",
        "raw_answer"
    ]
].copy()

llama_invalid = llama_invalid.sort_values(
    ["protocol", "image_number"]
)

# ------------------------------------------------------------
# 6. Salvar arquivos sem sobrescrever resultados
# ------------------------------------------------------------

outputs = {
    "florence_exp19_metrics_all_models.csv": metrics_df,
    "florence_exp19_paired_statistics.csv": paired_df,
    "florence_exp19_llama_invalid_responses.csv": llama_invalid
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes, sem sobrescrita: "
        + ", ".join(existing)
    )

for filename, data in outputs.items():
    data.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 7. Resultados
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("\n" + "=" * 75)
print("EXPERIMENTO 19 — MÉTRICAS COMPLETAS")
print("=" * 75)

print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 75)
print("COMPARAÇÕES PAREADAS — MCNEMAR + HOLM")
print("=" * 75)

print(
    paired_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 75)
print("RESPOSTAS INVÁLIDAS DO LLAMA")
print("=" * 75)

print(
    llama_invalid.groupby("protocol").size()
)

print("\n" + "=" * 75)
print("ARQUIVOS GERADOS")
print("=" * 75)

for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 19 CONCLUÍDO.")


Integridade verificada: 189 imagens e rótulos coincidentes.

EXPERIMENTO 19 — MÉTRICAS COMPLETAS
 model      protocol  n_total  n_valid  n_invalid  accuracy_valid  accuracy_operational  precision   recall  specificity       f1  balanced_accuracy  TN  FP  FN  TP
 Llama      A_direct      189      185          4        0.762162              0.746032   0.786517 0.736842     0.788889 0.760870           0.762865  71  19  25  70
 Llama B_comparative      189      186          3        0.704301              0.693122   0.644737 0.989899     0.379310 0.780876           0.684605  33  54   1  98
 Llama   C_affective      189      163         26        0.963190              0.830688   0.988764 0.946237     0.985714 0.967033           0.965975  69   1   5  88
  Qwen      A_direct      189      189          0        0.798942              0.798942   0.729323 0.979798     0.600000 0.836207           0.789899  54  36   2  97
  Qwen B_comparative      189      189          0        0.804233             

Analysis of Llama's response failures

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 20
# Falhas de resposta do Llama 3.2 Vision 11B
#
# H1: associação com discordância humana
# H2: associação com erros dos demais modelos
# H3: categorias presentes em respostas explicativas
# H4: associação com classe Pain/NoPain
#
# Sem inferência | Análise exploratória
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
from scipy.stats import (
    fisher_exact, mannwhitneyu, spearmanr
)
from statsmodels.stats.multitest import multipletests

ROOT = Path("/content/florence_visapp")
R = ROOT / "results"

llama = pd.read_csv(R / "llama32_vision11b_predictions.csv")
qwen = pd.read_csv(R / "qwen25_vl3b_predictions.csv")
blip = pd.read_csv(R / "blip2_experiment_10BC_predictions.csv")

def bool_col(s):
    return s.astype(str).str.lower().eq("true")

# ------------------------------------------------------------
# 1. Preparar dados por imagem
# ------------------------------------------------------------

def model_table(df, protocol, prefix, valid_col="valid"):
    x = df.loc[df["protocol"] == protocol].copy()

    assert len(x) == 189
    assert x["image_number"].is_unique

    x["is_valid"] = bool_col(x[valid_col])
    x["prediction"] = pd.to_numeric(
        x["predicted_label"], errors="coerce"
    )

    x["is_correct"] = (
        x["is_valid"] &
        (x["prediction"] == x["true_label"])
    )

    return x[[
        "image_number",
        "true_label",
        "is_valid",
        "is_correct"
    ]].rename(columns={
        "is_valid": f"{prefix}_valid",
        "is_correct": f"{prefix}_correct"
    })

L = model_table(llama, "C_affective", "llama")
Q = model_table(qwen, "C_affective", "qwen")
B = model_table(
    blip, "10C_affective", "blip",
    valid_col="valid_response"
)

data = L.merge(
    Q.drop(columns="true_label"),
    on="image_number",
    validate="one_to_one"
).merge(
    B.drop(columns="true_label"),
    on="image_number",
    validate="one_to_one"
)

assert len(data) == 189

# Validar referências entre arquivos
for original, protocol in [
    (qwen, "C_affective"),
    (blip, "10C_affective")
]:
    y = original.loc[
        original["protocol"] == protocol,
        ["image_number", "true_label"]
    ]
    check = data[[
        "image_number", "true_label"
    ]].merge(
        y, on="image_number",
        suffixes=("_base", "_other"),
        validate="one_to_one"
    )
    assert len(check) == 189
    assert (
        check["true_label_base"]
        == check["true_label_other"]
    ).all()

data["llama_invalid"] = ~data["llama_valid"]
data["qwen_error"] = ~data["qwen_correct"]
data["blip_failure"] = ~data["blip_correct"]
data["both_other_fail"] = (
    data["qwen_error"] & data["blip_failure"]
)

# ------------------------------------------------------------
# 2. Classificar respostas inválidas
# ------------------------------------------------------------

affective = [
    "pain", "happiness", "sadness",
    "anger", "disgust", "surprise"
]

def classify_invalid(text):
    text = str(text).lower()

    found = [
        cat for cat in affective
        if re.search(
            r"\b" + re.escape(cat) + r"\b",
            text
        )
    ]

    if len(found) == 0:
        kind = "no_category_detected"
    elif len(found) == 1:
        kind = "one_category_in_explanation"
    else:
        kind = "multiple_categories_in_explanation"

    return kind, "|".join(found)

invalid_raw = llama.loc[
    (llama["protocol"] == "C_affective") &
    (~bool_col(llama["valid"]))
].copy()

classified = invalid_raw["raw_answer"].apply(
    classify_invalid
)

invalid_raw["failure_type"] = classified.apply(
    lambda x: x[0]
)
invalid_raw["mentioned_categories"] = classified.apply(
    lambda x: x[1]
)

# ------------------------------------------------------------
# 3. Testes estatísticos
# ------------------------------------------------------------

tests = []

def fisher_test(outcome, name):
    tab = pd.crosstab(
        data["llama_invalid"],
        data[outcome]
    ).reindex(
        index=[False, True],
        columns=[False, True],
        fill_value=0
    )

    odds, p = fisher_exact(tab.to_numpy())

    tests.append({
        "hypothesis": name,
        "test": "Fisher exact",
        "statistic": odds,
        "p_raw": p,
        "n": len(data)
    })

    return tab

tab_qwen = fisher_test(
    "qwen_error", "H2_Qwen_error"
)
tab_blip = fisher_test(
    "blip_failure", "H2_BLIP_failure"
)
tab_both = fisher_test(
    "both_other_fail", "H2_both_fail"
)

data["pain_class"] = data["true_label"].eq(1)
tab_class = fisher_test(
    "pain_class", "H4_Pain_class"
)

# ------------------------------------------------------------
# 4. Dados humanos, quando disponíveis
# ------------------------------------------------------------

human_candidates = [
    R / "florence_exp16_integrated.csv",
    R / "human_model_error_association.csv",
    R / "human_fpd189_consensus.csv"
]

human_found = [
    p for p in human_candidates if p.exists()
]

human_status = (
    "Arquivos disponíveis: " +
    ", ".join(p.name for p in human_found)
    if human_found
    else "Dados humanos não disponíveis nesta sessão."
)

# Não inferir colunas nem indicadores humanos.
# A integração será realizada após verificar
# a estrutura real dos arquivos encontrados.

# ------------------------------------------------------------
# 5. Correção para múltiplos testes
# ------------------------------------------------------------

tests_df = pd.DataFrame(tests)

tests_df["holm_p"] = multipletests(
    tests_df["p_raw"],
    method="holm"
)[1]

tests_df["significant_holm"] = (
    tests_df["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 6. Descritivas
# ------------------------------------------------------------

summary = []

for label, subset in [
    ("Llama valid", data[~data["llama_invalid"]]),
    ("Llama invalid", data[data["llama_invalid"]])
]:
    summary.append({
        "group": label,
        "n": len(subset),
        "pain_images": int(subset["pain_class"].sum()),
        "qwen_errors": int(subset["qwen_error"].sum()),
        "blip_failures": int(subset["blip_failure"].sum()),
        "both_other_fail": int(
            subset["both_other_fail"].sum()
        ),
        "qwen_error_rate": subset["qwen_error"].mean(),
        "blip_failure_rate": subset["blip_failure"].mean()
    })

summary_df = pd.DataFrame(summary)

# ------------------------------------------------------------
# 7. Salvar
# ------------------------------------------------------------

outputs = {
    "florence_exp20_image_analysis.csv": data,
    "florence_exp20_invalid_response_types.csv":
        invalid_raw,
    "florence_exp20_tests.csv": tests_df,
    "florence_exp20_summary.csv": summary_df
}

existing = [
    filename for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: " +
        ", ".join(existing)
    )

for filename, df in outputs.items():
    df.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 8. Resultados
# ------------------------------------------------------------

print("=" * 72)
print("FLORENCE — EXPERIMENTO 20")
print("=" * 72)

print("\n=== DESCRITIVAS ===")
print(summary_df.to_string(index=False))

print("\n=== TESTES EXPLORATÓRIOS ===")
print(
    tests_df.to_string(
        index=False,
        float_format=lambda v: f"{v:.6f}"
    )
)

print("\n=== TIPOS DE RESPOSTAS INVÁLIDAS ===")
print(
    invalid_raw["failure_type"].value_counts()
)

print("\n=== CATEGORIAS MENCIONADAS ===")
print(
    invalid_raw[
        ["image_number", "true_label",
         "failure_type", "mentioned_categories",
         "raw_answer"]
    ].to_string(
        index=False,
        max_colwidth=100
    )
)

print("\n=== DADOS HUMANOS ===")
print(human_status)

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 20 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 20

=== DESCRITIVAS ===
        group   n  pain_images  qwen_errors  blip_failures  both_other_fail  qwen_error_rate  blip_failure_rate
  Llama valid 163           93            8             20                2         0.049080           0.122699
Llama invalid  26            6            2              9                2         0.076923           0.346154

=== TESTES EXPLORATÓRIOS ===
     hypothesis         test  statistic    p_raw   n   holm_p  significant_holm
  H2_Qwen_error Fisher exact   1.614583 0.630517 189 0.630517             False
H2_BLIP_failure Fisher exact   3.785294 0.007044 189 0.021131              True
   H2_both_fail Fisher exact   6.708333 0.091849 189 0.183697             False
  H4_Pain_class Fisher exact   0.225806 0.001407 189 0.005628              True

=== TIPOS DE RESPOSTAS INVÁLIDAS ===
failure_type
one_category_in_explanation           20
no_category_detected                   4
multiple_categories_in_explanation     2
Name: count, 

## Human integration and class-adjusted analysis

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 20C
# Llama × avaliações humanas × erros dos modelos
# Ajuste por classe Pain/NoPain
# Sem GPU | Sem novas inferências
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import StratifiedTable

R = Path("/content/florence_visapp/results")

human = pd.read_csv(
    R / "human_model_error_association.csv"
)

models = pd.read_csv(
    R / "florence_exp20_image_analysis.csv"
)

# ------------------------------------------------------------
# 1. Integração e validação
# ------------------------------------------------------------

assert len(human) == len(models) == 189
assert human["image_number"].is_unique
assert models["image_number"].is_unique

variables = [
    "human_disagreement",
    "indeterminate_fraction",
    "total_errors"
]

human_columns = [
    "image_number",
    "true_label",
    "n_human",
    "n_indeterminate",
    "human_disagreement",
    "indeterminate_fraction",
    "total_errors"
]

df = models.merge(
    human[human_columns],
    on="image_number",
    suffixes=("_model", "_human"),
    validate="one_to_one"
)

assert len(df) == 189

assert (
    df["true_label_model"]
    == df["true_label_human"]
).all(), "Inconsistência nas classes de referência"

df = df.rename(
    columns={"true_label_model": "true_label"}
).drop(columns="true_label_human")

df["llama_invalid"] = df["llama_invalid"].astype(bool)
df["blip_failure"] = df["blip_failure"].astype(bool)

assert df["llama_invalid"].sum() == 26
assert df["true_label"].sum() == 99

# ------------------------------------------------------------
# 2. Estatísticas descritivas
# ------------------------------------------------------------

descriptive = []

for variable in variables:

    for label, subset in [
        ("valid", df[~df["llama_invalid"]]),
        ("invalid", df[df["llama_invalid"]])
    ]:
        descriptive.append({
            "variable": variable,
            "group": label,
            "n": len(subset),
            "mean": subset[variable].mean(),
            "median": subset[variable].median(),
            "std": subset[variable].std()
        })

descriptive_df = pd.DataFrame(descriptive)

# ------------------------------------------------------------
# 3. Permutação estratificada por classe
# ------------------------------------------------------------

N_PERM = 10000
rng = np.random.default_rng(20261110)

permutation_results = []

classes = df["true_label"].to_numpy()
invalid = df["llama_invalid"].to_numpy()

class_indices = {
    c: np.flatnonzero(classes == c)
    for c in [0, 1]
}

def class_adjusted_difference(values, flags):
    """
    Diferença de médias inválidas - válidas
    dentro de cada classe, agregada com pesos
    proporcionais ao tamanho das classes.
    """
    differences = []
    weights = []

    for c in [0, 1]:
        idx = class_indices[c]
        v = values[idx]
        f = flags[idx]

        if f.sum() == 0 or (~f).sum() == 0:
            raise ValueError(
                "Estrato sem grupos válidos e inválidos"
            )

        differences.append(
            v[f].mean() - v[~f].mean()
        )
        weights.append(len(idx))

    return np.average(differences, weights=weights)

# Mesmas permutações para todos os indicadores
permuted_flags = []

for _ in range(N_PERM):
    flags = invalid.copy()

    for c in [0, 1]:
        idx = class_indices[c]
        flags[idx] = rng.permutation(flags[idx])

    permuted_flags.append(flags)

for variable in variables:

    values = df[variable].to_numpy(dtype=float)

    observed = class_adjusted_difference(
        values, invalid
    )

    null = np.array([
        class_adjusted_difference(values, flags)
        for flags in permuted_flags
    ])

    p = (
        1 + np.sum(np.abs(null) >= abs(observed))
    ) / (N_PERM + 1)

    permutation_results.append({
        "variable": variable,
        "adjusted_difference": observed,
        "permutation_p": p,
        "n_permutations": N_PERM
    })

perm_df = pd.DataFrame(permutation_results)

perm_df["holm_p"] = multipletests(
    perm_df["permutation_p"],
    method="holm"
)[1]

perm_df["significant_holm"] = (
    perm_df["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 4. Llama × BLIP-2: ajuste por classe
# ------------------------------------------------------------

strata_tables = []

for c in [0, 1]:

    subset = df[df["true_label"] == c]

    table = pd.crosstab(
        subset["llama_invalid"],
        subset["blip_failure"]
    ).reindex(
        index=[False, True],
        columns=[False, True],
        fill_value=0
    )

    strata_tables.append(table.to_numpy())

stratified = StratifiedTable(strata_tables)

blip_adjusted = pd.DataFrame([{
    "comparison": "Llama invalid vs BLIP failure",
    "adjustment": "Pain/NoPain",
    "common_odds_ratio": stratified.oddsratio_pooled,
    "ci_low": stratified.oddsratio_pooled_confint()[0],
    "ci_high": stratified.oddsratio_pooled_confint()[1],
    "cmh_p": stratified.test_null_odds().pvalue,
    "homogeneity_p":
        stratified.test_equal_odds().pvalue
}])

# ------------------------------------------------------------
# 5. Detalhamento por classe
# ------------------------------------------------------------

class_summary = []

for c in [0, 1]:
    subset = df[df["true_label"] == c]

    for status in [False, True]:
        group = subset[
            subset["llama_invalid"] == status
        ]

        class_summary.append({
            "class": "Pain" if c == 1 else "NoPain",
            "llama_invalid": status,
            "n": len(group),
            "blip_failure_rate":
                group["blip_failure"].mean(),
            "human_disagreement_mean":
                group["human_disagreement"].mean(),
            "indeterminate_fraction_mean":
                group["indeterminate_fraction"].mean(),
            "contrastive_errors_mean":
                group["total_errors"].mean()
        })

class_df = pd.DataFrame(class_summary)

# ------------------------------------------------------------
# 6. Salvar arquivos
# ------------------------------------------------------------

outputs = {
    "florence_exp20c_integrated.csv": df,
    "florence_exp20c_descriptive.csv": descriptive_df,
    "florence_exp20c_permutation_holm.csv": perm_df,
    "florence_exp20c_blip_class_adjusted.csv":
        blip_adjusted,
    "florence_exp20c_class_summary.csv": class_df
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 7. Exibir resultados
# ------------------------------------------------------------

print("=" * 75)
print("FLORENCE — EXPERIMENTO 20C")
print("=" * 75)

print("\n=== DESCRITIVAS ===")
print(
    descriptive_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== PERMUTAÇÃO AJUSTADA POR CLASSE + HOLM ===")
print(
    perm_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== LLAMA × BLIP-2 — AJUSTE POR CLASSE ===")
print(
    blip_adjusted.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== DESCRITIVAS POR CLASSE ===")
print(
    class_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 20C CONCLUÍDO.")


FLORENCE — EXPERIMENTO 20C

=== DESCRITIVAS ===
              variable   group   n     mean   median      std
    human_disagreement   valid 163 0.020727 0.000000 0.085146
    human_disagreement invalid  26 0.019231 0.000000 0.098058
indeterminate_fraction   valid 163 0.042025 0.000000 0.112038
indeterminate_fraction invalid  26 0.102564 0.000000 0.156893
          total_errors   valid 163 4.177914 3.000000 2.617416
          total_errors invalid  26 4.884615 4.000000 2.643133

=== PERMUTAÇÃO AJUSTADA POR CLASSE + HOLM ===
              variable  adjusted_difference  permutation_p  n_permutations   holm_p  significant_holm
    human_disagreement            -0.008685       0.793621           10000 0.793621             False
indeterminate_fraction             0.083136       0.003800           10000 0.011399              True
          total_errors             1.511630       0.031997           10000 0.063994             False

=== LLAMA × BLIP-2 — AJUSTE POR CLASSE ===
                   

# Restricted Decoding of Llama 3.2 Vision 11B


In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 21C
# LLAMA 3.2 VISION 11B — DECODIFICAÇÃO RESTRITA
# FPD-189 | 189 imagens | Salvamento incremental
# ============================================================

import torch
import pandas as pd
import numpy as np
import time
import csv
from pathlib import Path
from PIL import Image
from transformers import LogitsProcessor, LogitsProcessorList

R = Path("/content/florence_visapp/results")

SOURCE = R / "llama32_vision11b_predictions.csv"
OUTPUT = R / "llama32_vision11b_restricted_predictions.csv"

assert "model" in globals()
assert "processor" in globals()

tok = processor.tokenizer

categories = [
    "pain", "happiness", "sadness",
    "anger", "disgust", "surprise"
]

PROMPT = (
    "Which category best describes this person's facial "
    "expression: pain, happiness, sadness, anger, disgust, "
    "or surprise? Answer with one category only."
)

# ------------------------------------------------------------
# 1. Construir sequências permitidas
# ------------------------------------------------------------

sequences = {}

for category in categories:
    for text in [category, " " + category]:
        ids = tuple(
            tok.encode(text, add_special_tokens=False)
        )
        if ids:
            if ids in sequences and sequences[ids] != category:
                raise ValueError("Conflito de tokenização")
            sequences[ids] = category

assert len(sequences) == 12

eos_id = tok.eos_token_id

class CategoryConstraint(LogitsProcessor):

    def __init__(self, allowed, start_length, eos_id):
        self.allowed = list(allowed)
        self.start_length = start_length
        self.eos_id = eos_id

    def __call__(self, input_ids, scores):

        if input_ids.shape[0] != 1:
            raise ValueError("Use batch_size=1")

        prefix = tuple(
            input_ids[0, self.start_length:].tolist()
        )

        next_ids = set()

        for sequence in self.allowed:
            if sequence[:len(prefix)] == prefix:

                if len(prefix) == len(sequence):
                    next_ids.add(self.eos_id)

                elif len(prefix) < len(sequence):
                    next_ids.add(sequence[len(prefix)])

        if not next_ids:
            raise RuntimeError(
                f"Prefixo inválido: {prefix}"
            )

        constrained = torch.full_like(
            scores, float("-inf")
        )

        for token_id in next_ids:
            constrained[:, token_id] = scores[:, token_id]

        return constrained

# ------------------------------------------------------------
# 2. Carregar imagens e resultados anteriores
# ------------------------------------------------------------

original = pd.read_csv(SOURCE)

original = original.loc[
    original["protocol"] == "C_affective"
].copy()

original = original.sort_values("image_number")

assert len(original) == 189
assert original["image_number"].is_unique

columns = [
    "model",
    "protocol",
    "image_number",
    "true_label",
    "original_valid",
    "original_answer",
    "original_prediction",
    "restricted_category",
    "restricted_prediction",
    "restricted_correct",
    "generated_token_ids",
    "elapsed_seconds"
]

if OUTPUT.exists():

    previous = pd.read_csv(OUTPUT)

    if not set(columns).issubset(previous.columns):
        raise ValueError(
            "Arquivo de retomada incompatível"
        )

    if previous["image_number"].duplicated().any():
        raise ValueError(
            "Duplicação no arquivo de retomada"
        )

    completed = set(
        previous["image_number"].astype(int)
    )

    print(
        f"Retomando execução: {len(completed)} "
        "imagens já processadas."
    )

else:

    completed = set()

    with open(OUTPUT, "w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=columns
        )
        writer.writeheader()

# ------------------------------------------------------------
# 3. Executar inferências
# ------------------------------------------------------------

model.eval()

start_time = time.time()

for _, row in original.iterrows():

    image_id = int(row["image_number"])

    if image_id in completed:
        continue

    t0 = time.time()

    with Image.open(row["image_path"]) as im:
        image = im.convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": PROMPT}
        ]
    }]

    chat_text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        images=image,
        text=chat_text,
        return_tensors="pt"
    ).to("cuda")

    start_length = inputs["input_ids"].shape[1]

    constraint = CategoryConstraint(
        sequences.keys(),
        start_length,
        eos_id
    )

    with torch.inference_mode():

        generated = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=12,
            logits_processor=LogitsProcessorList(
                [constraint]
            ),
            pad_token_id=eos_id
        )

    new_tokens = generated[
        0, start_length:
    ].tolist()

    if new_tokens and new_tokens[-1] == eos_id:
        new_tokens = new_tokens[:-1]

    token_tuple = tuple(new_tokens)

    if token_tuple not in sequences:
        raise RuntimeError(
            f"Saída inválida na imagem {image_id}: "
            f"{new_tokens}"
        )

    category = sequences[token_tuple]

    prediction = int(category == "pain")
    truth = int(row["true_label"])

    record = {
        "model": "meta-llama/Llama-3.2-11B-Vision-Instruct",
        "protocol": "C_affective_restricted",
        "image_number": image_id,
        "true_label": truth,
        "original_valid": str(row["valid"]).lower() == "true",
        "original_answer": row["raw_answer"],
        "original_prediction": row["predicted_label"],
        "restricted_category": category,
        "restricted_prediction": prediction,
        "restricted_correct": prediction == truth,
        "generated_token_ids": str(new_tokens),
        "elapsed_seconds": time.time() - t0
    }

    with open(OUTPUT, "a", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=columns
        )
        writer.writerow(record)

    completed.add(image_id)

    if len(completed) % 20 == 0 or len(completed) == 189:
        print(
            f"Processadas: {len(completed)}/189 | "
            f"Última imagem: {image_id} | "
            f"Tempo acumulado: "
            f"{(time.time()-start_time)/60:.1f} min"
        )

# ------------------------------------------------------------
# 4. Verificação final
# ------------------------------------------------------------

results = pd.read_csv(OUTPUT)

assert len(results) == 189
assert results["image_number"].is_unique
assert set(results["image_number"]) == set(
    original["image_number"]
)

accuracy = (
    results["restricted_correct"]
    .astype(str)
    .str.lower()
    .eq("true")
    .mean()
)

print("\n" + "=" * 70)
print("EXPERIMENTO 21C — RESULTADO PRELIMINAR")
print("=" * 70)

print("Total:", len(results))
print("Respostas válidas:", len(results))
print("Acurácia operacional:", f"{accuracy:.6f}")

print("\nDistribuição das categorias:")
print(
    results["restricted_category"]
    .value_counts()
    .to_string()
)

print("\nArquivo salvo:")
print(OUTPUT)

print("\nEXPERIMENTO 21C CONCLUÍDO.")


Processadas: 20/189 | Última imagem: 20 | Tempo acumulado: 0.3 min
Processadas: 40/189 | Última imagem: 40 | Tempo acumulado: 0.7 min
Processadas: 60/189 | Última imagem: 60 | Tempo acumulado: 1.0 min
Processadas: 80/189 | Última imagem: 80 | Tempo acumulado: 1.3 min
Processadas: 100/189 | Última imagem: 100 | Tempo acumulado: 1.6 min
Processadas: 120/189 | Última imagem: 120 | Tempo acumulado: 2.0 min
Processadas: 140/189 | Última imagem: 140 | Tempo acumulado: 2.3 min
Processadas: 160/189 | Última imagem: 170 | Tempo acumulado: 2.6 min
Processadas: 180/189 | Última imagem: 190 | Tempo acumulado: 3.0 min
Processadas: 189/189 | Última imagem: 199 | Tempo acumulado: 3.1 min

EXPERIMENTO 21C — RESULTADO PRELIMINAR
Total: 189
Respostas válidas: 189
Acurácia operacional: 0.936508

Distribuição das categorias:
restricted_category
pain         93
anger        30
surprise     29
sadness      19
happiness    17
disgust       1

Arquivo salvo:
/content/florence_visapp/results/llama32_vision11b_

# Statistical evaluation Restricted Llama × Original Llama and Restricted Llama × Qwen

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 21D
# Avaliação estatística da decodificação restrita
# Llama original × Llama restrito × Qwen
# Sem novas inferências
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score,
    f1_score, balanced_accuracy_score
)
from statsmodels.stats.multitest import multipletests

R = Path("/content/florence_visapp/results")

original = pd.read_csv(
    R / "llama32_vision11b_predictions.csv"
)
restricted = pd.read_csv(
    R / "llama32_vision11b_restricted_predictions.csv"
)
qwen = pd.read_csv(
    R / "qwen25_vl3b_predictions.csv"
)

def boolean(s):
    return s.astype(str).str.lower().eq("true")

original = original[
    original["protocol"] == "C_affective"
].copy()

qwen = qwen[
    qwen["protocol"] == "C_affective"
].copy()

# ------------------------------------------------------------
# 1. Preparar dados
# ------------------------------------------------------------

a = original[[
    "image_number", "true_label",
    "predicted_label", "valid"
]].rename(columns={
    "predicted_label": "original_prediction",
    "valid": "original_valid"
})

b = restricted[[
    "image_number", "true_label",
    "restricted_prediction"
]].rename(columns={
    "true_label": "true_label_restricted"
})

c = qwen[[
    "image_number", "true_label",
    "predicted_label", "valid"
]].rename(columns={
    "true_label": "true_label_qwen",
    "predicted_label": "qwen_prediction",
    "valid": "qwen_valid"
})

df = a.merge(
    b, on="image_number", validate="one_to_one"
).merge(
    c, on="image_number", validate="one_to_one"
)

assert len(df) == 189
assert df["image_number"].is_unique

assert (
    df["true_label"] == df["true_label_restricted"]
).all()

assert (
    df["true_label"] == df["true_label_qwen"]
).all()

df["original_valid"] = boolean(df["original_valid"])
df["qwen_valid"] = boolean(df["qwen_valid"])

df["original_correct"] = (
    df["original_valid"] &
    (df["original_prediction"] == df["true_label"])
)

df["restricted_correct"] = (
    df["restricted_prediction"] == df["true_label"]
)

df["qwen_correct"] = (
    df["qwen_valid"] &
    (df["qwen_prediction"] == df["true_label"])
)

# ------------------------------------------------------------
# 2. Métricas completas
# ------------------------------------------------------------

metrics = []

for name, pred_col, valid_col in [
    ("Llama original", "original_prediction", "original_valid"),
    ("Llama restricted", "restricted_prediction", None),
    ("Qwen", "qwen_prediction", "qwen_valid")
]:

    valid = (
        np.ones(len(df), dtype=bool)
        if valid_col is None
        else df[valid_col].to_numpy()
    )

    subset = df.loc[valid]

    y = subset["true_label"].astype(int)
    p = subset[pred_col].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, p, labels=[0, 1]
    ).ravel()

    correct = (y == p).sum()

    metrics.append({
        "model": name,
        "n_valid": len(subset),
        "n_invalid": len(df) - len(subset),
        "accuracy_valid": accuracy_score(y, p),
        "accuracy_operational": correct / len(df),
        "precision": precision_score(y, p, zero_division=0),
        "recall": recall_score(y, p, zero_division=0),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y, p, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y, p),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    })

metrics_df = pd.DataFrame(metrics)

# ------------------------------------------------------------
# 3. McNemar exato + bootstrap pareado
# ------------------------------------------------------------

def paired_test(col_a, col_b, name, seed):

    ca = df[col_a].to_numpy(dtype=bool)
    cb = df[col_b].to_numpy(dtype=bool)

    a_only = int((ca & ~cb).sum())
    b_only = int((~ca & cb).sum())

    discordant = a_only + b_only

    p = (
        binomtest(a_only, discordant, 0.5).pvalue
        if discordant else 1.0
    )

    delta = ca.astype(int) - cb.astype(int)

    rng = np.random.default_rng(seed)

    boot = np.empty(10000)

    for i in range(10000):
        idx = rng.integers(
            0, len(delta), len(delta)
        )
        boot[i] = delta[idx].mean()

    low, high = np.quantile(
        boot, [0.025, 0.975]
    )

    return {
        "comparison": name,
        "n": len(df),
        "accuracy_a": ca.mean(),
        "accuracy_b": cb.mean(),
        "difference_pp": 100 * delta.mean(),
        "ci_low_pp": 100 * low,
        "ci_high_pp": 100 * high,
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "mcnemar_p": p
    }

tests = pd.DataFrame([
    paired_test(
        "restricted_correct",
        "original_correct",
        "Llama restricted vs original",
        2101
    ),
    paired_test(
        "restricted_correct",
        "qwen_correct",
        "Llama restricted vs Qwen",
        2102
    )
])

tests["holm_p"] = multipletests(
    tests["mcnemar_p"],
    method="holm"
)[1]

tests["significant_holm"] = tests["holm_p"] < 0.05

# ------------------------------------------------------------
# 4. Analisar as 26 respostas inválidas
# ------------------------------------------------------------

recovered = df.loc[
    ~df["original_valid"]
].copy()

recovery_summary = pd.DataFrame([{
    "n_originally_invalid": len(recovered),
    "restricted_correct": int(
        recovered["restricted_correct"].sum()
    ),
    "restricted_wrong": int(
        (~recovered["restricted_correct"]).sum()
    ),
    "recovery_accuracy": recovered[
        "restricted_correct"
    ].mean()
}])

# ------------------------------------------------------------
# 5. Mudanças nas imagens originalmente válidas
# ------------------------------------------------------------

previously_valid = df.loc[
    df["original_valid"]
].copy()

changes = previously_valid.loc[
    previously_valid["original_prediction"] !=
    previously_valid["restricted_prediction"]
].copy()

# ------------------------------------------------------------
# 6. Salvar
# ------------------------------------------------------------

outputs = {
    "florence_exp21d_metrics.csv": metrics_df,
    "florence_exp21d_paired_tests.csv": tests,
    "florence_exp21d_recovery.csv": recovery_summary,
    "florence_exp21d_changed_predictions.csv": changes,
    "florence_exp21d_image_comparison.csv": df
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 7. Resultados
# ------------------------------------------------------------

print("=" * 75)
print("FLORENCE — EXPERIMENTO 21D")
print("=" * 75)

print("\n=== MÉTRICAS ===")
print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== TESTES PAREADOS ===")
print(
    tests.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== RECUPERAÇÃO DAS 26 RESPOSTAS INVÁLIDAS ===")
print(recovery_summary.to_string(index=False))

print("\n=== MUDANÇAS EM RESPOSTAS ORIGINALMENTE VÁLIDAS ===")
print("Total de mudanças:", len(changes))

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 21D CONCLUÍDO.")


FLORENCE — EXPERIMENTO 21D

=== MÉTRICAS ===
           model  n_valid  n_invalid  accuracy_valid  accuracy_operational  precision   recall  specificity       f1  balanced_accuracy  TN  FP  FN  TP
  Llama original      163         26        0.963190              0.830688   0.988764 0.946237     0.985714 0.967033           0.965975  69   1   5  88
Llama restricted      189          0        0.936508              0.936508   0.967742 0.909091     0.966667 0.937500           0.937879  87   3   9  90
            Qwen      189          0        0.947090              0.947090   0.968421 0.929293     0.966667 0.948454           0.947980  87   3   7  92

=== TESTES PAREADOS ===
                  comparison   n  accuracy_a  accuracy_b  difference_pp  ci_low_pp  ci_high_pp  a_only_correct  b_only_correct  mcnemar_p   holm_p  significant_holm
Llama restricted vs original 189    0.936508    0.830688      10.582011   6.349206   15.343915              20               0   0.000002 0.000004           

# Consolidated Statistical Comparison of Generative Models

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 22 (VERSÃO CORRIGIDA)
# Comparação estatística consolidada — FPD-189
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import confusion_matrix

R = Path("/content/florence_visapp/results")

# ------------------------------------------------------------
# 1. Carregar resultados
# ------------------------------------------------------------

llama = pd.read_csv(
    R / "llama32_vision11b_predictions.csv"
)

llama_r = pd.read_csv(
    R / "llama32_vision11b_restricted_predictions.csv"
)

qwen = pd.read_csv(
    R / "qwen25_vl3b_predictions.csv"
)

blip = pd.read_csv(
    R / "blip2_experiment_10BC_predictions.csv"
)

def to_bool(s):
    values = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False
    }
    if not values.isin(mapping).all():
        raise ValueError(
            f"Valores booleanos inesperados: {values.unique()}"
        )
    return values.map(mapping).astype(bool)

def prepare(data, name, protocol, prediction, validity=None):

    d = data.loc[data["protocol"] == protocol].copy()

    if len(d) != 189:
        raise ValueError(
            f"{name}: esperado 189 registros, obtidos {len(d)}"
        )

    if d["image_number"].duplicated().any():
        raise ValueError(f"{name}: imagens duplicadas")

    d["image_number"] = pd.to_numeric(
        d["image_number"], errors="raise"
    ).astype(int)

    d["true_label"] = pd.to_numeric(
        d["true_label"], errors="raise"
    ).astype(int)

    d["prediction"] = pd.to_numeric(
        d[prediction], errors="coerce"
    )

    # Quando validity=None, todas as respostas devem
    # conter uma classificação binária válida.
    if validity is None:
        if not d["prediction"].isin([0, 1]).all():
            raise ValueError(
                f"{name}: predições inválidas"
            )
        d["is_valid"] = True
    else:
        d["is_valid"] = to_bool(d[validity])

        if not d.loc[
            d["is_valid"], "prediction"
        ].isin([0, 1]).all():
            raise ValueError(
                f"{name}: predições válidas fora de 0/1"
            )

    if not d["true_label"].isin([0, 1]).all():
        raise ValueError(f"{name}: rótulos inválidos")

    d["correct"] = (
        d["is_valid"]
        & d["prediction"].eq(d["true_label"])
    )

    return d[[
        "image_number", "true_label",
        "prediction", "is_valid", "correct"
    ]].rename(columns={
        "true_label": f"{name}_truth",
        "prediction": f"{name}_prediction",
        "is_valid": f"{name}_valid",
        "correct": f"{name}_correct"
    })

# ------------------------------------------------------------
# 2. Preparar quatro condições
# ------------------------------------------------------------

datasets = {
    "Qwen": prepare(
        qwen, "Qwen", "C_affective",
        "predicted_label", "valid"
    ),

    "Llama_restricted": prepare(
        llama_r, "Llama_restricted",
        "C_affective_restricted",
        "restricted_prediction",
        validity=None
    ),

    "BLIP2": prepare(
        blip, "BLIP2", "10C_affective",
        "predicted_label", "valid_response"
    ),

    "Llama_original": prepare(
        llama, "Llama_original", "C_affective",
        "predicted_label", "valid"
    )
}

names = list(datasets)

df = datasets[names[0]].copy()

for name in names[1:]:
    df = df.merge(
        datasets[name],
        on="image_number",
        validate="one_to_one"
    )

assert len(df) == 189
assert df["image_number"].is_unique

truth = df["Qwen_truth"].to_numpy(dtype=int)

for name in names:
    assert np.array_equal(
        df[f"{name}_truth"].to_numpy(dtype=int),
        truth
    ), f"Rótulos divergentes: {name}"

print("Integridade dos dados: OK — 189 imagens alinhadas.")

# ------------------------------------------------------------
# 3. Métricas descritivas
# ------------------------------------------------------------

metrics = []

for name in names:

    valid = df[f"{name}_valid"].to_numpy(dtype=bool)
    correct = df[f"{name}_correct"].to_numpy(dtype=bool)

    y = truth[valid]
    p = df.loc[
        valid, f"{name}_prediction"
    ].to_numpy(dtype=int)

    tn, fp, fn, tp = confusion_matrix(
        y, p, labels=[0, 1]
    ).ravel()

    precision = (
        tp / (tp + fp) if (tp + fp) else np.nan
    )
    recall = (
        tp / (tp + fn) if (tp + fn) else np.nan
    )
    specificity = (
        tn / (tn + fp) if (tn + fp) else np.nan
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else np.nan
    )

    metrics.append({
        "condition": name,
        "n_total": 189,
        "n_valid": int(valid.sum()),
        "n_invalid": int((~valid).sum()),
        "n_correct": int(correct.sum()),
        "accuracy_valid": correct.sum() / valid.sum(),
        "accuracy_operational": correct.mean(),
        "precision_valid": precision,
        "sensitivity_valid": recall,
        "specificity_valid": specificity,
        "f1_valid": f1,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    })

metrics_df = pd.DataFrame(metrics).sort_values(
    "accuracy_operational", ascending=False
)

# ------------------------------------------------------------
# 4. McNemar exato e bootstrap pareado
# ------------------------------------------------------------

def compare(a_name, b_name, seed):

    a = df[f"{a_name}_correct"].to_numpy(dtype=bool)
    b = df[f"{b_name}_correct"].to_numpy(dtype=bool)

    a_only = int((a & ~b).sum())
    b_only = int((~a & b).sum())

    discordant = a_only + b_only

    p_value = (
        binomtest(a_only, discordant, 0.5).pvalue
        if discordant else 1.0
    )

    delta = a.astype(int) - b.astype(int)

    rng = np.random.default_rng(seed)

    # Bootstrap pareado com 20.000 reamostragens
    bootstrap = np.empty(20000)

    for i in range(20000):
        idx = rng.integers(0, len(delta), len(delta))
        bootstrap[i] = delta[idx].mean()

    low, high = np.quantile(
        bootstrap, [0.025, 0.975]
    )

    return {
        "condition_a": a_name,
        "condition_b": b_name,
        "n": len(df),
        "accuracy_a": a.mean(),
        "accuracy_b": b.mean(),
        "difference_pp": 100 * delta.mean(),
        "ci95_low_pp": 100 * low,
        "ci95_high_pp": 100 * high,
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "discordant": discordant,
        "mcnemar_exact_p": p_value
    }

pairs = list(combinations(names, 2))

tests = pd.DataFrame([
    compare(a, b, 2200 + i)
    for i, (a, b) in enumerate(pairs)
])

tests["holm_p"] = multipletests(
    tests["mcnemar_exact_p"],
    method="holm"
)[1]

tests["significant_holm_005"] = tests["holm_p"] < 0.05

tests = tests.sort_values("holm_p").reset_index(drop=True)

# ------------------------------------------------------------
# 5. Concordância entre os quatro sistemas
# ------------------------------------------------------------

correct_cols = [
    f"{name}_correct" for name in names
]

df["n_models_correct"] = (
    df[correct_cols].sum(axis=1)
)

agreement = (
    df["n_models_correct"]
    .value_counts()
    .sort_index()
    .rename_axis("n_models_correct")
    .reset_index(name="n_images")
)

# ------------------------------------------------------------
# 6. Verificações de consistência com Exp. 21
# ------------------------------------------------------------

expected = {
    "Qwen": (189, 179),
    "Llama_restricted": (189, 177),
    "BLIP2": (180, 160),
    "Llama_original": (163, 157)
}

for name, (n_valid, n_correct) in expected.items():

    row = metrics_df.loc[
        metrics_df["condition"] == name
    ].iloc[0]

    assert int(row["n_valid"]) == n_valid, (
        f"{name}: divergência de validade"
    )

    assert int(row["n_correct"]) == n_correct, (
        f"{name}: divergência de acertos"
    )

print("Consistência com experimentos anteriores: OK.")

# ------------------------------------------------------------
# 7. Salvar resultados sem sobrescrever
# ------------------------------------------------------------

outputs = {
    "florence_exp22_metrics.csv": metrics_df,
    "florence_exp22_pairwise_mcnemar_holm.csv": tests,
    "florence_exp22_image_level.csv": df,
    "florence_exp22_agreement.csv": agreement
}

existing = [
    filename for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 8. Exibir resultados
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FLORENCE — EXPERIMENTO 22")
print("=" * 90)

print("\n=== MÉTRICAS ===")
print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== SEIS COMPARAÇÕES ESTATÍSTICAS ===")
print(
    tests.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== CONCORDÂNCIA ENTRE SISTEMAS ===")
print(agreement.to_string(index=False))

print("\n=== ARQUIVOS SALVOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 22 CONCLUÍDO.")


Integridade dos dados: OK — 189 imagens alinhadas.
Consistência com experimentos anteriores: OK.

FLORENCE — EXPERIMENTO 22

=== MÉTRICAS ===
       condition  n_total  n_valid  n_invalid  n_correct  accuracy_valid  accuracy_operational  precision_valid  sensitivity_valid  specificity_valid  f1_valid  TN  FP  FN  TP
            Qwen      189      189          0        179        0.947090              0.947090         0.968421           0.929293           0.966667  0.948454  87   3   7  92
Llama_restricted      189      189          0        177        0.936508              0.936508         0.967742           0.909091           0.966667  0.937500  87   3   9  90
           BLIP2      189      180          9        160        0.888889              0.846561         0.846847           0.969072           0.795181  0.903846  66  17   3  94
  Llama_original      189      163         26        157        0.963190              0.830688         0.988764           0.946237           0.985714  0.9

# Analysis of shared failures and their relationship to human evaluation.

## Integrated analysis

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 23A
# Falhas compartilhadas × Avaliação humana
# FPD-189 | Sem novas inferências
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

R = Path("/content/florence_visapp/results")

MODEL_FILE = R / "florence_exp22_image_level.csv"
HUMAN_FILE = R / "human_model_error_association.csv"

assert MODEL_FILE.exists(), f"Ausente: {MODEL_FILE}"
assert HUMAN_FILE.exists(), f"Ausente: {HUMAN_FILE}"

models = pd.read_csv(MODEL_FILE)
human = pd.read_csv(HUMAN_FILE)

# ------------------------------------------------------------
# 1. Validar e integrar os dados
# ------------------------------------------------------------

required_models = [
    "image_number",
    "Qwen_truth",
    "Qwen_correct",
    "Llama_restricted_correct",
    "BLIP2_correct",
    "Llama_original_correct",
    "n_models_correct"
]

required_human = [
    "image_number",
    "true_label",
    "n_human",
    "human_disagreement",
    "indeterminate_fraction",
    "reference_agreement_fraction",
    "n_indeterminate"
]

for col in required_models:
    assert col in models.columns, f"Coluna ausente: {col}"

for col in required_human:
    assert col in human.columns, f"Coluna ausente: {col}"

assert len(models) == 189
assert len(human) == 189

assert models["image_number"].is_unique
assert human["image_number"].is_unique

df = models.merge(
    human[required_human],
    on="image_number",
    validate="one_to_one"
)

assert len(df) == 189
assert (
    df["Qwen_truth"] == df["true_label"]
).all()

df["n_models_wrong"] = 4 - df["n_models_correct"]

# ------------------------------------------------------------
# 2. Distribuição das falhas compartilhadas
# ------------------------------------------------------------

distribution = (
    df.groupby("n_models_wrong")
    .agg(
        n_images=("image_number", "size"),
        mean_human_disagreement=(
            "human_disagreement", "mean"
        ),
        mean_indeterminate_fraction=(
            "indeterminate_fraction", "mean"
        ),
        mean_reference_agreement=(
            "reference_agreement_fraction", "mean"
        ),
        mean_human_ratings=("n_human", "mean")
    )
    .reset_index()
    .sort_values("n_models_wrong")
)

# ------------------------------------------------------------
# 3. Imagens com falhas compartilhadas
# ------------------------------------------------------------

all_wrong = df.loc[
    df["n_models_wrong"] == 4
].copy()

high_difficulty = df.loc[
    df["n_models_wrong"] >= 2
].copy()

columns_detail = [
    "image_number",
    "true_label",
    "n_models_wrong",
    "n_human",
    "human_disagreement",
    "indeterminate_fraction",
    "reference_agreement_fraction",
    "Qwen_correct",
    "Llama_restricted_correct",
    "BLIP2_correct",
    "Llama_original_correct"
]

all_wrong = all_wrong[columns_detail].sort_values(
    "image_number"
)

high_difficulty = high_difficulty[
    columns_detail
].sort_values(
    ["n_models_wrong", "image_number"],
    ascending=[False, True]
)

# ------------------------------------------------------------
# 4. Correlações de Spearman
# ------------------------------------------------------------

variables = [
    "human_disagreement",
    "indeterminate_fraction",
    "reference_agreement_fraction"
]

correlations = []

for variable in variables:

    rho, p = spearmanr(
        df["n_models_wrong"],
        df[variable]
    )

    correlations.append({
        "variable": variable,
        "spearman_rho": rho,
        "p_unadjusted": p
    })

correlations = pd.DataFrame(correlations)

# ------------------------------------------------------------
# 5. Permutação estratificada por classe
# ------------------------------------------------------------

# Estatística: correlação de Spearman.
# Permutamos o desfecho humano dentro de cada classe,
# preservando a composição Pain/NoPain.
#
# O teste avalia associação condicional à classe.
# Não pressupõe normalidade.

N_PERM = 10000
rng = np.random.default_rng(2301)

classes = df["true_label"].to_numpy()
errors = df["n_models_wrong"].to_numpy()

indices_by_class = [
    np.where(classes == c)[0]
    for c in np.unique(classes)
]

permutation_results = []

for variable in variables:

    values = df[variable].to_numpy(dtype=float)

    observed = spearmanr(errors, values).statistic

    null_stats = np.empty(N_PERM)

    for i in range(N_PERM):

        permuted = values.copy()

        for idx in indices_by_class:
            permuted[idx] = rng.permutation(
                values[idx]
            )

        null_stats[i] = spearmanr(
            errors, permuted
        ).statistic

    # Teste bilateral centrado na distribuição nula
    center = np.mean(null_stats)

    p_perm = (
        1 + np.sum(
            np.abs(null_stats - center)
            >= np.abs(observed - center)
        )
    ) / (N_PERM + 1)

    permutation_results.append({
        "variable": variable,
        "observed_spearman_rho": observed,
        "null_mean": center,
        "permutation_p": p_perm
    })

permutation_results = pd.DataFrame(
    permutation_results
)

permutation_results["holm_p"] = multipletests(
    permutation_results["permutation_p"],
    method="holm"
)[1]

permutation_results["significant_holm"] = (
    permutation_results["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 6. Resumo por classe real
# ------------------------------------------------------------

class_summary = (
    df.groupby(["true_label", "n_models_wrong"])
    .agg(
        n_images=("image_number", "size"),
        mean_disagreement=(
            "human_disagreement", "mean"
        ),
        mean_indeterminate=(
            "indeterminate_fraction", "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Salvar resultados
# ------------------------------------------------------------

outputs = {
    "florence_exp23_integrated.csv": df,
    "florence_exp23_distribution.csv": distribution,
    "florence_exp23_all_wrong.csv": all_wrong,
    "florence_exp23_high_difficulty.csv": high_difficulty,
    "florence_exp23_spearman.csv": correlations,
    "florence_exp23_permutation_holm.csv": permutation_results,
    "florence_exp23_class_summary.csv": class_summary
}

existing = [
    filename for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 8. Apresentar resultados
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("=" * 85)
print("FLORENCE — EXPERIMENTO 23A")
print("=" * 85)

print("\n=== DISTRIBUIÇÃO DAS FALHAS ===")
print(
    distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== IMAGENS EM QUE TODOS OS MODELOS FALHARAM ===")
print(
    all_wrong.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== CORRELAÇÕES DE SPEARMAN ===")
print(
    correlations.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== PERMUTAÇÃO E CORREÇÃO DE HOLM ===")
print(
    permutation_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== RESUMO POR CLASSE ===")
print(
    class_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== IMAGENS COM 2 OU MAIS FALHAS ===")
print("Total:", len(high_difficulty))
print(
    high_difficulty.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 23A CONCLUÍDO.")


FLORENCE — EXPERIMENTO 23A

=== DISTRIBUIÇÃO DAS FALHAS ===
 n_models_wrong  n_images  mean_human_disagreement  mean_indeterminate_fraction  mean_reference_agreement  mean_human_ratings
              0       135                 0.015679                     0.036543                  0.984321            3.888889
              1        36                 0.023148                     0.071759                  0.976852            4.027778
              2        10                 0.042857                     0.066667                  0.957143            4.500000
              3         5                 0.100000                     0.133333                  0.900000            4.000000
              4         3                 0.000000                     0.222222                  1.000000            3.666667

=== IMAGENS EM QUE TODOS OS MODELOS FALHARAM ===
 image_number  true_label  n_models_wrong  n_human  human_disagreement  indeterminate_fraction  reference_agreement_fraction  Qwen_cor

## Robustness Analysis of the Association between Model Failures and Human Undetermination

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 23B
# ROBUSTEZ: FALHAS COMPARTILHADAS × INDETERMINAÇÃO HUMANA
# FPD-189 | Sem novas inferências
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

R = Path("/content/florence_visapp/results")

SOURCE = R / "florence_exp23_integrated.csv"

assert SOURCE.exists(), f"Arquivo ausente: {SOURCE}"

df = pd.read_csv(SOURCE)

assert len(df) == 189
assert df["image_number"].is_unique

required = [
    "image_number",
    "true_label",
    "n_human",
    "n_indeterminate",
    "indeterminate_fraction",
    "n_models_wrong",
    "Qwen_correct",
    "Llama_restricted_correct",
    "BLIP2_correct"
]

for col in required:
    assert col in df.columns, f"Coluna ausente: {col}"

# ------------------------------------------------------------
# 1. Verificação dos dados humanos
# ------------------------------------------------------------

df["n_human"] = pd.to_numeric(
    df["n_human"], errors="raise"
).astype(int)

df["n_indeterminate"] = pd.to_numeric(
    df["n_indeterminate"], errors="raise"
).astype(int)

assert (df["n_human"] > 0).all()
assert (df["n_indeterminate"] >= 0).all()
assert (
    df["n_indeterminate"] <= df["n_human"]
).all()

computed_fraction = (
    df["n_indeterminate"] / df["n_human"]
)

assert np.allclose(
    computed_fraction,
    df["indeterminate_fraction"],
    atol=1e-8
), "Inconsistência nas frações de indeterminação"

def to_bool(s):
    values = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }
    if not values.isin(mapping).all():
        raise ValueError("Valores booleanos inesperados")
    return values.map(mapping).astype(bool)

for name in [
    "Qwen",
    "Llama_restricted",
    "BLIP2"
]:
    df[f"{name}_correct"] = to_bool(
        df[f"{name}_correct"]
    )

# Três sistemas: um resultado por arquitetura
df["n_wrong_3models"] = (
    (~df["Qwen_correct"]).astype(int)
    + (~df["Llama_restricted_correct"]).astype(int)
    + (~df["BLIP2_correct"]).astype(int)
)

assert df["n_wrong_3models"].between(0, 3).all()
assert df["n_models_wrong"].between(0, 4).all()

# ------------------------------------------------------------
# 2. Teste de permutação estratificada
# ------------------------------------------------------------

N_PERM = 20000
rng = np.random.default_rng(2302)

def stratified_permutation_test(
    data,
    predictor,
    outcome="indeterminate_fraction"
):
    x = data[predictor].to_numpy(dtype=float)
    y = data[outcome].to_numpy(dtype=float)
    classes = data["true_label"].to_numpy()

    observed = spearmanr(x, y).statistic

    if not np.isfinite(observed):
        raise ValueError(
            f"Correlação indefinida: {predictor}"
        )

    groups = [
        np.flatnonzero(classes == c)
        for c in np.unique(classes)
    ]

    null = np.empty(N_PERM)

    for i in range(N_PERM):
        yp = y.copy()

        for idx in groups:
            yp[idx] = rng.permutation(y[idx])

        null[i] = spearmanr(x, yp).statistic

    center = null.mean()

    p = (
        1 + np.sum(
            np.abs(null - center)
            >= np.abs(observed - center)
        )
    ) / (N_PERM + 1)

    return {
        "n_images": len(data),
        "spearman_rho": observed,
        "null_mean": center,
        "permutation_p": p
    }

# ------------------------------------------------------------
# 3. Três análises de correlação
# ------------------------------------------------------------

scenarios = [
    (
        "A_original_4_conditions",
        df,
        "n_models_wrong"
    ),
    (
        "B_three_models",
        df,
        "n_wrong_3models"
    ),
    (
        "C_without_all_wrong",
        df.loc[df["n_models_wrong"] < 4].copy(),
        "n_models_wrong"
    )
]

results = []

for label, subset, predictor in scenarios:

    result = stratified_permutation_test(
        subset,
        predictor
    )

    result["scenario"] = label
    result["predictor"] = predictor

    results.append(result)

permutation_df = pd.DataFrame(results)

# Família de três testes de sensibilidade
permutation_df["holm_p"] = multipletests(
    permutation_df["permutation_p"],
    method="holm"
)[1]

permutation_df["significant_holm"] = (
    permutation_df["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 4. Modelo binomial para contagens humanas
# ------------------------------------------------------------

# A resposta é a contagem de votos indeterminados
# e não indeterminados por imagem.
#
# Preditores:
# - número de sistemas que falharam
# - classe real Pain/NoPain
#
# Covariância robusta HC3 por imagem.
#
# Modelo associativo exploratório; não causal.

glm_results = []

for name, predictor in [
    ("four_conditions", "n_models_wrong"),
    ("three_models", "n_wrong_3models")
]:

    X = pd.DataFrame({
        "intercept": np.ones(len(df)),
        "model_failures": df[predictor].astype(float),
        "pain_class": df["true_label"].astype(float)
    })

    Y = np.column_stack([
        df["n_indeterminate"].to_numpy(),
        (
            df["n_human"] - df["n_indeterminate"]
        ).to_numpy()
    ])

    fitted = sm.GLM(
        Y,
        X,
        family=sm.families.Binomial()
    ).fit(cov_type="HC3")

    beta = fitted.params["model_failures"]
    se = fitted.bse["model_failures"]

    ci_low = beta - 1.96 * se
    ci_high = beta + 1.96 * se

    glm_results.append({
        "model": name,
        "n_images": len(df),
        "coefficient": beta,
        "robust_se": se,
        "odds_ratio_per_failure": np.exp(beta),
        "or_ci95_low": np.exp(ci_low),
        "or_ci95_high": np.exp(ci_high),
        "p_value": fitted.pvalues["model_failures"],
        "pain_coefficient": fitted.params["pain_class"]
    })

glm_df = pd.DataFrame(glm_results)

# Correção separada para a família dos dois GLMs
glm_df["holm_p"] = multipletests(
    glm_df["p_value"],
    method="holm"
)[1]

glm_df["significant_holm"] = (
    glm_df["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 5. Resumo descritivo dos três modelos
# ------------------------------------------------------------

three_model_summary = (
    df.groupby("n_wrong_3models")
    .agg(
        n_images=("image_number", "size"),
        total_human_ratings=("n_human", "sum"),
        total_indeterminate=("n_indeterminate", "sum"),
        mean_indeterminate_fraction=(
            "indeterminate_fraction", "mean"
        )
    )
    .reset_index()
)

three_model_summary["pooled_indeterminate_fraction"] = (
    three_model_summary["total_indeterminate"]
    / three_model_summary["total_human_ratings"]
)

# ------------------------------------------------------------
# 6. Identificar imagens com três modelos errados
# ------------------------------------------------------------

three_all_wrong = df.loc[
    df["n_wrong_3models"] == 3,
    [
        "image_number",
        "true_label",
        "n_human",
        "n_indeterminate",
        "indeterminate_fraction",
        "n_models_wrong"
    ]
].sort_values("image_number")

# ------------------------------------------------------------
# 7. Salvar resultados
# ------------------------------------------------------------

outputs = {
    "florence_exp23b_permutation_robustness.csv":
        permutation_df,

    "florence_exp23b_binomial_glm.csv":
        glm_df,

    "florence_exp23b_three_model_summary.csv":
        three_model_summary,

    "florence_exp23b_three_all_wrong.csv":
        three_all_wrong,

    "florence_exp23b_integrated.csv":
        df
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 8. Exibir resultados
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("=" * 90)
print("FLORENCE — EXPERIMENTO 23B")
print("ANÁLISE DE ROBUSTEZ")
print("=" * 90)

print("\n=== PERMUTAÇÕES E CORREÇÃO DE HOLM ===")
print(
    permutation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== MODELOS BINOMIAIS COM ERROS ROBUSTOS ===")
print(
    glm_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== DISTRIBUIÇÃO: TRÊS MODELOS ===")
print(
    three_model_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== IMAGENS EM QUE OS TRÊS MODELOS FALHARAM ===")
print(
    three_all_wrong.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 23B CONCLUÍDO.")


FLORENCE — EXPERIMENTO 23B
ANÁLISE DE ROBUSTEZ

=== PERMUTAÇÕES E CORREÇÃO DE HOLM ===
 n_images  spearman_rho  null_mean  permutation_p                scenario       predictor   holm_p  significant_holm
      189      0.200898   0.031845       0.017999 A_original_4_conditions  n_models_wrong 0.053997             False
      189      0.164734   0.020007       0.047298          B_three_models n_wrong_3models 0.094595             False
      186      0.161033   0.033914       0.079346     C_without_all_wrong  n_models_wrong 0.094595             False

=== MODELOS BINOMIAIS COM ERROS ROBUSTOS ===
          model  n_images  coefficient  robust_se  odds_ratio_per_failure  or_ci95_low  or_ci95_high  p_value  pain_coefficient   holm_p  significant_holm
four_conditions       189     0.397975   0.161496                1.488806     1.084851      2.043179 0.013728         -0.459805 0.025152              True
   three_models       189     0.549558   0.220214                1.732488     1.125175   

# Candidate likelihood scoring

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 24B.3
# LLAMA 3.2 VISION 11B
# CANDIDATE LIKELIHOOD SCORING
# 189 imagens | Prompt C | 12 sequências candidatas
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import time
import json
import gc

R = Path("/content/florence_visapp/results")

PATHS = R / "florence_exp24b_resolved_images.csv"
RESTRICTED = R / "llama32_vision11b_restricted_predictions.csv"

OUTPUT = R / "florence_exp24b3_likelihood_predictions.csv"
CHECKPOINT = R / "florence_exp24b3_checkpoint.csv"

assert PATHS.exists()
assert RESTRICTED.exists()
assert "model" in globals()
assert "processor" in globals()

images_df = pd.read_csv(PATHS).sort_values(
    "image_number"
).reset_index(drop=True)

restricted_df = pd.read_csv(RESTRICTED)

assert len(images_df) == 189
assert images_df["image_number"].is_unique

tokenizer = processor.tokenizer

categories = [
    "pain",
    "happiness",
    "sadness",
    "anger",
    "disgust",
    "surprise"
]

prompt_c = (
    "Which category best describes this person's facial "
    "expression: pain, happiness, sadness, anger, disgust, "
    "or surprise? Answer with one category only."
)

eos_id = tokenizer.eos_token_id
assert eos_id == 128009

candidates = []

for category in categories:
    for leading_space in [False, True]:

        text = (
            " " + category
            if leading_space else category
        )

        ids = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        candidates.append({
            "category": category,
            "text": text,
            "token_ids": ids + [eos_id]
        })

assert len(candidates) == 12

# ------------------------------------------------------------
# 1. Preparação do contexto multimodal
# ------------------------------------------------------------

def prepare_context(image):

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_c}
        ]
    }]

    formatted = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        images=image,
        text=formatted,
        return_tensors="pt"
    )

    device = next(model.parameters()).device

    return {
        k: v.to(device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

# ------------------------------------------------------------
# 2. Pontuar uma sequência candidata
# ------------------------------------------------------------

@torch.inference_mode()
def score_candidate(context, candidate_ids):

    input_ids = context["input_ids"]

    candidate = torch.tensor(
        [candidate_ids],
        dtype=input_ids.dtype,
        device=input_ids.device
    )

    full_ids = torch.cat(
        [input_ids, candidate],
        dim=1
    )

    full_attention = torch.cat([
        context["attention_mask"],
        torch.ones_like(candidate)
    ], dim=1)

    kwargs = {
        k: v
        for k, v in context.items()
        if k not in {
            "input_ids",
            "attention_mask"
        }
    }

    kwargs["input_ids"] = full_ids
    kwargs["attention_mask"] = full_attention
    kwargs["use_cache"] = False

    # Para Mllama, a cross-attention mask deve
    # cobrir também os tokens da resposta.
    if "cross_attention_mask" in kwargs:

        cam = kwargs["cross_attention_mask"]

        extension = cam[:, -1:, ...].expand(
            -1,
            candidate.shape[1],
            *cam.shape[2:]
        )

        kwargs["cross_attention_mask"] = torch.cat(
            [cam, extension],
            dim=1
        )

    outputs = model(**kwargs)

    logits = outputs.logits

    start = input_ids.shape[1] - 1
    end = start + candidate.shape[1]

    selected_logits = logits[:, start:end, :]

    log_probs = F.log_softmax(
        selected_logits.float(),
        dim=-1
    )

    token_scores = log_probs.gather(
        -1,
        candidate.unsqueeze(-1)
    ).squeeze(-1)

    score = token_scores.sum().item()

    del outputs, logits, log_probs
    del selected_logits, token_scores

    return score

# ------------------------------------------------------------
# 3. Processamento
# ------------------------------------------------------------

if OUTPUT.exists():
    raise FileExistsError(
        f"Resultado final já existe: {OUTPUT}"
    )

if CHECKPOINT.exists():

    results = pd.read_csv(
        CHECKPOINT
    ).to_dict("records")

    processed = {
        int(r["image_number"])
        for r in results
    }

    print(
        "Retomando checkpoint:",
        len(processed),
        "imagens"
    )

else:
    results = []
    processed = set()

model.eval()

start_time = time.time()

for i, row in images_df.iterrows():

    image_number = int(row["image_number"])

    if image_number in processed:
        continue

    image_path = Path(row["resolved_path"])

    assert image_path.exists(), image_path

    with Image.open(image_path) as im:
        image = im.convert("RGB")

    context = prepare_context(image)

    scores = []

    for candidate in candidates:

        score = score_candidate(
            context,
            candidate["token_ids"]
        )

        scores.append({
            "category": candidate["category"],
            "text": candidate["text"],
            "score": score
        })

    best = max(
        scores,
        key=lambda x: x["score"]
    )

    prediction = int(
        best["category"] == "pain"
    )

    truth = int(row["true_label"])

    results.append({
        "image_number": image_number,
        "true_label": truth,
        "predicted_category": best["category"],
        "predicted_label": prediction,
        "correct": prediction == truth,
        "best_log_likelihood": best["score"],
        "candidate_scores": json.dumps(
            scores,
            ensure_ascii=False
        )
    })

    processed.add(image_number)

    if (
        len(results) % 10 == 0
        or len(results) == 189
    ):

        pd.DataFrame(results).to_csv(
            CHECKPOINT,
            index=False
        )

        print(
            f"{len(results)}/189 imagens | "
            f"tempo: {(time.time()-start_time)/60:.1f} min"
        )

    del context
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Salvar resultados finais
# ------------------------------------------------------------

result_df = pd.DataFrame(results).sort_values(
    "image_number"
).reset_index(drop=True)

assert len(result_df) == 189
assert result_df["image_number"].is_unique

result_df.to_csv(
    OUTPUT,
    index=False
)

# ------------------------------------------------------------
# 5. Comparação inicial
# ------------------------------------------------------------

restricted = restricted_df[
    [
        "image_number",
        "restricted_category",
        "restricted_prediction"
    ]
]

comparison = result_df.merge(
    restricted,
    on="image_number",
    validate="one_to_one"
)

comparison["same_category"] = (
    comparison["predicted_category"]
    == comparison["restricted_category"]
)

comparison["same_binary"] = (
    comparison["predicted_label"]
    == comparison["restricted_prediction"]
)

print("\n" + "=" * 80)
print("FLORENCE — EXPERIMENTO 24B.3")
print("=" * 80)

print("\n=== ACURÁCIA ===")
print(
    "Likelihood scoring:",
    round(result_df["correct"].mean() * 100, 4),
    "%"
)

print(
    "Llama restrito:",
    round(
        (
            restricted_df["restricted_correct"]
            .astype(bool)
            .mean()
        ) * 100,
        4
    ),
    "%"
)

print("\n=== CONCORDÂNCIA ENTRE MÉTODOS ===")
print(
    "Mesma categoria:",
    int(comparison["same_category"].sum()),
    "/189"
)

print(
    "Mesma classificação binária:",
    int(comparison["same_binary"].sum()),
    "/189"
)

print("\n=== DISTRIBUIÇÃO DE CATEGORIAS ===")
print(
    result_df["predicted_category"]
    .value_counts()
    .to_string()
)

print("\n=== ARQUIVO SALVO ===")
print(OUTPUT)

print("\nEXPERIMENTO 24B.3 CONCLUÍDO.")


10/189 imagens | tempo: 1.9 min
20/189 imagens | tempo: 3.7 min
30/189 imagens | tempo: 5.6 min
40/189 imagens | tempo: 7.5 min
50/189 imagens | tempo: 9.3 min
60/189 imagens | tempo: 11.2 min
70/189 imagens | tempo: 13.1 min
80/189 imagens | tempo: 14.9 min
90/189 imagens | tempo: 16.8 min
100/189 imagens | tempo: 18.6 min
110/189 imagens | tempo: 20.5 min
120/189 imagens | tempo: 22.4 min
130/189 imagens | tempo: 24.3 min
140/189 imagens | tempo: 26.1 min
150/189 imagens | tempo: 28.0 min
160/189 imagens | tempo: 29.9 min
170/189 imagens | tempo: 31.7 min
180/189 imagens | tempo: 33.6 min
189/189 imagens | tempo: 35.3 min

FLORENCE — EXPERIMENTO 24B.3

=== ACURÁCIA ===
Likelihood scoring: 95.7672 %
Llama restrito: 93.6508 %

=== CONCORDÂNCIA ENTRE MÉTODOS ===
Mesma categoria: 181 /189
Mesma classificação binária: 185 /189

=== DISTRIBUIÇÃO DE CATEGORIAS ===
predicted_category
pain         93
anger        35
surprise     28
happiness    17
sadness      15
disgust       1

=== ARQUIVO 

# Verification of changes and statistical tests.

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 24C
# LIKELIHOOD SCORING: AUDITORIA E TESTES PAREADOS
# ============================================================

from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

R = Path("/content/florence_visapp/results")

lik = pd.read_csv(
    R / "florence_exp24b3_likelihood_predictions.csv"
)
res = pd.read_csv(
    R / "llama32_vision11b_restricted_predictions.csv"
)
qwen = pd.read_csv(
    R / "qwen25_vl3b_predictions.csv"
)
original = pd.read_csv(
    R / "llama32_vision11b_predictions.csv"
)

qwen = qwen[qwen["protocol"] == "C_affective"].copy()
original = original[
    original["protocol"] == "C_affective"
].copy()

# ------------------------------------------------------------
# 1. Alinhamento
# ------------------------------------------------------------

df = lik[
    [
        "image_number", "true_label",
        "predicted_category", "predicted_label"
    ]
].rename(columns={
    "predicted_category": "lik_category",
    "predicted_label": "lik_prediction"
})

df = df.merge(
    res[
        [
            "image_number",
            "restricted_category",
            "restricted_prediction"
        ]
    ],
    on="image_number",
    validate="one_to_one"
)

df = df.merge(
    qwen[
        [
            "image_number",
            "true_label",
            "predicted_label",
            "valid"
        ]
    ].rename(columns={
        "true_label": "qwen_truth",
        "predicted_label": "qwen_prediction",
        "valid": "qwen_valid"
    }),
    on="image_number",
    validate="one_to_one"
)

df = df.merge(
    original[
        [
            "image_number",
            "true_label",
            "predicted_label",
            "valid"
        ]
    ].rename(columns={
        "true_label": "original_truth",
        "predicted_label": "original_prediction",
        "valid": "original_valid"
    }),
    on="image_number",
    validate="one_to_one"
)

assert len(df) == 189
assert df["image_number"].is_unique
assert (df["true_label"] == df["qwen_truth"]).all()
assert (df["true_label"] == df["original_truth"]).all()

def as_bool(series):
    mapped = series.astype(str).str.lower().map({
        "true": True, "false": False,
        "1": True, "0": False
    })
    assert mapped.notna().all()
    return mapped.astype(bool)

df["qwen_valid"] = as_bool(df["qwen_valid"])
df["original_valid"] = as_bool(df["original_valid"])

truth = df["true_label"].astype(int)

df["lik_correct"] = (
    df["lik_prediction"] == truth
)

df["restricted_correct"] = (
    df["restricted_prediction"] == truth
)

df["qwen_correct"] = (
    df["qwen_valid"]
    & (df["qwen_prediction"] == truth)
)

df["original_correct"] = (
    df["original_valid"]
    & (df["original_prediction"] == truth)
)

expected = {
    "lik_correct": 181,
    "restricted_correct": 177,
    "qwen_correct": 179,
    "original_correct": 157
}

for col, n in expected.items():
    assert int(df[col].sum()) == n, (
        f"Inconsistência em {col}"
    )

# ------------------------------------------------------------
# 2. Mudanças entre estratégias do Llama
# ------------------------------------------------------------

changes = df.loc[
    df["lik_category"] != df["restricted_category"]
].copy()

changes["binary_changed"] = (
    changes["lik_prediction"]
    != changes["restricted_prediction"]
)

changes["corrected_error"] = (
    ~changes["restricted_correct"]
    & changes["lik_correct"]
)

changes["introduced_error"] = (
    changes["restricted_correct"]
    & ~changes["lik_correct"]
)

assert len(changes) == 8
assert int(changes["binary_changed"].sum()) == 4
assert int(changes["corrected_error"].sum()) == 4
assert int(changes["introduced_error"].sum()) == 0

# ------------------------------------------------------------
# 3. McNemar exato + bootstrap pareado
# ------------------------------------------------------------

rng = np.random.default_rng(2403)
B = 20000

comparisons = [
    ("Llama_restricted", "restricted_correct"),
    ("Qwen", "qwen_correct"),
    ("Llama_original", "original_correct")
]

tests = []

a = df["lik_correct"].to_numpy(dtype=bool)
n = len(a)

for name, col in comparisons:

    b = df[col].to_numpy(dtype=bool)

    a_only = int(np.sum(a & ~b))
    b_only = int(np.sum(~a & b))
    discordant = a_only + b_only

    p = (
        binomtest(a_only, discordant, 0.5).pvalue
        if discordant else 1.0
    )

    differences = (
        a.astype(float) - b.astype(float)
    )

    bootstrap = np.empty(B)

    for i in range(B):
        idx = rng.integers(0, n, size=n)
        bootstrap[i] = differences[idx].mean() * 100

    ci = np.percentile(bootstrap, [2.5, 97.5])

    tests.append({
        "comparison": f"Likelihood vs {name}",
        "accuracy_likelihood": a.mean(),
        "accuracy_other": b.mean(),
        "difference_pp": differences.mean() * 100,
        "ci95_low_pp": ci[0],
        "ci95_high_pp": ci[1],
        "lik_only_correct": a_only,
        "other_only_correct": b_only,
        "discordant": discordant,
        "mcnemar_exact_p": p
    })

tests = pd.DataFrame(tests)

tests["holm_p"] = multipletests(
    tests["mcnemar_exact_p"],
    method="holm"
)[1]

tests["significant_holm_005"] = (
    tests["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 4. Salvar
# ------------------------------------------------------------

outputs = {
    "florence_exp24c_image_comparison.csv": df,
    "florence_exp24c_changed_categories.csv": changes,
    "florence_exp24c_paired_tests.csv": tests
}

for filename in outputs:
    assert not (R / filename).exists(), (
        f"Arquivo já existe: {filename}"
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 5. Resultados
# ------------------------------------------------------------

print("=" * 85)
print("FLORENCE — EXPERIMENTO 24C")
print("=" * 85)

print("\n=== COMPARAÇÕES ESTATÍSTICAS ===")
print(tests.to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}"
))

print("\n=== OITO MUDANÇAS DE CATEGORIA ===")
print(changes[
    [
        "image_number",
        "true_label",
        "restricted_category",
        "lik_category",
        "binary_changed",
        "corrected_error",
        "introduced_error"
    ]
].to_string(index=False))

print("\n=== ARQUIVOS SALVOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 24C CONCLUÍDO.")


FLORENCE — EXPERIMENTO 24C

=== COMPARAÇÕES ESTATÍSTICAS ===
                    comparison  accuracy_likelihood  accuracy_other  difference_pp  ci95_low_pp  ci95_high_pp  lik_only_correct  other_only_correct  discordant  mcnemar_exact_p   holm_p  significant_holm_005
Likelihood vs Llama_restricted             0.957672        0.936508       2.116402     0.529101      4.232804                 4                   0           4         0.125000 0.250000                 False
            Likelihood vs Qwen             0.957672        0.947090       1.058201    -2.116402      4.232804                 6                   4          10         0.753906 0.753906                 False
  Likelihood vs Llama_original             0.957672        0.830688      12.698413     7.936508     17.460317                24                   0          24         0.000000 0.000000                  True

=== OITO MUDANÇAS DE CATEGORIA ===
 image_number  true_label restricted_category lik_category  binary_chan

# Final Statistical Consolidation of the Florence Generative Models

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 25
# CONSOLIDAÇÃO FINAL DOS MODELOS GENERATIVOS
# FPD-189 | 5 condições | 10 comparações pareadas
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import binomtest
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)
from statsmodels.stats.multitest import multipletests

R = Path("/content/florence_visapp/results")

FILES = {
    "Likelihood": "florence_exp24b3_likelihood_predictions.csv",
    "Qwen": "qwen25_vl3b_predictions.csv",
    "Llama_restricted": "llama32_vision11b_restricted_predictions.csv",
    "BLIP2": "blip2_experiment_10BC_predictions.csv",
    "Llama_original": "llama32_vision11b_predictions.csv"
}

for name, filename in FILES.items():
    assert (R / filename).exists(), (
        f"Arquivo ausente: {filename}"
    )

# ------------------------------------------------------------
# 1. Carregamento
# ------------------------------------------------------------

lik = pd.read_csv(R / FILES["Likelihood"])
qwen = pd.read_csv(R / FILES["Qwen"])
restricted = pd.read_csv(R / FILES["Llama_restricted"])
blip = pd.read_csv(R / FILES["BLIP2"])
original = pd.read_csv(R / FILES["Llama_original"])

qwen = qwen[
    qwen["protocol"] == "C_affective"
].copy()

blip = blip[
    blip["protocol"] == "10C_affective"
].copy()

original = original[
    original["protocol"] == "C_affective"
].copy()

# ------------------------------------------------------------
# 2. Funções de validação
# ------------------------------------------------------------

def parse_bool(series):

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }

    values = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
    )

    assert values.notna().all(), (
        "Valor booleano inválido"
    )

    return values.astype(bool)


def prepare_condition(
    source,
    name,
    prediction_col,
    validity_col=None
):

    assert len(source) == 189
    assert source["image_number"].is_unique

    result = source[
        ["image_number", "true_label", prediction_col]
        + ([validity_col] if validity_col else [])
    ].copy()

    result = result.rename(columns={
        "true_label": f"{name}_truth",
        prediction_col: f"{name}_prediction"
    })

    if validity_col:

        result[f"{name}_valid"] = parse_bool(
            result[validity_col]
        )

        result = result.drop(
            columns=[validity_col]
        )

    else:

        result[f"{name}_valid"] = True

    valid = result[f"{name}_valid"]

    predictions = pd.to_numeric(
        result[f"{name}_prediction"],
        errors="coerce"
    )

    assert predictions[valid].isin([0, 1]).all(), (
        f"Predições inválidas em {name}"
    )

    result[f"{name}_prediction"] = predictions

    return result


conditions = {
    "Likelihood": prepare_condition(
        lik, "Likelihood",
        "predicted_label"
    ),
    "Qwen": prepare_condition(
        qwen, "Qwen",
        "predicted_label",
        "valid"
    ),
    "Llama_restricted": prepare_condition(
        restricted, "Llama_restricted",
        "restricted_prediction"
    ),
    "BLIP2": prepare_condition(
        blip, "BLIP2",
        "predicted_label",
        "valid_response"
    ),
    "Llama_original": prepare_condition(
        original, "Llama_original",
        "predicted_label",
        "valid"
    )
}

# ------------------------------------------------------------
# 3. Alinhamento das 189 imagens
# ------------------------------------------------------------

df = conditions["Likelihood"].copy()

for name, table in conditions.items():

    if name == "Likelihood":
        continue

    df = df.merge(
        table,
        on="image_number",
        validate="one_to_one"
    )

assert len(df) == 189
assert df["image_number"].is_unique

truth = df["Likelihood_truth"].astype(int)

assert truth.isin([0, 1]).all()
assert int(truth.sum()) == 99

for name in conditions:

    assert (
        df[f"{name}_truth"].astype(int) == truth
    ).all(), f"Referência divergente: {name}"

    valid = df[f"{name}_valid"]

    df[f"{name}_correct"] = (
        valid
        & (df[f"{name}_prediction"] == truth)
    )

expected = {
    "Likelihood": (189, 181),
    "Qwen": (189, 179),
    "Llama_restricted": (189, 177),
    "BLIP2": (180, 160),
    "Llama_original": (163, 157)
}

for name, (n_valid, n_correct) in expected.items():

    assert int(df[f"{name}_valid"].sum()) == n_valid
    assert int(df[f"{name}_correct"].sum()) == n_correct

print("Integridade: OK — 189 imagens alinhadas.")
print("Consistência histórica: OK.")

# ------------------------------------------------------------
# 4. Métricas por condição
# ------------------------------------------------------------

metric_rows = []

for name in conditions:

    valid = df[f"{name}_valid"]

    yt = truth[valid].to_numpy(dtype=int)

    yp = df.loc[
        valid, f"{name}_prediction"
    ].to_numpy(dtype=int)

    tn, fp, fn, tp = confusion_matrix(
        yt, yp, labels=[0, 1]
    ).ravel()

    metric_rows.append({
        "condition": name,
        "n_total": len(df),
        "n_valid": int(valid.sum()),
        "n_invalid": int((~valid).sum()),
        "n_correct": int(df[f"{name}_correct"].sum()),
        "accuracy_valid": float(np.mean(yt == yp)),
        "accuracy_operational": float(
            df[f"{name}_correct"].mean()
        ),
        "precision_valid": precision_score(
            yt, yp, zero_division=0
        ),
        "sensitivity_valid": recall_score(
            yt, yp, zero_division=0
        ),
        "specificity_valid": (
            tn / (tn + fp)
            if (tn + fp) else np.nan
        ),
        "f1_valid": f1_score(
            yt, yp, zero_division=0
        ),
        "balanced_accuracy_valid":
            balanced_accuracy_score(yt, yp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 5. Dez testes pareados
# ------------------------------------------------------------

N_BOOT = 20000
rng = np.random.default_rng(2501)

names = list(conditions.keys())

pair_rows = []

for a_name, b_name in combinations(names, 2):

    a = df[
        f"{a_name}_correct"
    ].to_numpy(dtype=bool)

    b = df[
        f"{b_name}_correct"
    ].to_numpy(dtype=bool)

    a_only = int(np.sum(a & ~b))
    b_only = int(np.sum(~a & b))

    discordant = a_only + b_only

    p = (
        binomtest(
            a_only, discordant, 0.5
        ).pvalue
        if discordant else 1.0
    )

    difference = (
        a.astype(float) - b.astype(float)
    )

    bootstrap = np.empty(N_BOOT)

    for j in range(N_BOOT):

        idx = rng.integers(
            0, len(df), size=len(df)
        )

        bootstrap[j] = (
            difference[idx].mean() * 100
        )

    ci = np.percentile(
        bootstrap, [2.5, 97.5]
    )

    pair_rows.append({
        "condition_a": a_name,
        "condition_b": b_name,
        "n": len(df),
        "accuracy_a": a.mean(),
        "accuracy_b": b.mean(),
        "difference_pp": difference.mean() * 100,
        "ci95_low_pp": ci[0],
        "ci95_high_pp": ci[1],
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "discordant": discordant,
        "mcnemar_exact_p": p
    })

pairs = pd.DataFrame(pair_rows)

assert len(pairs) == 10

pairs["holm_p"] = multipletests(
    pairs["mcnemar_exact_p"],
    method="holm"
)[1]

pairs["significant_holm_005"] = (
    pairs["holm_p"] < 0.05
)

pairs = pairs.sort_values(
    "mcnemar_exact_p"
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Concordância entre os cinco sistemas
# ------------------------------------------------------------

correct_cols = [
    f"{name}_correct"
    for name in names
]

df["n_systems_correct"] = (
    df[correct_cols].sum(axis=1)
)

df["n_systems_wrong"] = (
    5 - df["n_systems_correct"]
)

agreement = (
    df.groupby("n_systems_correct")
    .size()
    .reindex(range(6), fill_value=0)
    .reset_index(name="n_images")
)

# ------------------------------------------------------------
# 7. Imagens difíceis
# ------------------------------------------------------------

difficult = df.loc[
    df["n_systems_wrong"] >= 3
].copy()

difficult = difficult.sort_values(
    ["n_systems_wrong", "image_number"],
    ascending=[False, True]
)

# ------------------------------------------------------------
# 8. Salvar arquivos
# ------------------------------------------------------------

outputs = {
    "florence_exp25_metrics.csv": metrics,
    "florence_exp25_pairwise_mcnemar_holm.csv": pairs,
    "florence_exp25_image_level.csv": df,
    "florence_exp25_agreement.csv": agreement,
    "florence_exp25_difficult_images.csv": difficult
}

existing = [
    filename
    for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(
        R / filename,
        index=False
    )

# ------------------------------------------------------------
# 9. Resultados
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("\n" + "=" * 95)
print("FLORENCE — EXPERIMENTO 25")
print("CONSOLIDAÇÃO ESTATÍSTICA FINAL")
print("=" * 95)

print("\n=== MÉTRICAS ===")
print(
    metrics.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== DEZ COMPARAÇÕES PAREADAS ===")
print(
    pairs.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n=== CONCORDÂNCIA ENTRE CINCO SISTEMAS ===")
print(agreement.to_string(index=False))

print("\n=== IMAGENS COM TRÊS OU MAIS FALHAS ===")
print(
    difficult[
        [
            "image_number",
            "Likelihood_truth",
            "n_systems_wrong",
            "Likelihood_correct",
            "Qwen_correct",
            "Llama_restricted_correct",
            "BLIP2_correct",
            "Llama_original_correct"
        ]
    ].to_string(index=False))

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 25 CONCLUÍDO.")


Integridade: OK — 189 imagens alinhadas.
Consistência histórica: OK.

FLORENCE — EXPERIMENTO 25
CONSOLIDAÇÃO ESTATÍSTICA FINAL

=== MÉTRICAS ===
       condition  n_total  n_valid  n_invalid  n_correct  accuracy_valid  accuracy_operational  precision_valid  sensitivity_valid  specificity_valid  f1_valid  balanced_accuracy_valid  TN  FP  FN  TP
      Likelihood      189      189          0        181        0.957672              0.957672         0.989247           0.929293           0.988889  0.958333                 0.959091  89   1   7  92
            Qwen      189      189          0        179        0.947090              0.947090         0.968421           0.929293           0.966667  0.948454                 0.947980  87   3   7  92
Llama_restricted      189      189          0        177        0.936508              0.936508         0.967742           0.909091           0.966667  0.937500                 0.937879  87   3   9  90
           BLIP2      189      180          9      

# Integration of contrastive and generative predictions

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 26B.2
# INTEGRAÇÃO CONTRASTIVOS × GENERATIVOS
#
# 6 arquiteturas: 15 comparações
# 8 condições: 28 comparações
# McNemar exato + Holm + bootstrap pareado
# ============================================================

from pathlib import Path
from itertools import combinations
import re

import numpy as np
import pandas as pd

from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

R = Path("/content/florence_visapp/results")

contrastive_files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv"
}

source25 = R / "florence_exp25_image_level.csv"
assert source25.exists()

# ------------------------------------------------------------
# 1. Base generativa do Experimento 25
# ------------------------------------------------------------

base = pd.read_csv(source25)

assert len(base) == 189
assert base["image_number"].is_unique

base = base.sort_values(
    "image_number"
).reset_index(drop=True)

truth = base["Likelihood_truth"].astype(int)

assert truth.isin([0, 1]).all()
assert int(truth.sum()) == 99

names_generatives = [
    "Likelihood",
    "Qwen",
    "Llama_restricted",
    "BLIP2",
    "Llama_original"
]

def parse_bool(series):

    result = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

    assert result.notna().all()

    return result.astype(bool)

for name in names_generatives:

    base[f"{name}_valid"] = parse_bool(
        base[f"{name}_valid"]
    )

    base[f"{name}_correct"] = parse_bool(
        base[f"{name}_correct"]
    )

    pred = pd.to_numeric(
        base[f"{name}_prediction"],
        errors="coerce"
    )

    valid = base[f"{name}_valid"]

    assert pred[valid].isin([0, 1]).all()

    calculated = (
        valid & (pred == truth)
    )

    assert (
        calculated.to_numpy()
        == base[f"{name}_correct"].to_numpy()
    ).all(), f"Inconsistência em {name}"

# ------------------------------------------------------------
# 2. Incorporar os três modelos contrastivos
# ------------------------------------------------------------

integrated = base.copy()

for name, filename in contrastive_files.items():

    source = pd.read_csv(R / filename)

    source = source[
        source["strategy"] == "affective"
    ].copy()

    assert len(source) == 189

    extracted = source["image"].astype(str).str.extract(
        r"(?:image|imagem)_(\d+)\.[^.\\/]+$",
        flags=re.IGNORECASE
    )[0]

    assert extracted.notna().all()

    source["image_number"] = extracted.astype(int)

    assert source["image_number"].is_unique

    source = source[
        ["image_number", "true_label", "predicted_label"]
    ].rename(columns={
        "true_label": f"{name}_truth",
        "predicted_label": f"{name}_prediction"
    })

    integrated = integrated.merge(
        source,
        on="image_number",
        validate="one_to_one",
        how="inner"
    )

    assert len(integrated) == 189

    assert (
        integrated[f"{name}_truth"].astype(int)
        == integrated["Likelihood_truth"].astype(int)
    ).all()

    integrated[f"{name}_valid"] = True

    integrated[f"{name}_correct"] = (
        integrated[f"{name}_prediction"].astype(int)
        == integrated["Likelihood_truth"].astype(int)
    )

# ------------------------------------------------------------
# 3. Condições e verificações
# ------------------------------------------------------------

main_names = [
    "CLIP",
    "SigLIP",
    "SigLIP2",
    "BLIP2",
    "Qwen",
    "Likelihood"
]

all_names = [
    "CLIP",
    "SigLIP",
    "SigLIP2",
    "BLIP2",
    "Qwen",
    "Likelihood",
    "Llama_restricted",
    "Llama_original"
]

expected_correct = {
    "CLIP": 167,
    "SigLIP": 177,
    "SigLIP2": 176,
    "BLIP2": 160,
    "Qwen": 179,
    "Likelihood": 181,
    "Llama_restricted": 177,
    "Llama_original": 157
}

for name, expected in expected_correct.items():

    observed = int(
        integrated[f"{name}_correct"].sum()
    )

    assert observed == expected, (
        f"{name}: esperado {expected}, "
        f"observado {observed}"
    )

print("INTEGRIDADE: OK — oito condições alinhadas.")

# ------------------------------------------------------------
# 4. Métricas
# ------------------------------------------------------------

metric_rows = []

for name in all_names:

    valid = integrated[f"{name}_valid"]

    yt = truth[valid].to_numpy(dtype=int)

    yp = integrated.loc[
        valid, f"{name}_prediction"
    ].to_numpy(dtype=int)

    tn, fp, fn, tp = confusion_matrix(
        yt, yp, labels=[0, 1]
    ).ravel()

    metric_rows.append({
        "condition": name,
        "n_total": 189,
        "n_valid": int(valid.sum()),
        "n_invalid": int((~valid).sum()),
        "n_correct": int(
            integrated[f"{name}_correct"].sum()
        ),
        "accuracy_operational": float(
            integrated[f"{name}_correct"].mean()
        ),
        "accuracy_valid": float(
            np.mean(yt == yp)
        ),
        "precision_valid": precision_score(
            yt, yp, zero_division=0
        ),
        "sensitivity_valid": recall_score(
            yt, yp, zero_division=0
        ),
        "specificity_valid": (
            tn / (tn + fp)
            if tn + fp else np.nan
        ),
        "f1_valid": f1_score(
            yt, yp, zero_division=0
        ),
        "balanced_accuracy_valid":
            balanced_accuracy_score(yt, yp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 5. Testes pareados
# ------------------------------------------------------------

def paired_tests(names, seed, n_boot=20000):

    rng = np.random.default_rng(seed)

    rows = []

    for a_name, b_name in combinations(names, 2):

        a = integrated[
            f"{a_name}_correct"
        ].to_numpy(dtype=bool)

        b = integrated[
            f"{b_name}_correct"
        ].to_numpy(dtype=bool)

        a_only = int(np.sum(a & ~b))
        b_only = int(np.sum(~a & b))

        discordant = a_only + b_only

        p = (
            binomtest(
                a_only,
                discordant,
                0.5
            ).pvalue
            if discordant else 1.0
        )

        difference = (
            a.astype(float)
            - b.astype(float)
        )

        # Bootstrap pareado: mesmas imagens
        # reamostradas para as duas condições.
        bootstrap = np.empty(n_boot)

        for i in range(n_boot):

            idx = rng.integers(
                0, len(difference),
                size=len(difference)
            )

            bootstrap[i] = (
                difference[idx].mean() * 100
            )

        low, high = np.percentile(
            bootstrap, [2.5, 97.5]
        )

        rows.append({
            "condition_a": a_name,
            "condition_b": b_name,
            "n": 189,
            "accuracy_a": a.mean(),
            "accuracy_b": b.mean(),
            "difference_pp": difference.mean() * 100,
            "ci95_low_pp": low,
            "ci95_high_pp": high,
            "a_only_correct": a_only,
            "b_only_correct": b_only,
            "discordant": discordant,
            "mcnemar_exact_p": p
        })

    result = pd.DataFrame(rows)

    result["holm_p"] = multipletests(
        result["mcnemar_exact_p"],
        method="holm"
    )[1]

    result["significant_holm_005"] = (
        result["holm_p"] < 0.05
    )

    return result.sort_values(
        "mcnemar_exact_p"
    ).reset_index(drop=True)

main_tests = paired_tests(
    main_names, seed=2601
)

all_tests = paired_tests(
    all_names, seed=2602
)

assert len(main_tests) == 15
assert len(all_tests) == 28

# ------------------------------------------------------------
# 6. Concordância entre condições
# ------------------------------------------------------------

integrated["n_correct_6"] = integrated[
    [f"{n}_correct" for n in main_names]
].sum(axis=1)

integrated["n_wrong_6"] = (
    6 - integrated["n_correct_6"]
)

integrated["n_correct_8"] = integrated[
    [f"{n}_correct" for n in all_names]
].sum(axis=1)

integrated["n_wrong_8"] = (
    8 - integrated["n_correct_8"]
)

agreement = pd.DataFrame({
    "n_correct_6": range(7)
})

counts6 = (
    integrated["n_correct_6"]
    .value_counts()
    .reindex(range(7), fill_value=0)
)

agreement["n_images_6"] = counts6.to_numpy()

# ------------------------------------------------------------
# 7. Arquivos de saída
# ------------------------------------------------------------

outputs = {
    "florence_exp26b2_integrated.csv": integrated,
    "florence_exp26b2_metrics.csv": metrics,
    "florence_exp26b2_main_15_mcnemar_holm.csv": main_tests,
    "florence_exp26b2_all_28_mcnemar_holm.csv": all_tests,
    "florence_exp26b2_agreement.csv": agreement
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(
        R / filename,
        index=False
    )

# ------------------------------------------------------------
# 8. Exibição
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FLORENCE — EXPERIMENTO 26B.2")
print("=" * 90)

print("\n=== MÉTRICAS ===")
print(metrics.to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}"
))

print("\n=== COMPARAÇÃO PRINCIPAL: 15 TESTES ===")
print(main_tests.to_string(
    index=False,
    float_format=lambda x: f"{x:.8f}"
))

print("\n=== ANÁLISE COMPLEMENTAR: 28 TESTES ===")
print(all_tests.to_string(
    index=False,
    float_format=lambda x: f"{x:.8f}"
))

print("\n=== CONCORDÂNCIA ENTRE SEIS ARQUITETURAS ===")
print(agreement.to_string(index=False))

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 26B.2 CONCLUÍDO.")


INTEGRIDADE: OK — oito condições alinhadas.

FLORENCE — EXPERIMENTO 26B.2

=== MÉTRICAS ===
       condition  n_total  n_valid  n_invalid  n_correct  accuracy_operational  accuracy_valid  precision_valid  sensitivity_valid  specificity_valid  f1_valid  balanced_accuracy_valid  TN  FP  FN  TP
            CLIP      189      189          0        167              0.883598        0.883598         0.846847           0.949495           0.811111  0.895238                 0.880303  73  17   5  94
          SigLIP      189      189          0        177              0.936508        0.936508         0.988764           0.888889           0.988889  0.936170                 0.938889  89   1  11  88
         SigLIP2      189      189          0        176              0.931217        0.931217         0.947917           0.919192           0.944444  0.933333                 0.931818  85   5   8  91
           BLIP2      189      180          9        160              0.846561        0.888889         0

# Analysis of complementarity between architectures

## Shared and exclusive errors

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 26C.1
# COMPLEMENTARIDADE ENTRE ARQUITETURAS
#
# Contraste: CLIP, SigLIP, SigLIP2
# Geração: BLIP2, Qwen, Llama likelihood
#
# Oracle = limite superior retrospectivo, NÃO um
# classificador implementável sem acesso aos rótulos.
#
# Não utiliza GPU.
# Não sobrescreve arquivos.
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

R = Path("/content/florence_visapp/results")

SOURCE = R / "florence_exp26b2_integrated.csv"

assert SOURCE.exists(), f"Arquivo não encontrado: {SOURCE}"

df = pd.read_csv(SOURCE)

contrastive = ["CLIP", "SigLIP", "SigLIP2"]
generative = ["BLIP2", "Qwen", "Likelihood"]
models = contrastive + generative

assert len(df) == 189
assert df["image_number"].is_unique

# ------------------------------------------------------------
# 1. Normalização e integridade
# ------------------------------------------------------------

def as_bool(s):
    converted = (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )
    assert converted.notna().all()
    return converted.astype(bool)

for model in models:
    df[f"{model}_correct"] = as_bool(
        df[f"{model}_correct"]
    )

    df[f"{model}_valid"] = as_bool(
        df[f"{model}_valid"]
    )

    assert not (
        df[f"{model}_correct"]
        & ~df[f"{model}_valid"]
    ).any()

assert (
    df["Likelihood_truth"].astype(int).isin([0, 1])
).all()

assert (
    df["Likelihood_truth"].astype(int).sum() == 99
)

df = df.sort_values(
    "image_number"
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Análise individual
# ------------------------------------------------------------

out = df[[
    "image_number",
    "Likelihood_truth"
]].copy()

out = out.rename(columns={
    "Likelihood_truth": "true_label"
})

for model in models:
    out[f"{model}_correct"] = df[
        f"{model}_correct"
    ].astype(int)

out["contrastive_correct_count"] = out[
    [f"{m}_correct" for m in contrastive]
].sum(axis=1)

out["generative_correct_count"] = out[
    [f"{m}_correct" for m in generative]
].sum(axis=1)

out["total_correct"] = (
    out["contrastive_correct_count"]
    + out["generative_correct_count"]
)

out["total_wrong"] = 6 - out["total_correct"]

out["any_contrastive_correct"] = (
    out["contrastive_correct_count"] > 0
)

out["any_generative_correct"] = (
    out["generative_correct_count"] > 0
)

out["all_six_wrong"] = (
    out["total_correct"] == 0
)

out["all_six_correct"] = (
    out["total_correct"] == 6
)

out["only_contrastive_family_correct"] = (
    out["any_contrastive_correct"]
    & ~out["any_generative_correct"]
)

out["only_generative_family_correct"] = (
    ~out["any_contrastive_correct"]
    & out["any_generative_correct"]
)

out["both_families_have_correct_model"] = (
    out["any_contrastive_correct"]
    & out["any_generative_correct"]
)

def correct_models(row):
    return ";".join(
        m for m in models
        if row[f"{m}_correct"] == 1
    )

def wrong_models(row):
    return ";".join(
        m for m in models
        if row[f"{m}_correct"] == 0
    )

out["correct_models"] = out.apply(
    correct_models, axis=1
)

out["wrong_models"] = out.apply(
    wrong_models, axis=1
)

# ------------------------------------------------------------
# 3. Complementaridade entre famílias
# ------------------------------------------------------------

N = len(out)

categories = [
    (
        "Both families have at least one correct model",
        out["both_families_have_correct_model"]
    ),
    (
        "Only contrastive family has a correct model",
        out["only_contrastive_family_correct"]
    ),
    (
        "Only generative family has a correct model",
        out["only_generative_family_correct"]
    ),
    (
        "All six models wrong",
        out["all_six_wrong"]
    )
]

group_rows = []

for label, mask in categories:
    group_rows.append({
        "category": label,
        "n_images": int(mask.sum()),
        "percentage": float(mask.mean() * 100),
        "image_numbers": ";".join(
            out.loc[mask, "image_number"]
            .astype(str)
        )
    })

group_table = pd.DataFrame(group_rows)

assert group_table["n_images"].sum() == N

# ------------------------------------------------------------
# 4. Oracle de pares
# ------------------------------------------------------------

pair_rows = []

for a, b in combinations(models, 2):

    ca = out[f"{a}_correct"].astype(bool)
    cb = out[f"{b}_correct"].astype(bool)

    either = ca | cb

    pair_rows.append({
        "model_a": a,
        "model_b": b,
        "family_a": (
            "contrastive"
            if a in contrastive else "generative"
        ),
        "family_b": (
            "contrastive"
            if b in contrastive else "generative"
        ),
        "a_correct": int(ca.sum()),
        "b_correct": int(cb.sum()),
        "both_correct": int((ca & cb).sum()),
        "a_only_correct": int((ca & ~cb).sum()),
        "b_only_correct": int((~ca & cb).sum()),
        "both_wrong": int((~ca & ~cb).sum()),
        "oracle_correct": int(either.sum()),
        "oracle_accuracy": float(either.mean()),
        "oracle_accuracy_pct": float(
            either.mean() * 100
        ),
        "gain_over_best_single_pp": float(
            (
                either.mean()
                - max(ca.mean(), cb.mean())
            ) * 100
        )
    })

pairs = pd.DataFrame(pair_rows)

pairs = pairs.sort_values(
    [
        "oracle_accuracy",
        "gain_over_best_single_pp"
    ],
    ascending=False
).reset_index(drop=True)

assert len(pairs) == 15

# ------------------------------------------------------------
# 5. Imagens difíceis
# ------------------------------------------------------------

difficult = out[
    out["total_wrong"] >= 3
].copy()

difficult = difficult.sort_values(
    ["total_wrong", "image_number"],
    ascending=[False, True]
)

# ------------------------------------------------------------
# 6. Validações contra Experimento 26B.2
# ------------------------------------------------------------

distribution = (
    out["total_correct"]
    .value_counts()
    .reindex(range(7), fill_value=0)
)

expected_distribution = {
    0: 1,
    1: 1,
    2: 1,
    3: 10,
    4: 12,
    5: 25,
    6: 139
}

assert distribution.to_dict() == expected_distribution

# ------------------------------------------------------------
# 7. Salvar sem sobrescrever
# ------------------------------------------------------------

outputs = {
    "florence_exp26c1_image_analysis.csv": out,
    "florence_exp26c1_group_complementarity.csv": group_table,
    "florence_exp26c1_pairwise_oracle.csv": pairs,
    "florence_exp26c1_difficult_images.csv": difficult
}

existing = [
    filename
    for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Os seguintes arquivos já existem: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(
        R / filename,
        index=False
    )

# ------------------------------------------------------------
# 8. Resultados
# ------------------------------------------------------------

print("=" * 80)
print("FLORENCE — EXPERIMENTO 26C.1")
print("COMPLEMENTARIDADE ENTRE ARQUITETURAS")
print("=" * 80)

print("\n=== COMPLEMENTARIDADE ENTRE FAMÍLIAS ===")
print(group_table[
    ["category", "n_images", "percentage"]
].to_string(
    index=False,
    float_format=lambda x: f"{x:.2f}"
))

print("\n=== IMAGENS ACERTADAS POR APENAS UMA FAMÍLIA ===")

for label, mask in [
    (
        "Apenas contrastivos",
        out["only_contrastive_family_correct"]
    ),
    (
        "Apenas generativos",
        out["only_generative_family_correct"]
    ),
    (
        "Todos erraram",
        out["all_six_wrong"]
    )
]:
    subset = out.loc[
        mask,
        [
            "image_number",
            "true_label",
            "correct_models",
            "wrong_models"
        ]
    ]

    print(f"\n{label}: {len(subset)} imagens")
    print(subset.to_string(index=False))

print("\n=== MELHORES COMBINAÇÕES ORACLE ===")
print(
    pairs[
        [
            "model_a",
            "model_b",
            "oracle_correct",
            "oracle_accuracy_pct",
            "gain_over_best_single_pp",
            "both_wrong"
        ]
    ].head(15).to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\n=== IMAGENS DIFÍCEIS (3 OU MAIS ERROS) ===")
print(
    difficult[
        [
            "image_number",
            "true_label",
            "total_wrong",
            "correct_models"
        ]
    ].to_string(index=False)
)

print("\n=== DISTRIBUIÇÃO DE ACERTOS ===")
print(distribution.to_string())

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 26C.1 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 26C.1
COMPLEMENTARIDADE ENTRE ARQUITETURAS

=== COMPLEMENTARIDADE ENTRE FAMÍLIAS ===
                                     category  n_images  percentage
Both families have at least one correct model       187       98.94
  Only contrastive family has a correct model         1        0.53
   Only generative family has a correct model         0        0.00
                         All six models wrong         1        0.53

=== IMAGENS ACERTADAS POR APENAS UMA FAMÍLIA ===

Apenas contrastivos: 1 imagens
 image_number  true_label correct_models                      wrong_models
           59           1        SigLIP2 CLIP;SigLIP;BLIP2;Qwen;Likelihood

Apenas generativos: 0 imagens
Empty DataFrame
Columns: [image_number, true_label, correct_models, wrong_models]
Index: []

Todos erraram: 1 imagens
 image_number  true_label correct_models                              wrong_models
          175           1                CLIP;SigLIP;SigLIP2;BLIP2;Qwen;Likelihood

=== 

## Majority vote

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 26C.2
# ENSEMBLES POR VOTAÇÃO MAJORITÁRIA
# Sem treinamento, ajuste de pesos ou uso do rótulo na decisão
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import confusion_matrix

R = Path("/content/florence_visapp/results")

df = pd.read_csv(
    R / "florence_exp26b2_integrated.csv"
).sort_values("image_number").reset_index(drop=True)

assert len(df) == 189
assert df["image_number"].is_unique

y = df["Likelihood_truth"].astype(int).to_numpy()
assert set(np.unique(y)) == {0, 1}

ensembles = {
    "SigLIP2_Qwen_Likelihood":
        ["SigLIP2", "Qwen", "Likelihood"],
    "CLIP_SigLIP_Qwen":
        ["CLIP", "SigLIP", "Qwen"],
    "SigLIP_SigLIP2_Qwen":
        ["SigLIP", "SigLIP2", "Qwen"]
}

predictions = pd.DataFrame({
    "image_number": df["image_number"],
    "true_label": y
})

for model in [
    "CLIP", "SigLIP", "SigLIP2",
    "Qwen", "Likelihood"
]:
    p = pd.to_numeric(
        df[f"{model}_prediction"],
        errors="raise"
    ).to_numpy()

    assert np.isin(p, [0, 1]).all()

    predictions[model] = p.astype(int)

for name, members in ensembles.items():
    votes = predictions[members].sum(axis=1)

    # Três modelos: dois ou mais votos indicam dor.
    predictions[name] = (votes >= 2).astype(int)

def metrics(name):
    p = predictions[name].to_numpy(dtype=int)
    correct = p == y

    tn, fp, fn, tp = confusion_matrix(
        y, p, labels=[0, 1]
    ).ravel()

    return {
        "condition": name,
        "n_correct": int(correct.sum()),
        "accuracy": float(correct.mean()),
        "sensitivity": float(tp / (tp + fn)),
        "specificity": float(tn / (tn + fp)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

names = [
    "Qwen",
    "Likelihood",
    "SigLIP",
    "SigLIP2",
    "CLIP"
] + list(ensembles)

results = pd.DataFrame([
    metrics(name) for name in names
]).sort_values("accuracy", ascending=False)

# ------------------------------------------------------------
# Testes pareados: três ensembles vs Qwen
# ------------------------------------------------------------

baseline = (
    predictions["Qwen"].to_numpy() == y
)

tests = []

for name in ensembles:
    candidate = (
        predictions[name].to_numpy() == y
    )

    ensemble_only = int(
        np.sum(candidate & ~baseline)
    )

    qwen_only = int(
        np.sum(~candidate & baseline)
    )

    n_discordant = ensemble_only + qwen_only

    p_value = (
        binomtest(
            ensemble_only,
            n_discordant,
            0.5
        ).pvalue
        if n_discordant else 1.0
    )

    tests.append({
        "ensemble": name,
        "ensemble_only_correct": ensemble_only,
        "qwen_only_correct": qwen_only,
        "discordant": n_discordant,
        "difference_pp": float(
            (candidate.mean() - baseline.mean()) * 100
        ),
        "mcnemar_exact_p": p_value
    })

tests = pd.DataFrame(tests)

tests["holm_p"] = multipletests(
    tests["mcnemar_exact_p"],
    method="holm"
)[1]

tests["significant_holm_005"] = (
    tests["holm_p"] < 0.05
)

# ------------------------------------------------------------
# Erros por imagem
# ------------------------------------------------------------

for name in ensembles:
    predictions[f"{name}_correct"] = (
        predictions[name].to_numpy() == y
    )

# ------------------------------------------------------------
# Salvar sem sobrescrever
# ------------------------------------------------------------

outputs = {
    "florence_exp26c2_ensemble_predictions.csv":
        predictions,
    "florence_exp26c2_ensemble_metrics.csv":
        results,
    "florence_exp26c2_ensemble_mcnemar.csv":
        tests
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

print("=" * 78)
print("FLORENCE — EXPERIMENTO 26C.2")
print("VOTAÇÃO MAJORITÁRIA SEM AJUSTE DE PARÂMETROS")
print("=" * 78)

print("\n=== DESEMPENHO ===")
print(results.to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}"
))

print("\n=== ENSEMBLES VS QWEN ===")
print(tests.to_string(
    index=False,
    float_format=lambda x: f"{x:.8f}"
))

print("\n=== IMAGENS ERRADAS POR CADA ENSEMBLE ===")
for name in ensembles:
    errors = predictions.loc[
        ~predictions[f"{name}_correct"],
        "image_number"
    ].tolist()

    print(f"{name}: {errors}")

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 26C.2 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 26C.2
VOTAÇÃO MAJORITÁRIA SEM AJUSTE DE PARÂMETROS

=== DESEMPENHO ===
              condition  n_correct  accuracy  sensitivity  specificity  TN  FP  FN  TP
       CLIP_SigLIP_Qwen        183  0.968254     0.959596     0.977778  88   2   4  95
SigLIP2_Qwen_Likelihood        183  0.968254     0.939394     1.000000  90   0   6  93
             Likelihood        181  0.957672     0.929293     0.988889  89   1   7  92
    SigLIP_SigLIP2_Qwen        180  0.952381     0.909091     1.000000  90   0   9  90
                   Qwen        179  0.947090     0.929293     0.966667  87   3   7  92
                 SigLIP        177  0.936508     0.888889     0.988889  89   1  11  88
                SigLIP2        176  0.931217     0.919192     0.944444  85   5   8  91
                   CLIP        167  0.883598     0.949495     0.811111  73  17   5  94

=== ENSEMBLES VS QWEN ===
               ensemble  ensemble_only_correct  qwen_only_correct  discordant  difference_pp  mc

# Integrating results with human assessments

## Relationship between ensemble errors and human evaluations

### Integration and characterization of errors

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 27A.1
# ENSEMBLES × AVALIAÇÕES HUMANAS
#
# Integração, auditoria e caracterização descritiva.
# Sem GPU. Sem novos modelos. Sem testes inferenciais.
# Não sobrescreve arquivos existentes.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

R = Path("/content/florence_visapp/results")

HUMAN_FILE = R / "human_model_error_association.csv"
ENSEMBLE_FILE = (
    R / "florence_exp26c2_ensemble_predictions.csv"
)

assert HUMAN_FILE.exists(), (
    f"Arquivo humano não encontrado: {HUMAN_FILE}"
)
assert ENSEMBLE_FILE.exists(), (
    f"Arquivo dos ensembles não encontrado: {ENSEMBLE_FILE}"
)

human = pd.read_csv(HUMAN_FILE)
ensemble = pd.read_csv(ENSEMBLE_FILE)

print("=" * 78)
print("FLORENCE — EXPERIMENTO 27A.1")
print("INTEGRAÇÃO DOS ENSEMBLES COM AS AVALIAÇÕES HUMANAS")
print("=" * 78)

print("\nArquivos carregados:")
print("Humanos:", len(human), "registros")
print("Ensembles:", len(ensemble), "registros")

# ------------------------------------------------------------
# 1. Validação dos identificadores
# ------------------------------------------------------------

assert "image_number" in human.columns
assert "image_number" in ensemble.columns

assert human["image_number"].notna().all()
assert ensemble["image_number"].notna().all()

human["image_number"] = (
    human["image_number"].astype(int)
)
ensemble["image_number"] = (
    ensemble["image_number"].astype(int)
)

assert human["image_number"].is_unique
assert ensemble["image_number"].is_unique

assert len(human) == 189
assert len(ensemble) == 189

assert (
    set(human["image_number"])
    == set(ensemble["image_number"])
)

# ------------------------------------------------------------
# 2. Identificar rótulos humanos
# ------------------------------------------------------------

required_human = [
    "image_number",
    "true_label",
    "n_human",
    "n_pain",
    "n_nopain",
    "n_indeterminate",
    "reference_agreement_fraction",
    "human_disagreement",
    "indeterminate_fraction"
]

missing = [
    c for c in required_human
    if c not in human.columns
]

assert not missing, (
    f"Colunas humanas ausentes: {missing}"
)

# ------------------------------------------------------------
# 3. Integração
# ------------------------------------------------------------

integrated = ensemble.merge(
    human,
    on="image_number",
    how="inner",
    suffixes=("_model", "_human"),
    validate="one_to_one"
)

assert len(integrated) == 189

assert (
    integrated["true_label_model"].astype(int)
    == integrated["true_label_human"].astype(int)
).all(), "Divergência nos rótulos de referência"

integrated["true_label"] = (
    integrated["true_label_model"].astype(int)
)

# ------------------------------------------------------------
# 4. Verificar os ensembles
# ------------------------------------------------------------

best_ensembles = [
    "CLIP_SigLIP_Qwen",
    "SigLIP2_Qwen_Likelihood"
]

for name in best_ensembles:

    assert name in integrated.columns

    integrated[f"{name}_correct"] = (
        integrated[name].astype(int)
        == integrated["true_label"]
    )

    assert (
        integrated[f"{name}_correct"].sum() == 183
    ), f"Quantidade de acertos inesperada: {name}"

integrated["both_ensembles_wrong"] = (
    ~integrated[
        "CLIP_SigLIP_Qwen_correct"
    ]
    &
    ~integrated[
        "SigLIP2_Qwen_Likelihood_correct"
    ]
)

integrated["either_ensemble_wrong"] = (
    ~integrated[
        "CLIP_SigLIP_Qwen_correct"
    ]
    |
    ~integrated[
        "SigLIP2_Qwen_Likelihood_correct"
    ]
)

integrated["both_ensembles_correct"] = (
    integrated[
        "CLIP_SigLIP_Qwen_correct"
    ]
    &
    integrated[
        "SigLIP2_Qwen_Likelihood_correct"
    ]
)

# ------------------------------------------------------------
# 5. Validação das avaliações humanas
# ------------------------------------------------------------

assert integrated["n_human"].notna().all()

print("\n=== AUDITORIA HUMANA ===")

print(
    "Total de avaliações humanas:",
    int(integrated["n_human"].sum())
)

print(
    "Mínimo por imagem:",
    int(integrated["n_human"].min())
)

print(
    "Máximo por imagem:",
    int(integrated["n_human"].max())
)

print(
    "Média por imagem:",
    round(integrated["n_human"].mean(), 3)
)

print(
    "Total Pain:",
    int(integrated["n_pain"].sum())
)

print(
    "Total NoPain:",
    int(integrated["n_nopain"].sum())
)

print(
    "Total indeterminado:",
    int(integrated["n_indeterminate"].sum())
)

# ------------------------------------------------------------
# 6. Caracterização das imagens erradas
# ------------------------------------------------------------

columns_display = [
    "image_number",
    "true_label",
    "n_human",
    "n_pain",
    "n_nopain",
    "n_indeterminate",
    "reference_agreement_fraction",
    "human_disagreement",
    "indeterminate_fraction"
]

error_rows = []

for name in best_ensembles:

    wrong = integrated.loc[
        ~integrated[f"{name}_correct"]
    ].copy()

    wrong["ensemble"] = name

    error_rows.append(wrong)

    print(f"\n=== ERROS: {name} ===")
    print(
        wrong[columns_display].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

errors = pd.concat(
    error_rows,
    ignore_index=True
)

# ------------------------------------------------------------
# 7. Erros compartilhados
# ------------------------------------------------------------

shared = integrated.loc[
    integrated["both_ensembles_wrong"]
].copy()

print("\n=== ERROS COMPARTILHADOS ===")
print(
    shared[columns_display].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    "\nQuantidade de erros compartilhados:",
    len(shared)
)

# ------------------------------------------------------------
# 8. Comparações descritivas
# ------------------------------------------------------------

descriptive_rows = []

human_variables = [
    "reference_agreement_fraction",
    "human_disagreement",
    "indeterminate_fraction",
    "n_human"
]

for name in best_ensembles:

    for status, mask in [
        (
            "correct",
            integrated[f"{name}_correct"]
        ),
        (
            "wrong",
            ~integrated[f"{name}_correct"]
        )
    ]:

        subset = integrated.loc[mask]

        row = {
            "ensemble": name,
            "status": status,
            "n_images": len(subset)
        }

        for variable in human_variables:

            values = pd.to_numeric(
                subset[variable],
                errors="coerce"
            )

            row[f"{variable}_mean"] = (
                values.mean()
            )

            row[f"{variable}_median"] = (
                values.median()
            )

        descriptive_rows.append(row)

descriptive = pd.DataFrame(descriptive_rows)

print("\n=== COMPARAÇÃO DESCRITIVA ===")
print(
    descriptive.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 9. Salvar sem sobrescrever
# ------------------------------------------------------------

outputs = {
    "florence_exp27a1_human_ensemble_integrated.csv":
        integrated,
    "florence_exp27a1_ensemble_errors.csv":
        errors,
    "florence_exp27a1_human_descriptive.csv":
        descriptive
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(
        R / filename,
        index=False
    )

print("\n=== ARQUIVOS GERADOS ===")

for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 27A.1 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 27A.1
INTEGRAÇÃO DOS ENSEMBLES COM AS AVALIAÇÕES HUMANAS

Arquivos carregados:
Humanos: 189 registros
Ensembles: 189 registros

=== AUDITORIA HUMANA ===
Total de avaliações humanas: 746
Mínimo por imagem: 3
Máximo por imagem: 7
Média por imagem: 3.947
Total Pain: 365
Total NoPain: 350
Total indeterminado: 31

=== ERROS: CLIP_SigLIP_Qwen ===
 image_number  true_label  n_human  n_pain  n_nopain  n_indeterminate  reference_agreement_fraction  human_disagreement  indeterminate_fraction
           44           1        3       3         0                0                        1.0000              0.0000                  0.0000
           56           1        5       5         0                0                        1.0000              0.0000                  0.0000
           59           1        5       5         0                0                        1.0000              0.0000                  0.0000
           98           0        3       0         2      

## To formally test whether the occurrence of ensemble errors is associated with the fraction of indeterminate responses and human disagreement.

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 27B
# ERROS DOS ENSEMBLES × INCERTEZA HUMANA
#
# Permutação estratificada pela classe verdadeira
# 2 ensembles × 2 variáveis = 4 testes
# Correção de Holm
#
# Sem GPU. Sem novos modelos.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

R = Path("/content/florence_visapp/results")

SOURCE = (
    R / "florence_exp27a1_human_ensemble_integrated.csv"
)

df = pd.read_csv(SOURCE)

assert len(df) == 189
assert df["image_number"].is_unique

ensembles = [
    "CLIP_SigLIP_Qwen",
    "SigLIP2_Qwen_Likelihood"
]

variables = [
    "human_disagreement",
    "indeterminate_fraction"
]

y = df["true_label"].astype(int).to_numpy()

assert set(np.unique(y)) == {0, 1}

rng = np.random.default_rng(2702)
N_PERM = 20000

# ------------------------------------------------------------
# 1. Teste de permutação estratificado
# ------------------------------------------------------------

def stratified_permutation_test(
    errors,
    values,
    strata,
    n_perm=20000
):
    errors = np.asarray(errors, dtype=bool)
    values = np.asarray(values, dtype=float)
    strata = np.asarray(strata)

    assert np.isfinite(values).all()
    assert errors.sum() > 0
    assert (~errors).sum() > 0

    observed = (
        values[errors].mean()
        - values[~errors].mean()
    )

    indices = [
        np.where(strata == label)[0]
        for label in np.unique(strata)
    ]

    extreme = 0

    for _ in range(n_perm):

        permuted = errors.copy()

        for idx in indices:
            permuted[idx] = rng.permutation(
                errors[idx]
            )

        statistic = (
            values[permuted].mean()
            - values[~permuted].mean()
        )

        if abs(statistic) >= abs(observed) - 1e-12:
            extreme += 1

    p_value = (
        extreme + 1
    ) / (
        n_perm + 1
    )

    return observed, p_value

# ------------------------------------------------------------
# 2. Executar os quatro testes
# ------------------------------------------------------------

rows = []

for ensemble in ensembles:

    predictions = df[ensemble].astype(int).to_numpy()

    errors = predictions != y

    assert int(errors.sum()) == 6

    for variable in variables:

        values = pd.to_numeric(
            df[variable],
            errors="raise"
        ).to_numpy(dtype=float)

        difference, p = stratified_permutation_test(
            errors=errors,
            values=values,
            strata=y,
            n_perm=N_PERM
        )

        rows.append({
            "ensemble": ensemble,
            "human_variable": variable,
            "n_errors": int(errors.sum()),
            "mean_errors": float(
                values[errors].mean()
            ),
            "mean_correct": float(
                values[~errors].mean()
            ),
            "difference_errors_minus_correct":
                float(difference),
            "permutation_p": float(p)
        })

tests = pd.DataFrame(rows)

tests["holm_p"] = multipletests(
    tests["permutation_p"],
    method="holm"
)[1]

tests["significant_holm_005"] = (
    tests["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 3. Distribuição de erros por classe
# ------------------------------------------------------------

class_rows = []

for ensemble in ensembles:

    errors = (
        df[ensemble].astype(int).to_numpy()
        != y
    )

    for label in [0, 1]:

        mask = y == label

        class_rows.append({
            "ensemble": ensemble,
            "true_label": label,
            "n_images": int(mask.sum()),
            "n_errors": int(
                np.sum(errors & mask)
            ),
            "error_rate": float(
                np.mean(errors[mask])
            )
        })

class_table = pd.DataFrame(class_rows)

# ------------------------------------------------------------
# 4. Salvar resultados
# ------------------------------------------------------------

outputs = {
    "florence_exp27b_permutation_holm.csv":
        tests,
    "florence_exp27b_errors_by_class.csv":
        class_table
}

existing = [
    filename
    for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(
        R / filename,
        index=False
    )

# ------------------------------------------------------------
# 5. Exibir resultados
# ------------------------------------------------------------

print("=" * 80)
print("FLORENCE — EXPERIMENTO 27B")
print("INCERTEZA HUMANA × ERROS DOS ENSEMBLES")
print("=" * 80)

print("\n=== TESTES DE PERMUTAÇÃO ===")
print(
    tests.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== ERROS POR CLASSE ===")
print(
    class_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== ARQUIVOS GERADOS ===")

for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 27B CONCLUÍDO.")


FLORENCE — EXPERIMENTO 27B
INCERTEZA HUMANA × ERROS DOS ENSEMBLES

=== TESTES DE PERMUTAÇÃO ===
               ensemble         human_variable  n_errors  mean_errors  mean_correct  difference_errors_minus_correct  permutation_p   holm_p  significant_holm_005
       CLIP_SigLIP_Qwen     human_disagreement         6     0.000000      0.021194                        -0.021194       0.970251 0.970251                 False
       CLIP_SigLIP_Qwen indeterminate_fraction         6     0.166667      0.046539                         0.120128       0.019499 0.077996                 False
SigLIP2_Qwen_Likelihood     human_disagreement         6     0.071429      0.018852                         0.052576       0.098745 0.197490                 False
SigLIP2_Qwen_Likelihood indeterminate_fraction         6     0.111111      0.048361                         0.062750       0.052597 0.157792                 False

=== ERROS POR CLASSE ===
               ensemble  true_label  n_images  n_errors  error_

# Inventory and integrity of CSVs

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28A.1
# AUDITORIA FINAL: INVENTÁRIO E INTEGRIDADE DOS CSVs
#
# Sem GPU, sem inferência e sem reprocessamento.
# Preserva todos os arquivos existentes.
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib

R = Path("/content/florence_visapp/results")

assert R.exists(), f"Diretório não encontrado: {R}"

print("=" * 80)
print("FLORENCE — EXPERIMENTO 28A.1")
print("AUDITORIA FINAL DOS RESULTADOS EXPERIMENTAIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Inventário de todos os CSVs
# ------------------------------------------------------------

files = sorted(R.glob("*.csv"))

assert files, "Nenhum arquivo CSV encontrado."

records = []
problems = []

for path in files:

    record = {
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "rows": np.nan,
        "columns": np.nan,
        "duplicate_rows": np.nan,
        "missing_cells": np.nan,
        "sha256": "",
        "status": "OK"
    }

    try:
        data = pd.read_csv(path)

        record["rows"] = len(data)
        record["columns"] = len(data.columns)
        record["duplicate_rows"] = int(
            data.duplicated().sum()
        )
        record["missing_cells"] = int(
            data.isna().sum().sum()
        )

        h = hashlib.sha256()

        with path.open("rb") as f:
            for chunk in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):
                h.update(chunk)

        record["sha256"] = h.hexdigest()

        if data.empty:
            record["status"] = "EMPTY"
            problems.append(
                f"Arquivo vazio: {path.name}"
            )

    except Exception as exc:

        record["status"] = "READ_ERROR"

        problems.append(
            f"{path.name}: {type(exc).__name__}: {exc}"
        )

    records.append(record)

inventory = pd.DataFrame(records)

# ------------------------------------------------------------
# 2. Verificação dos arquivos centrais
# ------------------------------------------------------------

required = {
    "contrastive": [
        "clip_vit_b32_predictions.csv",
        "siglip_base_predictions.csv",
        "siglip2_base_predictions.csv"
    ],
    "generative": [
        "florence_exp25_image_level.csv"
    ],
    "architecture_comparison": [
        "florence_exp26b2_integrated.csv",
        "florence_exp26b2_metrics.csv",
        "florence_exp26b2_main_15_mcnemar_holm.csv",
        "florence_exp26b2_all_28_mcnemar_holm.csv"
    ],
    "complementarity": [
        "florence_exp26c1_image_analysis.csv",
        "florence_exp26c1_pairwise_oracle.csv"
    ],
    "ensembles": [
        "florence_exp26c2_ensemble_predictions.csv",
        "florence_exp26c2_ensemble_metrics.csv",
        "florence_exp26c2_ensemble_mcnemar.csv"
    ],
    "human_assessment": [
        "human_model_error_association.csv",
        "florence_exp27a1_human_ensemble_integrated.csv",
        "florence_exp27b_permutation_holm.csv"
    ]
}

checks = []

for category, filenames in required.items():

    for filename in filenames:

        path = R / filename

        checks.append({
            "category": category,
            "filename": filename,
            "exists": path.exists(),
            "size_bytes": (
                path.stat().st_size
                if path.exists() else 0
            )
        })

checks = pd.DataFrame(checks)

# ------------------------------------------------------------
# 3. Auditoria estrutural das tabelas por imagem
# ------------------------------------------------------------

image_files = [
    "florence_exp25_image_level.csv",
    "florence_exp26b2_integrated.csv",
    "florence_exp26c1_image_analysis.csv",
    "florence_exp26c2_ensemble_predictions.csv",
    "human_model_error_association.csv",
    "florence_exp27a1_human_ensemble_integrated.csv"
]

structure = []

for filename in image_files:

    path = R / filename

    if not path.exists():
        continue

    data = pd.read_csv(path)

    id_column = next(
        (
            col for col in [
                "image_number",
                "image"
            ]
            if col in data.columns
        ),
        None
    )

    if id_column is None:
        unique_ids = np.nan
        duplicate_ids = np.nan
    else:
        unique_ids = data[id_column].nunique()
        duplicate_ids = int(
            data[id_column].duplicated().sum()
        )

    structure.append({
        "filename": filename,
        "rows": len(data),
        "id_column": id_column,
        "unique_ids": unique_ids,
        "duplicate_ids": duplicate_ids,
        "has_189_rows": len(data) == 189
    })

structure = pd.DataFrame(structure)

# ------------------------------------------------------------
# 4. Exibição
# ------------------------------------------------------------

print("\n=== INVENTÁRIO GERAL ===")
print("Total de CSVs:", len(files))

print("\n=== ARQUIVOS CENTRAIS ===")
print(checks.to_string(index=False))

print("\n=== INTEGRIDADE DAS TABELAS POR IMAGEM ===")
print(structure.to_string(index=False))

print("\n=== ARQUIVOS COM PROBLEMAS DE LEITURA OU VAZIOS ===")

if problems:
    for problem in problems:
        print("-", problem)
else:
    print("Nenhum problema de leitura detectado.")

print("\n=== ARQUIVOS COM VALORES AUSENTES ===")

missing = inventory.loc[
    inventory["missing_cells"].fillna(0) > 0,
    ["filename", "rows", "missing_cells"]
]

if missing.empty:
    print("Nenhum.")
else:
    print(missing.to_string(index=False))

print("\n=== ARQUIVOS COM LINHAS DUPLICADAS ===")

duplicates = inventory.loc[
    inventory["duplicate_rows"].fillna(0) > 0,
    ["filename", "rows", "duplicate_rows"]
]

if duplicates.empty:
    print("Nenhum.")
else:
    print(duplicates.to_string(index=False))

# ------------------------------------------------------------
# 5. Exportar auditoria sem sobrescrever
# ------------------------------------------------------------

outputs = {
    "florence_exp28a1_file_inventory.csv": inventory,
    "florence_exp28a1_required_files.csv": checks,
    "florence_exp28a1_image_structure.csv": structure
}

existing = [
    filename
    for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos de auditoria já existentes: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(
        R / filename,
        index=False
    )

print("\n=== RESUMO ===")
print("Arquivos CSV auditados:", len(files))
print(
    "Arquivos centrais encontrados:",
    int(checks["exists"].sum()),
    "/",
    len(checks)
)
print("Problemas de leitura:", len(problems))

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 28A.1 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28A.1
AUDITORIA FINAL DOS RESULTADOS EXPERIMENTAIS

=== INVENTÁRIO GERAL ===
Total de CSVs: 105

=== ARQUIVOS CENTRAIS ===
               category                                       filename  exists  size_bytes
            contrastive                   clip_vit_b32_predictions.csv    True       36705
            contrastive                    siglip_base_predictions.csv    True       37771
            contrastive                   siglip2_base_predictions.csv    True       38147
             generative                 florence_exp25_image_level.csv    True       15829
architecture_comparison                florence_exp26b2_integrated.csv    True       25543
architecture_comparison                   florence_exp26b2_metrics.csv    True        1496
architecture_comparison      florence_exp26b2_main_15_mcnemar_holm.csv    True        2524
architecture_comparison       florence_exp26b2_all_28_mcnemar_holm.csv    True        4639
        complementarity            

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28A.2
# AUDITORIA CRUZADA DE RESULTADOS E ESTATÍSTICAS
#
# Sem GPU. Sem novas inferências.
# Sem sobrescrever arquivos anteriores.
# ============================================================

from pathlib import Path
from itertools import combinations
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

R = Path("/content/florence_visapp/results")

models = [
    "CLIP", "SigLIP", "SigLIP2",
    "BLIP2", "Qwen", "Likelihood",
    "Llama_restricted", "Llama_original"
]

main_models = [
    "CLIP", "SigLIP", "SigLIP2",
    "BLIP2", "Qwen", "Likelihood"
]

integrated = pd.read_csv(
    R / "florence_exp26b2_integrated.csv"
)

metrics_saved = pd.read_csv(
    R / "florence_exp26b2_metrics.csv"
)

ensemble_saved = pd.read_csv(
    R / "florence_exp26c2_ensemble_predictions.csv"
)

human = pd.read_csv(
    R / "human_model_error_association.csv"
)

mcnemar_saved = pd.read_csv(
    R / "florence_exp26b2_main_15_mcnemar_holm.csv"
)

assert len(integrated) == 189
assert integrated["image_number"].is_unique

integrated = integrated.sort_values(
    "image_number"
).reset_index(drop=True)

y = integrated["CLIP_truth"].astype(int).to_numpy()

assert np.sum(y == 1) == 99
assert np.sum(y == 0) == 90

issues = []
checks = []

def record(name, passed, detail=""):
    checks.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail)
    })
    if not passed:
        issues.append(f"{name}: {detail}")

def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.to_numpy(dtype=bool)

    normalized = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }

    converted = normalized.map(mapping)

    if converted.isna().any():
        raise ValueError(
            f"Valores booleanos inesperados: "
            f"{series.name}"
        )

    return converted.to_numpy(dtype=bool)

print("=" * 80)
print("FLORENCE — EXPERIMENTO 28A.2")
print("AUDITORIA CRUZADA DOS RESULTADOS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Alinhamento e métricas individuais
# ------------------------------------------------------------

rows = []
correctness = {}

for model in models:

    truth = integrated[
        f"{model}_truth"
    ].astype(int).to_numpy()

    record(
        f"{model}: truth alignment",
        np.array_equal(truth, y)
    )

    valid = to_bool(
        integrated[f"{model}_valid"]
    )

    prediction = pd.to_numeric(
        integrated[f"{model}_prediction"],
        errors="coerce"
    ).to_numpy()

    valid_prediction_ok = np.isin(
        prediction[valid], [0, 1]
    ).all()

    record(
        f"{model}: valid predictions binary",
        valid_prediction_ok
    )

    correct = (
        valid
        & (prediction == y)
    )

    saved_correct = to_bool(
        integrated[f"{model}_correct"]
    )

    record(
        f"{model}: correctness alignment",
        np.array_equal(correct, saved_correct)
    )

    correctness[model] = correct

    tn = int(np.sum(valid & (y == 0) & (prediction == 0)))
    fp = int(np.sum(valid & (y == 0) & (prediction == 1)))
    fn = int(np.sum(valid & (y == 1) & (prediction == 0)))
    tp = int(np.sum(valid & (y == 1) & (prediction == 1)))

    rows.append({
        "condition": model,
        "n_images": len(y),
        "n_valid": int(valid.sum()),
        "n_invalid": int((~valid).sum()),
        "n_correct": int(correct.sum()),
        "operational_accuracy": float(correct.mean()),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

recomputed = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2. Comparação com métricas salvas
# ------------------------------------------------------------

record(
    "Metrics: conditions present",
    set(models).issubset(
        set(metrics_saved["condition"])
    )
)

saved_index = metrics_saved.set_index("condition")

for _, row in recomputed.iterrows():

    model = row["condition"]

    if model not in saved_index.index:
        continue

    saved = saved_index.loc[model]

    for col in [
        "n_valid", "n_invalid", "n_correct",
        "TN", "FP", "FN", "TP"
    ]:

        if col in saved_index.columns:

            record(
                f"{model}: saved {col}",
                int(saved[col]) == int(row[col])
            )

    if "operational_accuracy" in saved_index.columns:

        record(
            f"{model}: saved operational accuracy",
            np.isclose(
                float(saved["operational_accuracy"]),
                row["operational_accuracy"],
                atol=1e-9
            )
        )

# ------------------------------------------------------------
# 3. Reproduzir os três ensembles
# ------------------------------------------------------------

ensemble_definitions = {
    "SigLIP2_Qwen_Likelihood":
        ["SigLIP2", "Qwen", "Likelihood"],
    "CLIP_SigLIP_Qwen":
        ["CLIP", "SigLIP", "Qwen"],
    "SigLIP_SigLIP2_Qwen":
        ["SigLIP", "SigLIP2", "Qwen"]
}

aligned_ensemble = ensemble_saved.set_index(
    "image_number"
).loc[
    integrated["image_number"]
].reset_index()

for name, members in ensemble_definitions.items():

    votes = np.column_stack([
        integrated[f"{model}_prediction"]
        .astype(int)
        .to_numpy()
        for model in members
    ])

    recomputed_prediction = (
        votes.sum(axis=1) >= 2
    ).astype(int)

    saved_prediction = aligned_ensemble[
        name
    ].astype(int).to_numpy()

    record(
        f"Ensemble {name}: vote reproduction",
        np.array_equal(
            recomputed_prediction,
            saved_prediction
        )
    )

    record(
        f"Ensemble {name}: expected 183/180",
        int(np.sum(recomputed_prediction == y))
        == (
            180 if name == "SigLIP_SigLIP2_Qwen"
            else 183
        )
    )

# ------------------------------------------------------------
# 4. Avaliações humanas
# ------------------------------------------------------------

record(
    "Human: 189 unique images",
    len(human) == 189
    and human["image_number"].is_unique
)

record(
    "Human: image IDs aligned",
    set(human["image_number"])
    == set(integrated["image_number"])
)

human_index = human.set_index("image_number").loc[
    integrated["image_number"]
]

record(
    "Human: reference labels aligned",
    np.array_equal(
        human_index["true_label"].astype(int).to_numpy(),
        y
    )
)

for column, expected in [
    ("n_human", 746),
    ("n_pain", 365),
    ("n_nopain", 350),
    ("n_indeterminate", 31)
]:

    record(
        f"Human: total {column}",
        int(human[column].sum()) == expected
    )

if "n_unrecognized" in human.columns:

    total_categories = (
        human["n_pain"]
        + human["n_nopain"]
        + human["n_indeterminate"]
        + human["n_unrecognized"]
    )

    record(
        "Human: category totals per image",
        np.array_equal(
            total_categories.to_numpy(),
            human["n_human"].to_numpy()
        )
    )

# ------------------------------------------------------------
# 5. Recalcular McNemar exato (15 pares)
# ------------------------------------------------------------

mcnemar_rows = []

for a, b in combinations(main_models, 2):

    ca = correctness[a]
    cb = correctness[b]

    a_only = int(np.sum(ca & ~cb))
    b_only = int(np.sum(~ca & cb))

    discordant = a_only + b_only

    p = (
        binomtest(
            a_only,
            discordant,
            0.5
        ).pvalue
        if discordant else 1.0
    )

    mcnemar_rows.append({
        "model_a": a,
        "model_b": b,
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "discordant": discordant,
        "mcnemar_exact_p": p
    })

mcnemar = pd.DataFrame(mcnemar_rows)

mcnemar["holm_p"] = multipletests(
    mcnemar["mcnemar_exact_p"],
    method="holm"
)[1]

record(
    "McNemar: 15 comparisons",
    len(mcnemar) == 15
)

record(
    "McNemar: 4 significant after Holm",
    int((mcnemar["holm_p"] < 0.05).sum()) == 4
)

# ------------------------------------------------------------
# 6. Salvar auditoria
# ------------------------------------------------------------

audit = pd.DataFrame(checks)

outputs = {
    "florence_exp28a2_audit_checks.csv":
        audit,
    "florence_exp28a2_recomputed_metrics.csv":
        recomputed,
    "florence_exp28a2_recomputed_mcnemar.csv":
        mcnemar
}

existing = [
    name for name in outputs
    if (R / name).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for name, table in outputs.items():
    table.to_csv(R / name, index=False)

# ------------------------------------------------------------
# 7. Exibir síntese
# ------------------------------------------------------------

print("\n=== MÉTRICAS RECONSTRUÍDAS ===")
print(
    recomputed.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== AUDITORIA ===")
print("Verificações:", len(audit))
print("Aprovadas:", int(audit["passed"].sum()))
print("Falhas:", len(issues))

if issues:
    print("\n=== INCONSISTÊNCIAS ===")
    for issue in issues:
        print("-", issue)
else:
    print("Nenhuma inconsistência detectada nos testes executados.")

print("\n=== MCNEMAR PRINCIPAL ===")
print(
    mcnemar.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n=== ARQUIVOS GERADOS ===")
for name in outputs:
    print(R / name)

print("\nEXPERIMENTO 28A.2 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28A.2
AUDITORIA CRUZADA DOS RESULTADOS

=== MÉTRICAS RECONSTRUÍDAS ===
       condition  n_images  n_valid  n_invalid  n_correct  operational_accuracy  TN  FP  FN  TP
            CLIP       189      189          0        167              0.883598  73  17   5  94
          SigLIP       189      189          0        177              0.936508  89   1  11  88
         SigLIP2       189      189          0        176              0.931217  85   5   8  91
           BLIP2       189      180          9        160              0.846561  66  17   3  94
            Qwen       189      189          0        179              0.947090  87   3   7  92
      Likelihood       189      189          0        181              0.957672  89   1   7  92
Llama_restricted       189      189          0        177              0.936508  87   3   9  90
  Llama_original       189      163         26        157              0.830688  69   1   5  88

=== AUDITORIA ===
Verificações: 97
Aprova

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28B.1
# AUDITORIA DAS ESTRATÉGIAS DE PROMPTING
#
# Inspeção de protocolos, colunas e cobertura.
# Sem GPU, sem inferências e sem sobrescrita.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

R = Path("/content/florence_visapp/results")

files = {
    "CLIP": "clip_vit_b32_predictions.csv",
    "SigLIP": "siglip_base_predictions.csv",
    "SigLIP2": "siglip2_base_predictions.csv",
    "CLIP_semantic": "clip_semantic_variants.csv",
    "SigLIP_semantic": "siglip_semantic_variants.csv",
    "SigLIP2_semantic": "siglip2_semantic_variants.csv"
}

print("=" * 78)
print("FLORENCE — EXPERIMENTO 28B.1")
print("AUDITORIA DOS PROTOCOLOS DE PROMPTING")
print("=" * 78)

for model, filename in files.items():

    path = R / filename

    print("\n" + "-" * 78)
    print("MODELO / ARQUIVO:", model, filename)

    if not path.exists():
        print("STATUS: ARQUIVO NÃO ENCONTRADO")
        continue

    df = pd.read_csv(path)

    print("Linhas:", len(df))
    print("Colunas:", len(df.columns))
    print("Nomes das colunas:")
    print(df.columns.tolist())

    print("\nPrimeiros registros:")
    print(df.head(3).to_string(index=False))

    for column in [
        "strategy",
        "variant",
        "prompt_variant",
        "prompt_type"
    ]:
        if column in df.columns:
            print(f"\nDistribuição de {column}:")
            print(
                df[column]
                .value_counts(dropna=False)
                .to_string()
            )

    if "image" in df.columns:

        print(
            "\nImagens distintas:",
            df["image"].nunique()
        )

    if "image_number" in df.columns:

        print(
            "\nIDs distintos:",
            df["image_number"].nunique()
        )

    if "true_label" in df.columns:

        print("\nDistribuição de rótulos:")
        print(
            df["true_label"]
            .value_counts(dropna=False)
            .to_string()
        )

    if "correct" in df.columns:

        print("\nDistribuição de acertos:")
        print(
            df["correct"]
            .value_counts(dropna=False)
            .to_string()
        )

    print(
        "\nCélulas ausentes:",
        int(df.isna().sum().sum())
    )

print("\n" + "=" * 78)
print("EXPERIMENTO 28B.1 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28B.1
AUDITORIA DOS PROTOCOLOS DE PROMPTING

------------------------------------------------------------------------------
MODELO / ARQUIVO: CLIP clip_vit_b32_predictions.csv
Linhas: 378
Colunas: 8
Nomes das colunas:
['image', 'true_label', 'model', 'strategy', 'predicted_label', 'predicted_prompt', 'pain_score', 'correct']

Primeiros registros:
                        image  true_label                        model  strategy  predicted_label  predicted_prompt  pain_score  correct
  PainImages/pain/image_1.jpg           1 openai/clip-vit-base-patch32    binary                1                 0   25.946909        1
  PainImages/pain/image_1.jpg           1 openai/clip-vit-base-patch32 affective                1                 0   25.946909        1
PainImages/pain/image_103.jpg           1 openai/clip-vit-base-patch32    binary                1                 0   27.360876        1

Distribuição de strategy:
strategy
binary       189
affective    189

Imagens d

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28B.2
# AUDITORIA ESTATÍSTICA DAS ESTRATÉGIAS DE PROMPTING
#
# 3 arquiteturas × 4 estratégias
# 9 comparações pareadas contra o afetivo original
# McNemar exato + Holm (família de 9 testes)
#
# Sem GPU, sem inferência, sem sobrescrever arquivos.
# ============================================================

from pathlib import Path
from itertools import combinations
import re

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import confusion_matrix

R = Path("/content/florence_visapp/results")

sources = {
    "CLIP": {
        "original": "clip_vit_b32_predictions.csv",
        "semantic": "clip_semantic_variants.csv"
    },
    "SigLIP": {
        "original": "siglip_base_predictions.csv",
        "semantic": "siglip_semantic_variants.csv"
    },
    "SigLIP2": {
        "original": "siglip2_base_predictions.csv",
        "semantic": "siglip2_semantic_variants.csv"
    }
}

strategies = [
    "binary",
    "affective",
    "B_experiential",
    "C_observable"
]

expected_ids = set(range(1, 141)) | set(range(151, 200))

def extract_id(path):
    match = re.search(
        r"(?:image|imagem)_(\d+)\.[^./\\]+$",
        str(path),
        flags=re.IGNORECASE
    )
    if match is None:
        raise ValueError(f"ID não reconhecido: {path}")
    return int(match.group(1))

def load_source(filename, model):
    path = R / filename
    assert path.exists(), f"Arquivo ausente: {path}"

    data = pd.read_csv(path)

    required = [
        "image", "true_label", "strategy",
        "predicted_label", "correct"
    ]
    assert set(required).issubset(data.columns)

    data["image_number"] = data["image"].map(extract_id)
    data["architecture"] = model

    for col in ["true_label", "predicted_label", "correct"]:
        data[col] = pd.to_numeric(
            data[col], errors="raise"
        ).astype(int)

        assert data[col].isin([0, 1]).all(), (
            f"Valores inesperados em {filename}: {col}"
        )

    assert not data.duplicated(
        ["image_number", "strategy"]
    ).any(), f"Duplicação em {filename}"

    assert (
        data["correct"]
        == (
            data["predicted_label"]
            == data["true_label"]
        ).astype(int)
    ).all(), f"Coluna correct inconsistente: {filename}"

    return data[
        [
            "architecture", "image_number",
            "true_label", "strategy",
            "predicted_label", "correct"
        ]
    ]

frames = []

for model, filenames in sources.items():
    original = load_source(filenames["original"], model)
    semantic = load_source(filenames["semantic"], model)

    assert set(original["strategy"]) == {
        "binary", "affective"
    }
    assert set(semantic["strategy"]) == {
        "B_experiential", "C_observable"
    }

    frames.extend([original, semantic])

data = pd.concat(frames, ignore_index=True)

# ------------------------------------------------------------
# 1. Auditoria da cobertura e dos rótulos
# ------------------------------------------------------------

for model in sources:
    subset = data[data["architecture"] == model]

    assert len(subset) == 756

    for strategy in strategies:
        part = subset[subset["strategy"] == strategy]

        assert len(part) == 189
        assert set(part["image_number"]) == expected_ids
        assert part["image_number"].is_unique
        assert int(part["true_label"].sum()) == 99

truth_consistency = data.groupby(
    "image_number"
)["true_label"].nunique()

assert (truth_consistency == 1).all(), (
    "Divergência entre rótulos de referência"
)

# ------------------------------------------------------------
# 2. Matriz por imagem, modelo e estratégia
# ------------------------------------------------------------

wide = data.pivot(
    index=["architecture", "image_number", "true_label"],
    columns="strategy",
    values="predicted_label"
).reset_index()

wide.columns.name = None

assert len(wide) == 567
assert not wide[strategies].isna().any().any()

for strategy in strategies:
    wide[strategy] = wide[strategy].astype(int)

# ------------------------------------------------------------
# 3. Reconstrução das métricas
# ------------------------------------------------------------

metrics_rows = []

for model in sources:
    subset = wide[wide["architecture"] == model]
    y = subset["true_label"].to_numpy()

    for strategy in strategies:
        p = subset[strategy].to_numpy()

        tn, fp, fn, tp = confusion_matrix(
            y, p, labels=[0, 1]
        ).ravel()

        precision = (
            tp / (tp + fp)
            if tp + fp else np.nan
        )
        recall = tp / (tp + fn)
        specificity = tn / (tn + fp)

        f1 = (
            2 * tp / (2 * tp + fp + fn)
            if 2 * tp + fp + fn else np.nan
        )

        metrics_rows.append({
            "architecture": model,
            "strategy": strategy,
            "n_images": len(y),
            "n_correct": int(np.sum(p == y)),
            "accuracy": float(np.mean(p == y)),
            "precision": float(precision),
            "sensitivity": float(recall),
            "specificity": float(specificity),
            "balanced_accuracy": float(
                (recall + specificity) / 2
            ),
            "f1": float(f1),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        })

metrics = pd.DataFrame(metrics_rows)

# ------------------------------------------------------------
# 4. McNemar exato: afetivo original vs alternativas
# ------------------------------------------------------------

tests_rows = []

for model in sources:
    subset = wide[wide["architecture"] == model]
    y = subset["true_label"].to_numpy()

    reference = subset["affective"].to_numpy() == y

    for alternative in [
        "binary", "B_experiential", "C_observable"
    ]:
        candidate = subset[alternative].to_numpy() == y

        affective_only = int(
            np.sum(reference & ~candidate)
        )
        alternative_only = int(
            np.sum(~reference & candidate)
        )

        discordant = affective_only + alternative_only

        p_value = (
            binomtest(
                affective_only, discordant, 0.5
            ).pvalue
            if discordant else 1.0
        )

        tests_rows.append({
            "architecture": model,
            "reference": "affective",
            "alternative": alternative,
            "affective_only_correct": affective_only,
            "alternative_only_correct": alternative_only,
            "discordant": discordant,
            "affective_gain_pp": float(
                100 * (
                    reference.mean() - candidate.mean()
                )
            ),
            "mcnemar_exact_p": float(p_value)
        })

tests = pd.DataFrame(tests_rows)

assert len(tests) == 9

tests["holm_p"] = multipletests(
    tests["mcnemar_exact_p"],
    method="holm"
)[1]

tests["significant_holm_005"] = (
    tests["holm_p"] < 0.05
)

# ------------------------------------------------------------
# 5. Estabilidade entre as três formulações afetivas
# ------------------------------------------------------------

stability_rows = []

affective_variants = [
    "affective",
    "B_experiential",
    "C_observable"
]

for model in sources:
    subset = wide[wide["architecture"] == model]

    predictions = subset[
        affective_variants
    ].to_numpy()

    all_agree = (
        predictions == predictions[:, [0]]
    ).all(axis=1)

    stability_rows.append({
        "architecture": model,
        "n_images": len(subset),
        "all_three_agree": int(all_agree.sum()),
        "all_three_agreement_fraction": float(
            all_agree.mean()
        ),
        "affective_vs_experiential_agreement": float(
            np.mean(predictions[:, 0] == predictions[:, 1])
        ),
        "affective_vs_observable_agreement": float(
            np.mean(predictions[:, 0] == predictions[:, 2])
        ),
        "experiential_vs_observable_agreement": float(
            np.mean(predictions[:, 1] == predictions[:, 2])
        )
    })

stability = pd.DataFrame(stability_rows)

# ------------------------------------------------------------
# 6. Salvar sem sobrescrever
# ------------------------------------------------------------

outputs = {
    "florence_exp28b2_prompt_predictions.csv": wide,
    "florence_exp28b2_prompt_metrics.csv": metrics,
    "florence_exp28b2_prompt_mcnemar_holm.csv": tests,
    "florence_exp28b2_prompt_stability.csv": stability
}

existing = [
    filename for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: " + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 7. Exibir resultados
# ------------------------------------------------------------

print("=" * 80)
print("FLORENCE — EXPERIMENTO 28B.2")
print("AUDITORIA ESTATÍSTICA DAS ESTRATÉGIAS DE PROMPTING")
print("=" * 80)

print("\n=== MÉTRICAS POR MODELO E ESTRATÉGIA ===")
print(metrics.to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}"
))

print("\n=== MCNEMAR EXATO + HOLM (9 TESTES) ===")
print(tests.to_string(
    index=False,
    float_format=lambda x: f"{x:.8f}"
))

print("\n=== ESTABILIDADE DAS PREDIÇÕES BINÁRIAS ===")
print(stability.to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}"
))

print("\n=== RESUMO ===")
print("Arquiteturas:", len(sources))
print("Estratégias por arquitetura:", len(strategies))
print("Predições auditadas:", len(data))
print("Comparações estatísticas:", len(tests))
print(
    "Comparações significativas após Holm:",
    int(tests["significant_holm_005"].sum())
)

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 28B.2 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28B.2
AUDITORIA ESTATÍSTICA DAS ESTRATÉGIAS DE PROMPTING

=== MÉTRICAS POR MODELO E ESTRATÉGIA ===
architecture       strategy  n_images  n_correct  accuracy  precision  sensitivity  specificity  balanced_accuracy       f1  TN  FP  FN  TP
        CLIP         binary       189        127  0.671958   0.643411     0.838384     0.488889           0.663636 0.728070  44  46  16  83
        CLIP      affective       189        167  0.883598   0.846847     0.949495     0.811111           0.880303 0.895238  73  17   5  94
        CLIP B_experiential       189        128  0.677249   0.952381     0.404040     0.977778           0.690909 0.567376  88   2  59  40
        CLIP   C_observable       189        158  0.835979   0.947368     0.727273     0.955556           0.841414 0.822857  86   4  27  72
      SigLIP         binary       189        134  0.708995   0.775000     0.626263     0.800000           0.713131 0.692737  72  18  37  62
      SigLIP      affective       189 

# Consolidation of scientific results for the manuscript.

## Automatic consolidation of tables

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28C.1
# CONSOLIDAÇÃO DAS TABELAS CIENTÍFICAS — VISAPP
#
# Somente dados existentes.
# Sem GPU, inferência ou reprocessamento.
# Não sobrescreve arquivos anteriores.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

R = Path("/content/florence_visapp/results")

inputs = {
    "architecture":
        "florence_exp28a2_recomputed_metrics.csv",
    "prompt":
        "florence_exp28b2_prompt_metrics.csv",
    "prompt_tests":
        "florence_exp28b2_prompt_mcnemar_holm.csv",
    "stability":
        "florence_exp28b2_prompt_stability.csv",
    "ensemble":
        "florence_exp26c2_ensemble_metrics.csv",
    "human":
        "human_model_error_association.csv",
    "human_tests":
        "florence_exp27b_permutation_holm.csv"
}

data = {}

for key, filename in inputs.items():
    path = R / filename
    assert path.exists(), f"Arquivo ausente: {path}"
    data[key] = pd.read_csv(path)

print("=" * 80)
print("FLORENCE — EXPERIMENTO 28C.1")
print("CONSOLIDAÇÃO DAS TABELAS CIENTÍFICAS")
print("=" * 80)

# ------------------------------------------------------------
# TABELA A — ARQUITETURAS E CONDIÇÕES
# ------------------------------------------------------------

architecture = data["architecture"].copy()

expected = {
    "CLIP": 167,
    "SigLIP": 177,
    "SigLIP2": 176,
    "BLIP2": 160,
    "Qwen": 179,
    "Likelihood": 181,
    "Llama_restricted": 177,
    "Llama_original": 157
}

assert set(architecture["condition"]) == set(expected)

for model, n_correct in expected.items():
    row = architecture.loc[
        architecture["condition"] == model
    ]
    assert len(row) == 1
    assert int(row["n_correct"].iloc[0]) == n_correct

architecture["accuracy_pct"] = (
    100 * architecture["operational_accuracy"]
)

architecture["protocol_role"] = (
    architecture["condition"].map({
        "CLIP": "primary",
        "SigLIP": "primary",
        "SigLIP2": "primary",
        "BLIP2": "primary",
        "Qwen": "primary",
        "Likelihood": "exploratory_likelihood",
        "Llama_restricted": "secondary_decoding",
        "Llama_original": "secondary_decoding"
    })
)

architecture["architecture_family"] = (
    architecture["condition"].map({
        "CLIP": "contrastive",
        "SigLIP": "contrastive",
        "SigLIP2": "contrastive",
        "BLIP2": "generative",
        "Qwen": "generative",
        "Likelihood": "generative",
        "Llama_restricted": "generative",
        "Llama_original": "generative"
    })
)

architecture = architecture.sort_values(
    "operational_accuracy",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# TABELA B — PROMPTING
# ------------------------------------------------------------

prompt = data["prompt"].copy()

assert len(prompt) == 12
assert set(prompt["architecture"]) == {
    "CLIP", "SigLIP", "SigLIP2"
}

prompt["accuracy_pct"] = 100 * prompt["accuracy"]
prompt["sensitivity_pct"] = 100 * prompt["sensitivity"]
prompt["specificity_pct"] = 100 * prompt["specificity"]

prompt = prompt.sort_values(
    ["architecture", "strategy"]
).reset_index(drop=True)

# ------------------------------------------------------------
# TABELA C — TESTES DE PROMPTING
# ------------------------------------------------------------

prompt_tests = data["prompt_tests"].copy()

assert len(prompt_tests) == 9

prompt_tests["significant_holm"] = (
    prompt_tests["holm_p"] < 0.05
)

assert int(prompt_tests["significant_holm"].sum()) == 4

# ------------------------------------------------------------
# TABELA D — ENSEMBLES
# ------------------------------------------------------------

ensemble = data["ensemble"].copy()

print("\nColunas originais dos ensembles:")
print(ensemble.columns.tolist())

# Preservar todas as métricas originais.
# Acrescentar classificação metodológica.

ensemble["analysis_role"] = "exploratory_posthoc"

# ------------------------------------------------------------
# TABELA E — AVALIAÇÃO HUMANA
# ------------------------------------------------------------

human = data["human"].copy()

assert len(human) == 189
assert human["image_number"].is_unique

assert int(human["n_human"].sum()) == 746
assert int(human["n_pain"].sum()) == 365
assert int(human["n_nopain"].sum()) == 350
assert int(human["n_indeterminate"].sum()) == 31

human_summary = pd.DataFrame([{
    "n_images": len(human),
    "n_assessments": int(human["n_human"].sum()),
    "n_pain_responses": int(human["n_pain"].sum()),
    "n_nopain_responses": int(human["n_nopain"].sum()),
    "n_indeterminate_responses":
        int(human["n_indeterminate"].sum()),
    "min_assessments_per_image":
        int(human["n_human"].min()),
    "median_assessments_per_image":
        float(human["n_human"].median()),
    "mean_assessments_per_image":
        float(human["n_human"].mean()),
    "max_assessments_per_image":
        int(human["n_human"].max())
}])

# ------------------------------------------------------------
# TABELA F — ESTABILIDADE
# ------------------------------------------------------------

stability = data["stability"].copy()

assert len(stability) == 3

# ------------------------------------------------------------
# AUDITORIA FINAL DA CONSOLIDAÇÃO
# ------------------------------------------------------------

checks = pd.DataFrame([
    {
        "item": "architecture_conditions",
        "expected": 8,
        "observed": len(architecture)
    },
    {
        "item": "prompt_conditions",
        "expected": 12,
        "observed": len(prompt)
    },
    {
        "item": "prompt_tests",
        "expected": 9,
        "observed": len(prompt_tests)
    },
    {
        "item": "human_images",
        "expected": 189,
        "observed": len(human)
    },
    {
        "item": "human_assessments",
        "expected": 746,
        "observed": int(human["n_human"].sum())
    },
    {
        "item": "stability_architectures",
        "expected": 3,
        "observed": len(stability)
    }
])

checks["passed"] = (
    checks["expected"] == checks["observed"]
)

assert checks["passed"].all()

# ------------------------------------------------------------
# SALVAR — SEM SOBRESCREVER
# ------------------------------------------------------------

outputs = {
    "florence_exp28c1_table_architectures.csv":
        architecture,
    "florence_exp28c1_table_prompting.csv":
        prompt,
    "florence_exp28c1_table_prompt_tests.csv":
        prompt_tests,
    "florence_exp28c1_table_ensembles.csv":
        ensemble,
    "florence_exp28c1_table_human_summary.csv":
        human_summary,
    "florence_exp28c1_table_stability.csv":
        stability,
    "florence_exp28c1_consolidation_checks.csv":
        checks
}

existing = [
    filename for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for filename, table in outputs.items():
    table.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# EXIBIÇÃO
# ------------------------------------------------------------

print("\n=== TABELA A: ARQUITETURAS ===")
print(
    architecture[
        [
            "condition",
            "n_correct",
            "n_invalid",
            "accuracy_pct",
            "protocol_role"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\n=== TABELA B: PROMPTING ===")
print(
    prompt[
        [
            "architecture",
            "strategy",
            "n_correct",
            "accuracy_pct",
            "sensitivity_pct",
            "specificity_pct"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\n=== TABELA C: TESTES DE PROMPTING ===")
print(
    prompt_tests[
        [
            "architecture",
            "alternative",
            "affective_gain_pp",
            "holm_p",
            "significant_holm"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n=== TABELA D: ENSEMBLES ===")
print(ensemble.to_string(index=False))

print("\n=== TABELA E: AVALIAÇÕES HUMANAS ===")
print(human_summary.to_string(index=False))

print("\n=== TABELA F: ESTABILIDADE ===")
print(stability.to_string(index=False))

print("\n=== VERIFICAÇÕES ===")
print(checks.to_string(index=False))

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 28C.1 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28C.1
CONSOLIDAÇÃO DAS TABELAS CIENTÍFICAS

Colunas originais dos ensembles:
['condition', 'n_correct', 'accuracy', 'sensitivity', 'specificity', 'TN', 'FP', 'FN', 'TP']

=== TABELA A: ARQUITETURAS ===
       condition  n_correct  n_invalid  accuracy_pct          protocol_role
      Likelihood        181          0        95.767 exploratory_likelihood
            Qwen        179          0        94.709                primary
          SigLIP        177          0        93.651                primary
Llama_restricted        177          0        93.651     secondary_decoding
         SigLIP2        176          0        93.122                primary
            CLIP        167          0        88.360                primary
           BLIP2        160          9        84.656                primary
  Llama_original        157         26        83.069     secondary_decoding

=== TABELA B: PROMPTING ===
architecture       strategy  n_correct  accuracy_pct  sensitiv

## Editorial consolidation of Florence's scientific tables for VISAPP.

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28C.2
# TABELAS EDITORIAIS E VERIFICAÇÃO ESTATÍSTICA — VISAPP
#
# Somente resultados existentes.
# Sem GPU e sem novas inferências.
# Não sobrescreve arquivos anteriores.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

R = Path("/content/florence_visapp/results")

print("=" * 80)
print("FLORENCE — EXPERIMENTO 28C.2")
print("CONSOLIDAÇÃO EDITORIAL PARA O VISAPP")
print("=" * 80)

def read_required(filename):
    path = R / filename
    assert path.exists(), f"Arquivo não encontrado: {path}"
    return pd.read_csv(path)

architecture = read_required(
    "florence_exp28c1_table_architectures.csv"
)
prompt = read_required(
    "florence_exp28c1_table_prompting.csv"
)
stability = read_required(
    "florence_exp28c1_table_stability.csv"
)
ensemble = read_required(
    "florence_exp28c1_table_ensembles.csv"
)
mcnemar_original = read_required(
    "florence_exp26b2_main_15_mcnemar_holm.csv"
)
mcnemar_recomputed = read_required(
    "florence_exp28a2_recomputed_mcnemar.csv"
)

# ------------------------------------------------------------
# 1. TABLE II — MODEL PERFORMANCE
# ------------------------------------------------------------

model_names = {
    "CLIP": "CLIP ViT-B/32",
    "SigLIP": "SigLIP Base",
    "SigLIP2": "SigLIP2 Base",
    "BLIP2": "BLIP-2",
    "Qwen": "Qwen2.5-VL-3B",
    "Likelihood": "Llama 3.2 Vision (likelihood)",
    "Llama_restricted": "Llama 3.2 Vision (constrained)",
    "Llama_original": "Llama 3.2 Vision (original)"
}

table_models = architecture.copy()

table_models["Model / protocol"] = (
    table_models["condition"].map(model_names)
)

table_models["Accuracy (%)"] = (
    100 * table_models["operational_accuracy"]
)

table_models["Valid responses"] = (
    table_models["n_valid"]
)

table_models["Invalid responses"] = (
    table_models["n_invalid"]
)

def safe_div(a, b):
    return a / b if b else np.nan

table_models["Sensitivity (%)"] = [
    100 * safe_div(tp, tp + fn)
    for tp, fn in zip(
        table_models["TP"],
        table_models["FN"]
    )
]

table_models["Specificity (%)"] = [
    100 * safe_div(tn, tn + fp)
    for tn, fp in zip(
        table_models["TN"],
        table_models["FP"]
    )
]

table_models["F1"] = [
    safe_div(
        2 * tp,
        2 * tp + fp + fn
    )
    for tp, fp, fn in zip(
        table_models["TP"],
        table_models["FP"],
        table_models["FN"]
    )
]

table_models["Evidence role"] = (
    table_models["protocol_role"].map({
        "primary": "Main comparison",
        "secondary_decoding": "Decoding analysis",
        "exploratory_likelihood": "Exploratory scoring"
    })
)

table_models = table_models[
    [
        "Model / protocol",
        "Evidence role",
        "n_correct",
        "Valid responses",
        "Invalid responses",
        "Accuracy (%)",
        "Sensitivity (%)",
        "Specificity (%)",
        "F1"
    ]
].sort_values(
    "Accuracy (%)", ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. TABLE III — PROMPTING PERFORMANCE
# ------------------------------------------------------------

prompt_names = {
    "binary": "Binary",
    "affective": "Original affective",
    "B_experiential": "Experiential affective",
    "C_observable": "Observable affective"
}

table_prompt = prompt.copy()

table_prompt["Prompt configuration"] = (
    table_prompt["strategy"].map(prompt_names)
)

table_prompt = table_prompt[
    [
        "architecture",
        "Prompt configuration",
        "n_correct",
        "accuracy_pct",
        "sensitivity_pct",
        "specificity_pct",
        "f1"
    ]
].rename(columns={
    "architecture": "Architecture",
    "n_correct": "Correct",
    "accuracy_pct": "Accuracy (%)",
    "sensitivity_pct": "Sensitivity (%)",
    "specificity_pct": "Specificity (%)",
    "f1": "F1"
})

# ------------------------------------------------------------
# 3. TABLE IV — PROMPT STABILITY
# ------------------------------------------------------------

table_stability = stability[
    [
        "architecture",
        "all_three_agree",
        "all_three_agreement_fraction",
        "affective_vs_experiential_agreement",
        "affective_vs_observable_agreement"
    ]
].copy()

for col in [
    "all_three_agreement_fraction",
    "affective_vs_experiential_agreement",
    "affective_vs_observable_agreement"
]:
    table_stability[col] *= 100

table_stability = table_stability.rename(columns={
    "architecture": "Architecture",
    "all_three_agree": "Stable images",
    "all_three_agreement_fraction": "Three-prompt agreement (%)",
    "affective_vs_experiential_agreement":
        "Original vs experiential (%)",
    "affective_vs_observable_agreement":
        "Original vs observable (%)"
})

# ------------------------------------------------------------
# 4. EXPLORATORY ENSEMBLES — SEPARATE FROM BASELINES
# ------------------------------------------------------------

ensemble_names = [
    "CLIP_SigLIP_Qwen",
    "SigLIP2_Qwen_Likelihood",
    "SigLIP_SigLIP2_Qwen"
]

table_ensembles = ensemble[
    ensemble["condition"].isin(ensemble_names)
].copy()

table_baselines = ensemble[
    ~ensemble["condition"].isin(ensemble_names)
].copy()

assert len(table_ensembles) == 3
assert len(table_baselines) == 5

table_ensembles["Evidence role"] = (
    "Exploratory post-hoc ensemble"
)

table_baselines["Evidence role"] = (
    "Individual-model reference"
)

# ------------------------------------------------------------
# 5. MCNEMAR — CONFERÊNCIA DAS ESTRUTURAS
# ------------------------------------------------------------

print("\n=== COLUNAS MCNEMAR ORIGINAL ===")
print(mcnemar_original.columns.tolist())

print("\n=== COLUNAS MCNEMAR RECONSTRUÍDO ===")
print(mcnemar_recomputed.columns.tolist())

assert len(mcnemar_original) == 15
assert len(mcnemar_recomputed) == 15

# Verificação direta dos valores reproduzidos,
# sem assumir nomes de colunas do arquivo antigo.

recomputed_significant = int(
    (mcnemar_recomputed["holm_p"] < 0.05).sum()
)

assert recomputed_significant == 4

# ------------------------------------------------------------
# 6. LOCALIZAR INTERVALOS DE CONFIANÇA EXISTENTES
# ------------------------------------------------------------

ci_candidates = []

for path in sorted(R.glob("*.csv")):

    name = path.name.lower()

    if any(term in name for term in [
        "bootstrap", "confidence", "interval", "_ci"
    ]):

        try:
            df = pd.read_csv(path)

            ci_candidates.append({
                "filename": path.name,
                "rows": len(df),
                "columns": ", ".join(df.columns)
            })

        except Exception as exc:
            ci_candidates.append({
                "filename": path.name,
                "rows": np.nan,
                "columns": f"READ ERROR: {exc}"
            })

ci_inventory = pd.DataFrame(
    ci_candidates,
    columns=["filename", "rows", "columns"]
)

# ------------------------------------------------------------
# 7. SALVAR TABELAS
# ------------------------------------------------------------

outputs = {
    "florence_exp28c2_table_model_performance.csv":
        table_models,
    "florence_exp28c2_table_prompt_performance.csv":
        table_prompt,
    "florence_exp28c2_table_prompt_stability.csv":
        table_stability,
    "florence_exp28c2_table_exploratory_ensembles.csv":
        table_ensembles,
    "florence_exp28c2_table_ensemble_baselines.csv":
        table_baselines,
    "florence_exp28c2_ci_inventory.csv":
        ci_inventory
}

existing = [
    filename for filename in outputs
    if (R / filename).exists()
]

if existing:
    raise FileExistsError(
        "Arquivos já existentes: "
        + ", ".join(existing)
    )

for filename, df in outputs.items():
    df.to_csv(R / filename, index=False)

# ------------------------------------------------------------
# 8. EXIBIR
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

print("\n=== TABLE II — MODEL PERFORMANCE ===")
print(table_models.to_string(
    index=False,
    float_format=lambda x: f"{x:.3f}"
))

print("\n=== TABLE III — PROMPT PERFORMANCE ===")
print(table_prompt.to_string(
    index=False,
    float_format=lambda x: f"{x:.3f}"
))

print("\n=== TABLE IV — PROMPT STABILITY ===")
print(table_stability.to_string(
    index=False,
    float_format=lambda x: f"{x:.3f}"
))

print("\n=== EXPLORATORY ENSEMBLES ===")
print(table_ensembles.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\n=== ENSEMBLE BASELINES ===")
print(table_baselines.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\n=== CONFIDENCE INTERVAL FILE CANDIDATES ===")
if ci_inventory.empty:
    print("Nenhum arquivo identificado pelo nome.")
else:
    print(ci_inventory.to_string(index=False))

print("\n=== VERIFICAÇÕES ===")
print("Condições experimentais:", len(table_models))
print("Configurações de prompting:", len(table_prompt))
print("Ensembles exploratórios:", len(table_ensembles))
print("Modelos de referência:", len(table_baselines))
print("Comparações McNemar:", len(mcnemar_recomputed))
print("Comparações significativas:", recomputed_significant)

print("\n=== ARQUIVOS GERADOS ===")
for filename in outputs:
    print(R / filename)

print("\nEXPERIMENTO 28C.2 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28C.2
CONSOLIDAÇÃO EDITORIAL PARA O VISAPP

=== COLUNAS MCNEMAR ORIGINAL ===
['condition_a', 'condition_b', 'n', 'accuracy_a', 'accuracy_b', 'difference_pp', 'ci95_low_pp', 'ci95_high_pp', 'a_only_correct', 'b_only_correct', 'discordant', 'mcnemar_exact_p', 'holm_p', 'significant_holm_005']

=== COLUNAS MCNEMAR RECONSTRUÍDO ===
['model_a', 'model_b', 'a_only_correct', 'b_only_correct', 'discordant', 'mcnemar_exact_p', 'holm_p']

=== TABLE II — MODEL PERFORMANCE ===
              Model / protocol       Evidence role  n_correct  Valid responses  Invalid responses  Accuracy (%)  Sensitivity (%)  Specificity (%)    F1
 Llama 3.2 Vision (likelihood) Exploratory scoring        181              189                  0        95.767           92.929           98.889 0.958
                 Qwen2.5-VL-3B     Main comparison        179              189                  0        94.709           92.929           96.667 0.948
                   SigLIP Base     Main comparison 

## Final conference on confidence intervals and statistical tests.

In [ ]:

# ============================================================
# FLORENCE — EXPERIMENTO 28C.3
# CONFERÊNCIA FINAL: MCNEMAR E INTERVALOS DE CONFIANÇA
# Sem GPU, sem reamostragem e sem sobrescrita.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

R = Path("/content/florence_visapp/results")

original = pd.read_csv(
    R / "florence_exp26b2_main_15_mcnemar_holm.csv"
)
recomputed = pd.read_csv(
    R / "florence_exp28a2_recomputed_mcnemar.csv"
)
accuracy_ci = pd.read_csv(
    R / "accuracy_bootstrap_ci95.csv"
)
gain_ci = pd.read_csv(
    R / "prompt_gain_bootstrap_ci95.csv"
)

assert len(original) == len(recomputed) == 15

# ------------------------------------------------------------
# 1. Comparação orientada dos 15 pares
# ------------------------------------------------------------

rows = []

for _, a in original.iterrows():

    x = a["condition_a"]
    y = a["condition_b"]

    match = recomputed.loc[
        (
            (recomputed["model_a"] == x)
            & (recomputed["model_b"] == y)
        )
        |
        (
            (recomputed["model_a"] == y)
            & (recomputed["model_b"] == x)
        )
    ]

    assert len(match) == 1, (
        f"Par ausente ou duplicado: {x}, {y}"
    )

    b = match.iloc[0]

    same_order = (
        b["model_a"] == x
        and b["model_b"] == y
    )

    a_only = (
        b["a_only_correct"]
        if same_order
        else b["b_only_correct"]
    )

    b_only = (
        b["b_only_correct"]
        if same_order
        else b["a_only_correct"]
    )

    checks = {
        "a_only": int(a["a_only_correct"]) == int(a_only),
        "b_only": int(a["b_only_correct"]) == int(b_only),
        "discordant": (
            int(a["discordant"]) == int(b["discordant"])
        ),
        "exact_p": np.isclose(
            a["mcnemar_exact_p"],
            b["mcnemar_exact_p"],
            atol=1e-10,
            rtol=0
        ),
        "holm_p": np.isclose(
            a["holm_p"],
            b["holm_p"],
            atol=1e-10,
            rtol=0
        ),
        "significance": (
            bool(a["significant_holm_005"])
            == bool(b["holm_p"] < 0.05)
        )
    }

    rows.append({
        "condition_a": x,
        "condition_b": y,
        "all_checks_passed": all(checks.values()),
        "failed_checks": ",".join(
            k for k, passed in checks.items()
            if not passed
        ),
        "difference_pp": a["difference_pp"],
        "ci95_low_pp": a["ci95_low_pp"],
        "ci95_high_pp": a["ci95_high_pp"],
        "holm_p": a["holm_p"]
    })

audit = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2. Inspecionar os intervalos de confiança existentes
# ------------------------------------------------------------

print("=" * 78)
print("FLORENCE — EXPERIMENTO 28C.3")
print("=" * 78)

print("\n=== CONFERÊNCIA MCNEMAR ===")
print(
    audit.to_string(
        index=False,
        float_format=lambda v: f"{v:.8f}"
    )
)

print("\nComparações aprovadas:",
      int(audit["all_checks_passed"].sum()),
      "/", len(audit))

print("\n=== IC 95% — ACURÁCIA ===")
print(
    accuracy_ci.to_string(
        index=False,
        float_format=lambda v: f"{v:.6f}"
    )
)

print("\n=== IC 95% — GANHOS DE PROMPTING ===")
print(
    gain_ci.to_string(
        index=False,
        float_format=lambda v: f"{v:.6f}"
    )
)

print("\n=== ESTRUTURA DOS INTERVALOS ===")

for name, df in [
    ("accuracy_ci", accuracy_ci),
    ("prompt_gain_ci", gain_ci)
]:
    print(name, ":", df.columns.tolist())

# ------------------------------------------------------------
# 3. Verificações dos intervalos
# ------------------------------------------------------------

assert (
    accuracy_ci["ci95_lower"]
    <= accuracy_ci["accuracy"]
).all()

assert (
    accuracy_ci["accuracy"]
    <= accuracy_ci["ci95_upper"]
).all()

assert (
    gain_ci["ci95_lower_pp"]
    <= gain_ci["gain_pp"]
).all()

assert (
    gain_ci["gain_pp"]
    <= gain_ci["ci95_upper_pp"]
).all()

assert (
    original["ci95_low_pp"]
    <= original["difference_pp"]
).all()

assert (
    original["difference_pp"]
    <= original["ci95_high_pp"]
).all()

print("\nIntervalos com estimativas dentro dos limites: OK")

# ------------------------------------------------------------
# 4. Exportar conferência
# ------------------------------------------------------------

output = R / "florence_exp28c3_mcnemar_audit.csv"

if output.exists():
    raise FileExistsError(
        f"Arquivo já existente: {output}"
    )

audit.to_csv(output, index=False)

print("\nArquivo gerado:", output)

if audit["all_checks_passed"].all():
    print("\nAUDITORIA MCNEMAR APROVADA.")
else:
    print("\nATENÇÃO: EXISTEM DIVERGÊNCIAS A INVESTIGAR.")

print("\nEXPERIMENTO 28C.3 CONCLUÍDO.")


FLORENCE — EXPERIMENTO 28C.3

=== CONFERÊNCIA MCNEMAR ===
condition_a condition_b  all_checks_passed failed_checks  difference_pp  ci95_low_pp  ci95_high_pp     holm_p
      BLIP2  Likelihood               True                 -11.11111111 -16.40211640   -5.82010582 0.00155574
      BLIP2        Qwen               True                 -10.05291005 -15.87301587   -4.76190476 0.01229074
     SigLIP       BLIP2               True                   8.99470899   3.70370370   14.81481481 0.03010410
    SigLIP2       BLIP2               True                   8.46560847   3.70370370   13.75661376 0.03010410
       CLIP  Likelihood               True                  -7.40740741 -12.69841270   -2.64550265 0.07271826
       CLIP        Qwen               True                  -6.34920635 -11.11111111   -1.58730159 0.16900539
       CLIP      SigLIP               True                  -5.29100529 -10.58201058    0.00000000 0.88863432
       CLIP     SigLIP2               True                  -4